# Commentary Ingestion Pipeline

In [2]:
import os
import re
import json
import time
import random
import datetime
from pathlib import Path
from dotenv import load_dotenv

import pandas as pd
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
from youtube_transcript_api import (
    YouTubeTranscriptApi,
    NoTranscriptFound,
    TranscriptsDisabled,
    VideoUnavailable,
)
from rapidfuzz import fuzz
import anthropic

load_dotenv()

PELOTON_YOUTUBE_API_KEY = os.getenv('PELOTON_YOUTUBE_API_KEY')
PROCESSED_DATA  = Path('../data/processed')
RAW_DIR         = Path('../data/commentary/raw')
EXTRACTED_DIR   = Path('../data/commentary/extracted')
CACHE_DIR       = Path('../data/commentary/cache')

RAW_DIR.mkdir(parents=True, exist_ok=True)
EXTRACTED_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

CACHE_FILE = CACHE_DIR / 'all_channel_videos.parquet'

ytt     = YouTubeTranscriptApi()
youtube = build('youtube', 'v3', developerKey=PELOTON_YOUTUBE_API_KEY)
claude  = anthropic.Anthropic()

print(f'YouTube API key: {"SET" if PELOTON_YOUTUBE_API_KEY else "MISSING"}')
print(f'Raw dir:         {RAW_DIR}')
print(f'Extracted dir:   {EXTRACTED_DIR}')
print(f'Cache dir:       {CACHE_DIR}')
print(f'Raw files:       {len(list(RAW_DIR.glob("*.json")))}')
print(f'Extracted files: {len(list(EXTRACTED_DIR.glob("*.json")))}')
print(f'Cache exists:    {CACHE_FILE.exists()}')


YouTube API key: SET
Raw dir:         ..\data\commentary\raw
Extracted dir:   ..\data\commentary\extracted
Cache dir:       ..\data\commentary\cache
Raw files:       897
Extracted files: 345
Cache exists:    True


In [3]:
CHANNEL_REGISTRY = [
            {"id": "UCqZQlzSHbVJrwrn5XvzrzcA", "name": "NBC Sports",         "coverage": "Grand Tours, Monuments, WorldTour"},
            {"id": "UCu7phdCr-raU7OaJfEpHZww", "name": "GCN Racing",          "coverage": "All WorldTour"},
            {"id": "UCfDfvvMARk4TKcC62ALi6eA", "name": "TNT Sports Cycling",  "coverage": "Grand Tours, European classics"},
            # {"id": "UCuTaETsuCOkJ0H_GAztWt0Q", "name": "GCN",                 "coverage": "All WorldTour"},
            # Official race channels
            {"id": "UCSpycUnuU0IVF7gGIhGojhg", "name": "Tour de France",      "coverage": "Tour de France official"},
            {"id": "UCe10BxbsFg9Kbmkg-ean_Dg", "name": "Giro d'Italia",       "coverage": "Giro d'Italia official"},
            {"id": "UCf7iHZIcKEhiN34-fETtNCA", "name": "La Vuelta",           "coverage": "Vuelta a España official"},
            # {"id": "UCozt5iXNqmhU1I7tcjJ0UFQ", "name": "Eurosport",    "coverage": "All WorldTour"},
            {"id": "UCm0Qjs5OBrv3-d6kKBshEbg", "name": "Tour Down Under",     "coverage": "Tour Down Under official"},
            {"id": "UCXgba6tOLghtJuXaD8LBHWg", "name": "inCycle",             "coverage": "All WorldTour"},
            {"id": "UCcbBlBEtCZ2lX7bodgi02Xg", "name": "Velon",               "coverage": "All WorldTour"},
            {"id": "UClhp9g6TPiqCTOlcw0ROfNg", "name": "TNT Sports",          "coverage": "All WorldTour"},
            
        ]
print(f'Channels: {[c["name"] for c in CHANNEL_REGISTRY]}')


Channels: ['NBC Sports', 'GCN Racing', 'TNT Sports Cycling', 'Tour de France', "Giro d'Italia", 'La Vuelta', 'Tour Down Under', 'inCycle', 'Velon', 'TNT Sports']


In [4]:
# ── CELL 3: Build / refresh local YouTube cache ─────────────────────────────
# Run once to download all uploads from the four channels.
# After that, run weekly (or after a race airs) to pick up new videos.
# Quota cost: ~1–2 units per 50 videos fetched — effectively free.

def _get_upload_playlist(channel_id: str) -> str | None:
    """Get the uploads playlist ID for a channel. Costs 1 quota unit."""
    resp = youtube.channels().list(part='contentDetails', id=channel_id).execute()
    items = resp.get('items', [])
    if not items:
        return None
    return items[0]['contentDetails']['relatedPlaylists']['uploads']


def _fetch_playlist_page(playlist_id: str, page_token=None):
    """Fetch one page of playlist items. Costs 1 quota unit."""
    kwargs = dict(part='snippet', playlistId=playlist_id, maxResults=50)
    if page_token:
        kwargs['pageToken'] = page_token
    return youtube.playlistItems().list(**kwargs).execute()


def fetch_channel_videos(channel: dict, max_pages: int = 500) -> pd.DataFrame:
    """
    Download all video metadata from a channel's uploads playlist.
    max_pages=500 → up to 25,000 videos per channel.
    """
    playlist_id = _get_upload_playlist(channel['id'])
    if not playlist_id:
        print(f'  No uploads playlist found for {channel["name"]}')
        return pd.DataFrame()

    rows, page_token, pages = [], None, 0
    while pages < max_pages:
        resp       = _fetch_playlist_page(playlist_id, page_token)
        pages     += 1
        for item in resp.get('items', []):
            s = item['snippet']
            vid_id = s.get('resourceId', {}).get('videoId')
            if not vid_id or s.get('title') == 'Private video':
                continue
            rows.append({
                'video_id':  vid_id,
                'title':     s['title'],
                'published': s['publishedAt'],
                'channel':   channel['name'],
                'channel_id': channel['id'],
            })
        page_token = resp.get('nextPageToken')
        if not page_token:
            break
        time.sleep(0.1)

    df = pd.DataFrame(rows)
    if not df.empty:
        df['published'] = pd.to_datetime(df['published'], utc=True)
    return df


def build_video_cache(force_refresh: bool = False) -> pd.DataFrame:
    """
    Download all channel uploads and save to parquet.
    Safe to re-run — loads from disk unless force_refresh=True.
    """
    if CACHE_FILE.exists() and not force_refresh:
        df = pd.read_parquet(CACHE_FILE)
        age_hrs = (datetime.datetime.utcnow() -
                   datetime.datetime.fromtimestamp(CACHE_FILE.stat().st_mtime)).seconds / 3600
        print(f'Cache loaded: {len(df):,} videos (last updated {age_hrs:.1f}h ago)')
        print(f'Run build_video_cache(force_refresh=True) to re-download')
        return df

    print('Downloading channel uploads — this takes a few minutes...')
    all_dfs = []
    for ch in CHANNEL_REGISTRY:
        print(f'  {ch["name"]}...')
        ch_df = fetch_channel_videos(ch)
        print(f'    → {len(ch_df):,} videos')
        all_dfs.append(ch_df)
        time.sleep(1)

    df = pd.concat(all_dfs, ignore_index=True)
    df.to_parquet(CACHE_FILE)
    print(f'\nCache saved: {len(df):,} videos → {CACHE_FILE}')
    return df


def refresh_recent(days_back: int = 30) -> pd.DataFrame:
    """
    Lightweight refresh: re-fetch only the first few pages of each channel
    to pick up new uploads without re-downloading the entire history.
    Costs ~8 quota units total (one per channel × 2 pages).
    """
    if not CACHE_FILE.exists():
        print('No cache found — run build_video_cache() first')
        return pd.DataFrame()

    existing = pd.read_parquet(CACHE_FILE)
    cutoff   = pd.Timestamp.utcnow() - pd.Timedelta(days=days_back)
    new_rows = []

    for ch in CHANNEL_REGISTRY:
        playlist_id = _get_upload_playlist(ch['id'])
        if not playlist_id:
            continue
        page_token = None
        for _ in range(4):   # ≤4 pages = 200 newest videos per channel
            resp = _fetch_playlist_page(playlist_id, page_token)
            for item in resp.get('items', []):
                s = item['snippet']
                vid_id = s.get('resourceId', {}).get('videoId')
                if not vid_id:
                    continue
                pub = pd.to_datetime(s['publishedAt'], utc=True)
                if pub < cutoff:
                    break
                if vid_id not in existing['video_id'].values:
                    new_rows.append({'video_id': vid_id, 'title': s['title'],
                                     'published': pub, 'channel': ch['name'],
                                     'channel_id': ch['id']})
            page_token = resp.get('nextPageToken')
            if not page_token:
                break
        time.sleep(0.5)

    if new_rows:
        fresh = pd.DataFrame(new_rows)
        merged = pd.concat([existing, fresh], ignore_index=True).drop_duplicates('video_id')
        merged.to_parquet(CACHE_FILE)
        print(f'Refresh complete: added {len(new_rows)} new videos ({len(merged):,} total)')
        return merged
    else:
        print(f'Refresh complete: no new videos found')
        return existing


# ── RUN ONCE (or weekly) ──────────────────────────────────────────────────────
# First time: downloads everything (~few hundred quota units)
# After that: loads from disk instantly
all_videos = build_video_cache()


Cache loaded: 66,370 videos (last updated 5.2h ago)
Run build_video_cache(force_refresh=True) to re-download


C:\Users\Admin\AppData\Local\Temp\ipykernel_8204\992991223.py:67: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  age_hrs = (datetime.datetime.utcnow() -


In [5]:
# ── CELL: Fetch older NBC Sports videos ──────────────────────────────────────
# NBC Sports has ~44,000 total videos but the standard playlist pull
# only captures the most recent ~20,000 (400 pages × 50 results).
# This cell fetches the older half by skipping to page 400 and collecting
# from there onwards.
#
# Quota cost: ~900 units total (400 skip + 500 collect × 1 unit/page)
# Runtime:    ~5-8 minutes (mostly the skip phase)
#
# Run ONCE after your initial build_video_cache() call.
# Safe to re-run — drop_duplicates handles any overlap.

def fetch_nbc_older(skip_pages: int = 400) -> pd.DataFrame:
    """
    Fetch older NBC Sports videos by skipping past the most recent pages.

    The standard playlist pull gets the newest 20,000 videos (pages 1-400).
    This gets everything from page 400 onwards — the older catalog.

    Args:
        skip_pages: Pages to skip before collecting. Default 400 = skip
                    the 20,000 videos already in your cache.
    """
    nbc_channel = next(c for c in CHANNEL_REGISTRY if c['name'] == 'NBC Sports')

    # Get upload playlist
    resp = youtube.channels().list(
        part='contentDetails', id=nbc_channel['id']
    ).execute()
    playlist_id = resp['items'][0]['contentDetails']['relatedPlaylists']['uploads']

    # ── Phase 1: Skip forward ─────────────────────────────────────────────────
    print(f'Skipping {skip_pages} pages (~{skip_pages * 50:,} videos)...')
    print('This takes about 2-3 minutes. Grab a coffee.')
    page_token = None

    for i in range(skip_pages):
        resp = youtube.playlistItems().list(
            part='snippet', playlistId=playlist_id,
            maxResults=50, pageToken=page_token
        ).execute() if page_token else youtube.playlistItems().list(
            part='snippet', playlistId=playlist_id, maxResults=50
        ).execute()

        page_token = resp.get('nextPageToken')
        if not page_token:
            print(f'Playlist ended at page {i} — only {i * 50:,} videos total')
            return pd.DataFrame()
        if i % 100 == 0 and i > 0:
            print(f'  Skipped {i} pages ({i * 50:,} videos)...')
        time.sleep(0.05)

    print(f'Skip complete. Now collecting from page {skip_pages} onwards...')

    # ── Phase 2: Collect from here ────────────────────────────────────────────
    rows  = []
    pages = 0

    while pages < 500:
        kwargs = dict(part='snippet', playlistId=playlist_id, maxResults=50)
        if page_token:
            kwargs['pageToken'] = page_token
        resp   = youtube.playlistItems().list(**kwargs).execute()
        pages += 1

        for item in resp.get('items', []):
            s = item['snippet']
            vid_id = s.get('resourceId', {}).get('videoId')
            if not vid_id or s.get('title') == 'Private video':
                continue
            rows.append({
                'video_id':   vid_id,
                'title':      s['title'],
                'published':  s['publishedAt'],
                'channel':    'NBC Sports',
                'channel_id': nbc_channel['id'],
            })

        page_token = resp.get('nextPageToken')
        if not page_token:
            break
        time.sleep(0.1)

    df = pd.DataFrame(rows)
    if not df.empty:
        df['published'] = pd.to_datetime(df['published'], utc=True)

    print(f'Collected {len(df):,} older NBC Sports videos ({pages} pages)')
    return df


def merge_nbc_older_into_cache(nbc_older_df: pd.DataFrame) -> pd.DataFrame:
    """Merge older NBC videos into the existing cache."""
    existing = pd.read_parquet(CACHE_FILE)
    print(f'Existing cache: {len(existing):,} videos')

    merged = (
        pd.concat([existing, nbc_older_df], ignore_index=True)
        .drop_duplicates('video_id')
        .reset_index(drop=True)
    )
    merged.to_parquet(CACHE_FILE)

    added = len(merged) - len(existing)
    print(f'Cache updated: {len(existing):,} → {len(merged):,} (+{added:,} new NBC videos)')
    return merged


# # ── Run ───────────────────────────────────────────────────────────────────────
# nbc_older = fetch_nbc_older(skip_pages=400)

# if not nbc_older.empty:
#     # Preview what era we're getting
#     print(f'\nDate range of older videos:')
#     print(f'  Oldest: {nbc_older["published"].min().date()}')
#     print(f'  Newest: {nbc_older["published"].max().date()}')
#     print(f'\nSample titles:')
#     for title in nbc_older.head(5)['title']:
#         print(f'  {title}')

#     # Merge into cache
#     all_videos = merge_nbc_older_into_cache(nbc_older)
#     print(f'\nCache now has {len(all_videos):,} videos total')

In [6]:
# ── CELL 4: Helper functions + race index ────────────────────────────────────
def parse_race_name_and_stage(race_name_full: str):
    clean       = re.sub(r'^\d{4}\s+', '', race_name_full).strip()
    stage_match = re.search(r'Stage\s+(\d+)', clean, re.IGNORECASE)
    stage       = int(stage_match.group(1)) if stage_match else None
    race        = re.sub(r'\s*Stage\s+\d+', '', clean, flags=re.IGNORECASE).strip()
    return race, stage


def make_label(race_name, race_date, stage):
    label = f'{str(race_date)[:4]} {race_name}'
    return label + (f' Stage {stage}' if stage else '')


def make_safe_name(label):
    return re.sub(r'[^a-z0-9]+', '_', label.lower()).strip('_')


def fetch_transcript(video_id: str) -> dict:
    try:
        transcript = ytt.fetch(video_id)
        raw_text   = ' '.join([s.text for s in transcript])
        clean_text = re.sub(r'\[.*?\]', '', raw_text)
        clean_text = re.sub(r'\(.*?\)', '', clean_text)
        clean_text = re.sub(r'\s+', ' ', clean_text).strip()
        duration   = round(transcript[-1].start, 0) if transcript else 0
        return {
            'success':       True,
            'video_id':      video_id,
            'snippet_count': len(transcript),
            'raw_chars':     len(raw_text),
            'clean_chars':   len(clean_text),
            'duration_mins': round(duration / 60, 1),
            'clean_text':    clean_text,
            'preview_start': clean_text[:500],
            'preview_end':   clean_text[-500:],
            'error':         None,
        }
    except NoTranscriptFound:
        return {'success': False, 'video_id': video_id, 'error': 'no_transcript'}
    except TranscriptsDisabled:
        return {'success': False, 'video_id': video_id, 'error': 'transcripts_disabled'}
    except VideoUnavailable:
        return {'success': False, 'video_id': video_id, 'error': 'video_unavailable'}
    except Exception as e:
        return {'success': False, 'video_id': video_id, 'error': str(e)}


def save_no_video(label, race_name, race_date, stage, reason='no_video_found') -> Path:
    safe_name = make_safe_name(label)
    out_path  = RAW_DIR / f'{safe_name}.json'
    result = {
        'label':      label,
        'race_name':  race_name,
        'race_date':  str(race_date)[:10],
        'stage':      stage,
        'video':      None,
        'transcript': None,
        'status':     reason,
    }
    with open(out_path, 'w', encoding='utf-8') as f:
        json.dump(result, f, indent=2)
    return out_path


# Build race index — sort newest-first so recent (more likely to have transcripts) run first
merged_df = pd.read_csv(PROCESSED_DATA / 'merged_uci_races.csv', low_memory=False)
merged_df['Date'] = pd.to_datetime(merged_df['Date'])

race_index = (
    merged_df[['Race Name', 'Race_results', 'Date', 'Year_results', 'Stage_results']]
    .drop_duplicates('Race Name')
    .sort_values('Date', ascending=False)   # ← newest first (more likely to have transcripts/videos)
    .reset_index(drop=True)
)

raw_files   = list(RAW_DIR.glob('*.json'))
saved_names = {f.stem for f in raw_files}

statuses = {}
for path in raw_files:
    with open(path, encoding='utf-8') as f:
        data = json.load(f)
    s = data.get('status', 'unknown')
    statuses[s] = statuses.get(s, 0) + 1

transcript_saved = statuses.get('transcript_saved', 0)
no_video         = statuses.get('no_video_found', 0)
video_found      = statuses.get('video_found', 0)
ip_errors        = sum(v for k, v in statuses.items() if 'block' in k.lower() or 'ip' in k.lower())
other_errors     = sum(v for k, v in statuses.items()
                       if k not in ['transcript_saved', 'no_video_found', 'video_found', 'unknown']
                       and 'block' not in k.lower() and 'ip' not in k.lower())
remaining        = len(race_index) - len(raw_files)

print(f'Total races in dataset:   {len(race_index):,}')
print(f'Already processed:        {len(raw_files):,}')
print(f'  Transcripts saved:      {transcript_saved}')
print(f'  Video found (pending):  {video_found}')
print(f'  No video found:         {no_video}')
print(f'  IP/transcript errors:   {ip_errors + other_errors}')
print(f'Remaining (unprocessed):  {remaining}')
print(f'Est. time at 500/run:     {remaining // 500 + 1} runs (no quota limit!)')


Total races in dataset:   896
Already processed:        897
  Transcripts saved:      538
  Video found (pending):  38
  No video found:         78
  IP/transcript errors:   781
Remaining (unprocessed):  -1
Est. time at 500/run:     0 runs (no quota limit!)


In [7]:
# ── CELL 5: Local matching — zero quota cost ─────────────────────────────────
# Synced with src/peloton_iq/commentary/transcript.py

SKIP_KEYWORDS = [
    'preview', 'interview', 'training', 'behind the scenes',
    'beginner', 'guide', 'how to', 'shorts', 'power data',
    'riders to watch', 'ones to watch', 'top 10',
    'what to watch', 'race preview', 'stage preview',
    'team time trial preparation', 'kitchen', 'mechanic',
    'neutral service', 'french phrases', 'injuries',
    'dutch corner', 'how to watch',
    # Route-map / elevation-profile graphics — static images, no spoken
    # content, so they never have captions (always trip TranscriptsDisabled).
    # Synced from src/peloton_iq/commentary/transcript.py.
    'planimetria', 'altimetria', 'route map', 'course map',
    'profil', 'profile reveal', 'percorso',
]
WOMEN_KEYWORDS = ['femmes', 'feminine', 'women', 'ladies', 'rosa', 'giro rosa']

# Channel priority tiers — higher = better
# Official race channels (tier 4) beat generalist channels on their own race only
CHANNEL_TIER = {
    'NBC Sports':         4,
    'GCN Racing':         1,
    'TNT Sports Cycling': 2,
    # 'Eurosport':          2,
    # 'GCN':                1,
    'Tour de France':     4,   # official — TdF stages only
    "Giro d'Italia":     4,   # official — Giro stages only
    'La Vuelta':          4,   # official — Vuelta stages only
    'Tour Down Under': 4,
    'inCycle': 2,
    'Velon': 2,
    'TNT Sports': 2,
}

# Official channel → which races they cover
OFFICIAL_CHANNEL_RACES = {
    'Tour de France': ['tour de france'],
    "Giro d'Italia": ["giro d'italia", 'giro d italia', 'giro ditalia'],
    'La Vuelta':      ['vuelta a espana', 'vuelta', 'la vuelta'],
    'Tour Down Under': ['tour down under', 'santos tour down under']
}

# Known wrong-race patterns — penalize heavily when title contains these
# for races that shouldn't match them
WRONG_RACE_PATTERNS = {
    # Race name keyword → list of title patterns that indicate a DIFFERENT race
    "giro d'italia":    ['giro rosa', 'giro women', 'giro donne'],
    "tour de france":    ['tour de la provence', 'tour down under', 'tour of britain',
                         'tour de romandie', 'tour de pologne', 'tour de suisse'],
    "tour de suisse":    ['tour de france', 'tour de la provence'],
    "tour de pologne":   ['tour de france', 'tour de la provence'],
    "vuelta a espana":   ['vuelta a san juan', 'vuelta al pais vasco'],
    "tirreno-adriatico": ['giro rosa'],
    "paris-nice":        ['giro rosa'],
    "volta a catalunya": ['giro rosa'],
    "itzulia basque country": ['giro rosa'],
    "tour de romandie":  ['tour de france'],
}


def _get_channel_tier(channel: str, race_name: str) -> int:
    """Official channels only get their tier 4 bonus for their own race."""
    base_tier = CHANNEL_TIER.get(channel, 0)
    if base_tier == 4:
        race_lower = race_name.lower()
        allowed = OFFICIAL_CHANNEL_RACES.get(channel, [])
        if not any(r in race_lower for r in allowed):
            return 1
    return base_tier


def _wrong_race_penalty(race_name: str, title: str) -> float:
    """
    Penalize videos that are clearly the wrong race despite fuzzy name match.
    E.g. "Giro Rosa" matching "Giro d'Italia", "Tour de Provence" matching "Tour de France".
    """
    race_lower  = race_name.lower()
    title_lower = title.lower()
    for race_key, bad_patterns in WRONG_RACE_PATTERNS.items():
        if race_key in race_lower:
            for pattern in bad_patterns:
                if pattern in title_lower:
                    return -80.0   # strong enough to drop below any threshold
    return 0.0


def _is_nbc_priority(race_name: str) -> bool:
    rn = race_name.lower()
    return any(r in rn for r in ['tour de france', 'vuelta a espana', 'vuelta españa'])


def _is_gcn_priority(race_name: str) -> bool:
    rn = race_name.lower()
    return any(r in rn for r in [
        "giro d'italia", 'giro d italia', 'paris-roubaix', 'paris roubaix',
        'tour of flanders', 'ronde van vlaanderen', 'liege-bastogne-liege',
        'milan-san remo', 'milan san remo', 'il lombardia',
    ])


def score_local_video(row, race_name: str, race_date: str, stage=None) -> float:
    score     = 0.0
    title     = str(row['title']).lower()
    channel   = str(row.get('channel', ''))
    race_dt   = pd.to_datetime(race_date, utc=True)
    pub_dt    = pd.to_datetime(row['published'], utc=True)
    days_diff = abs((pub_dt.date() - race_dt.date()).days)

    # Wrong race hard penalty — apply first, can exit early
    wrong_penalty = _wrong_race_penalty(race_name, title)
    if wrong_penalty < 0:
        return wrong_penalty

    # Women's race penalty
    if any(k in title for k in WOMEN_KEYWORDS):
        return -80.0

    # Publish date proximity
    if days_diff == 0:      score += 60
    elif days_diff <= 1:    score += 50
    elif days_diff <= 3:    score += 35
    elif days_diff <= 7:    score += 15
    elif days_diff > 365:   score -= 40 * abs(pub_dt.year - race_dt.year)

    # Race name fuzzy match
    name_score = fuzz.partial_ratio(race_name.lower(), title)
    score += name_score * 0.25
    if name_score < 40:     score -= 40

    # Stage match
    if stage:
        score += 30 if any(p in title for p in [
            f'stage {stage}', f'stage{stage}',
            f'étape {stage}', f'tappa {stage}', f'etapa {stage}'
        ]) else -10

    # Year in title
    if str(race_date)[:4] in title:
        score += 15

    # Content type bonuses
    if 'extended highlights' in title:  score += 20
    elif 'extended' in title:           score += 12
    elif 'highlights' in title:         score += 5
    if 'full race' in title:            score += 8

    # Channel tier bonus (official channels get tier 4 only for their own race)
    score += _get_channel_tier(channel, race_name) * 5

    # NBC priority grand tours
    if _is_nbc_priority(race_name) and channel == 'NBC Sports':
        score += 20
        if 'extended highlights' in title:
            score += 25

    # Official race channel bonus
    for official_channel, races in OFFICIAL_CHANNEL_RACES.items():
        if channel == official_channel:
            race_lower = race_name.lower()
            if any(r in race_lower for r in races):
                score += 25
                if 'highlights' in title:
                    score += 15

    # GCN priority races
    if _is_gcn_priority(race_name) and channel in ('GCN Racing', 'TNT Sports Cycling', 'Eurosport'):
        score += 15

    # Skip penalties
    if any(k in title for k in SKIP_KEYWORDS):  score -= 30

    return round(score, 2)


def find_best_video_local(race_name, race_date, stage=None, threshold=75.0):
    if all_videos is None or all_videos.empty:
        return {'found': False, 'error': 'cache_empty'}

    race_year  = str(race_date)[:4]
    candidates = all_videos[
        all_videos['title'].str.contains(race_year, case=False, na=False)
    ].copy()

    if candidates.empty:
        return {'found': False}

    keywords = [w for w in race_name.split() if len(w) > 3][:2]
    if keywords:
        pattern = '|'.join(re.escape(k) for k in keywords)
        narrow  = candidates[candidates['title'].str.contains(pattern, case=False, na=False)]
        if not narrow.empty:
            candidates = narrow

    # NBC fast-path for grand tours
    if _is_nbc_priority(race_name):
        nbc = candidates[candidates['channel'] == 'NBC Sports'].copy()
        if not nbc.empty:
            nbc['match_score'] = nbc.apply(
                lambda r: score_local_video(r, race_name, race_date, stage), axis=1
            )
            nbc  = nbc.sort_values('match_score', ascending=False)
            best = nbc.iloc[0]
            if best['match_score'] >= threshold:
                nbc_candidates = nbc.head(5).copy()
                nbc_candidates['published'] = nbc_candidates['published'].astype(str)
                return {
                    'found': True, 'video_id': best['video_id'],
                    'title': best['title'], 'published': str(best['published']),
                    'channel': best['channel'],
                    'url': f"https://youtube.com/watch?v={best['video_id']}",
                    'match_score': float(best['match_score']),
                    'all_candidates': nbc_candidates.to_dict('records'),
                }

    # Full pool scoring
    candidates['match_score'] = candidates.apply(
        lambda r: score_local_video(r, race_name, race_date, stage), axis=1
    )
    candidates = candidates.sort_values('match_score', ascending=False)
    # After scoring the full candidate pool, before returning:
    # Prefer "extended highlights" videos if one scores above threshold
    extended = candidates[
        candidates['title'].str.contains('extended highlights', case=False, na=False)
    ]
    if not extended.empty and extended.iloc[0]['match_score'] >= threshold:
        best = extended.iloc[0]
    else:
        best = candidates.iloc[0]

    if best['match_score'] < threshold:
        return {'found': False}

    candidates_out = candidates.head(10).copy()
    candidates_out['published'] = candidates_out['published'].astype(str)

    return {
        'found': True, 'video_id': best['video_id'],
        'title': best['title'], 'published': str(best['published']),
        'channel': best['channel'],
        'url': f"https://youtube.com/watch?v={best['video_id']}",
        'match_score': float(best['match_score']),
        'all_candidates': candidates_out.to_dict('records'),
    }


# ── run_automated_search — MISSING from the original notebook ───────────────
# This was called at the bottom of the notebook (the "RUN" cell) but never
# actually defined anywhere — likely lost in a past edit. Reconstructed here
# to mirror TranscriptFetcher.run_local_matching() in transcript.py: same
# per-race loop, same video_found/no_video_found JSON record shape, same
# default retry_no_video_found=True behavior (races marked no_video_found
# get automatically re-attempted on the next run, so cache refreshes or
# scoring changes can pick up matches that didn't exist/qualify before).

def run_automated_search(
    max_races: int = None,
    threshold: float = 75.0,
    verbose: bool = True,
    retry_no_video_found: bool = True,
) -> dict:
    """
    Match all unprocessed races against the local video cache.
    Zero YouTube API quota — runs as fast as your CPU.
    Safe to run multiple times.

    By default also re-processes races with no_video_found status
    (retry_no_video_found=True), so cache refreshes / scoring changes
    automatically get a second chance to match on the next run.
    """
    if all_videos is None or all_videos.empty:
        print('No video cache loaded. Run CELL 3 first.')
        return {'found': 0, 'not_found': 0, 'skipped': 0}

    to_process = []
    for _, row in race_index.iterrows():
        race_name, stage = parse_race_name_and_stage(row['Race Name'])
        race_date        = str(row['Date'])[:10]
        label             = make_label(race_name, race_date, stage)
        out_path          = RAW_DIR / f'{make_safe_name(label)}.json'

        if not out_path.exists():
            to_process.append((race_name, race_date, stage, label, out_path))
        elif retry_no_video_found:
            try:
                with open(out_path, encoding='utf-8') as f:
                    existing = json.load(f)
                if existing.get('status') == 'no_video_found':
                    to_process.append((race_name, race_date, stage, label, out_path))
            except Exception:
                pass

    limit = max_races if max_races is not None else len(to_process)
    print(f'Local cache: {len(all_videos):,} videos across {all_videos["channel"].nunique()} channels')
    print(f'Races to process: {len(to_process)} (limit: {limit})')
    print(f'{chr(8212)*55}')

    found = not_found = 0
    for i, (race_name, race_date, stage, label, out_path) in enumerate(to_process[:limit]):
        if i % 100 == 0:
            print(f'  [{i} new | {found} found | {not_found} no_video]')

        video = find_best_video_local(race_name, race_date, stage, threshold=threshold)

        if video.get('found'):
            record = {
                'label': label, 'race_name': race_name,
                'race_date': race_date, 'stage': stage,
                'video': {k: v for k, v in video.items() if k != 'found'},
                'transcript': None, 'status': 'video_found',
            }
            with open(out_path, 'w', encoding='utf-8') as f:
                json.dump(record, f, indent=2, ensure_ascii=False)
            found += 1
            if verbose:
                print(f'  ✓ [{video["match_score"]:.0f}] [{video["channel"]}] {video["title"][:55]}')
        else:
            record = {
                'label': label, 'race_name': race_name,
                'race_date': race_date, 'stage': stage,
                'video': None, 'transcript': None, 'status': 'no_video_found',
            }
            with open(out_path, 'w', encoding='utf-8') as f:
                json.dump(record, f, indent=2)
            not_found += 1

    print(f'{chr(8212)*55}')
    print(f'Search complete (no API quota used)')
    print(f'  Videos found:    {found}')
    print(f'  No video:        {not_found}')
    print(f'  Skipped:         {len(to_process) - limit}')
    return {'found': found, 'not_found': not_found, 'skipped': len(to_process) - limit}

In [22]:
search_stats = run_automated_search(max_races=None, verbose=True, threshold=75.0)

Local cache: 66,246 videos across 10 channels
Races to process: 81 (limit: 81)
———————————————————————————————————————————————————————
  [0 new | 0 found | 0 no_video]
  ✓ [86] [GCN Racing] Big Hitters Battle It Out In Baltimore! | Maryland Cycl
———————————————————————————————————————————————————————
Search complete (no API quota used)
  Videos found:    1
  No video:        80
  Skipped:         0


In [8]:
# ── CELL 6: Fetch transcripts ────────────────────────────────────────────────
# No API quota — just HTTP requests to YouTube.
# Uses deliberate delays to avoid IP blocking.
#
# FIXED error classification — synced with src/peloton_iq/commentary/transcript.py:
# Known per-video failures (no_transcript / transcripts_disabled / video_unavailable)
# are matched by EXACT equality first, since these come from fetch_transcript()'s
# typed except branches and are normal, expected outcomes (the video just has no
# captions) — never an IP block, and the batch should continue past them.
#
# ROOT CAUSE OF THE OLD BUG: the previous check was
#     is_ip_block = any(kw in error_msg.lower() for kw in ['blocked', 'ip', '429', 'too many'])
# "ip" as a bare substring matches inside the word "transcript" itself
# ("no_transcr-IP-t"), so every NoTranscriptFound result was misclassified
# as an IP block — explaining "1 success then immediate IP block" even
# after waiting 24-48h, since there was never a real block to wait out.

# Known, expected per-video failures — NOT IP blocks. Matched by exact
# equality, before any keyword/substring search runs.
KNOWN_NON_BLOCK_ERRORS = {'no_transcript', 'transcripts_disabled', 'video_unavailable'}


def run_transcript_fetch(
    max_transcripts: int = 50,
    delay_seconds: float = 45.0,
) -> dict:
    pending = []
    for path in sorted(RAW_DIR.glob('*.json')):
        with open(path, encoding='utf-8') as f:
            data = json.load(f)
        if data.get('status') == 'video_found':
            pending.append((path, data))

    total = min(len(pending), max_transcripts)
    print(f'Videos pending transcript: {len(pending)}')
    print(f'Fetching up to:            {total}')
    print(f'Delay between requests:    {delay_seconds}s')
    print(f'Estimated time:            {total * delay_seconds / 60:.1f} minutes')
    print(f'{chr(8212)*55}')

    success = errors = ip_blocked = 0

    for path, data in pending[:max_transcripts]:
        label    = data['label']
        video    = data['video']
        video_id = video['video_id']

        print(f'\nFetching: {label}')
        print(f'  {video["title"][:60]}')

        transcript = fetch_transcript(video_id)

        if transcript['success']:
            data['transcript'] = {
                'snippet_count': transcript['snippet_count'],
                'raw_chars':     transcript['raw_chars'],
                'clean_chars':   transcript['clean_chars'],
                'duration_mins': transcript['duration_mins'],
                'clean_text':    transcript['clean_text'],
                'preview_start': transcript['preview_start'],
                'preview_end':   transcript['preview_end'],
            }
            data['status'] = 'transcript_saved'
            with open(path, 'w', encoding='utf-8') as f:
                json.dump(data, f, indent=2, ensure_ascii=False)
            print(f'  ✓ {transcript["clean_chars"]:,} chars | {transcript["duration_mins"]:.1f} min')
            success += 1

        else:
            error_msg = transcript['error']

            if error_msg in KNOWN_NON_BLOCK_ERRORS:
                # Expected, normal outcome (no captions / disabled / unavailable)
                # — log and CONTINUE. Do NOT halt the batch for these.
                data['status'] = f'transcript_error:{error_msg}'
                with open(path, 'w', encoding='utf-8') as f:
                    json.dump(data, f, indent=2)
                errors += 1
                print(f'  – skipped ({error_msg})')

            else:
                # Genuinely unclassified exception text — THIS is where real
                # network/HTTP/rate-limit errors land. Word-boundary matched
                # so "ip" can't match inside "transcript", "description", etc.
                is_ip_block = bool(re.search(
                    r'\b(ip|blocked|429|too many requests)\b', error_msg.lower()
                ))
                if is_ip_block:
                    data['status'] = 'transcript_error:ip_blocked'
                    with open(path, 'w', encoding='utf-8') as f:
                        json.dump(data, f, indent=2)
                    ip_blocked += 1
                    print(f'\n⚠ IP blocking detected — stopping.')
                    print(f'  Saved {success} transcripts before block.')
                    break
                else:
                    data['status'] = f'transcript_error:{error_msg[:80]}'
                    with open(path, 'w', encoding='utf-8') as f:
                        json.dump(data, f, indent=2)
                    errors += 1
                    print(f'  ✗ {error_msg[:80]}')

        actual_delay = delay_seconds + random.uniform(-5, 10)
        actual_delay = max(20, actual_delay)
        print(f'  Waiting {actual_delay:.0f}s...')
        time.sleep(actual_delay)

    ip_blocked_total = sum(
        1 for p in RAW_DIR.glob('*.json')
        if json.load(open(p, encoding='utf-8')).get('status') == 'transcript_error:ip_blocked'
    )

    print(f'\n{chr(8212)*55}')
    print(f'Fetch complete')
    print(f'  Transcripts saved:   {success}')
    print(f'  IP blocked:          {ip_blocked}')
    print(f'  Other errors:        {errors}')
    print(f'  Still pending retry: {ip_blocked_total}')
    return {'success': success, 'ip_blocked': ip_blocked, 'errors': errors}

In [8]:
# fetch_stats = run_transcript_fetch(
#     max_transcripts=50,
#     delay_seconds=45.0,
# )

In [9]:
# ── CELL 7: Manual override ──────────────────────────────────────────────────
# Use to: add a video you found manually, retry IP-blocked races, or fix bad matches.
#
# retry_ip_blocked() error classification FIXED — synced with
# src/peloton_iq/commentary/transcript.py: known per-video failures
# (no_transcript / transcripts_disabled / video_unavailable) are matched by
# EXACT equality before any keyword search, and the keyword search itself
# uses \b word-boundary regex so "ip" can't match inside "transcript".
# See CELL 6 for the full root-cause writeup — this was the same bug.

KNOWN_NON_BLOCK_ERRORS = {'no_transcript', 'transcripts_disabled', 'video_unavailable'}


def show_manual_todo() -> list:
    todo = []
    for path in sorted(RAW_DIR.glob('*.json')):
        with open(path, encoding='utf-8') as f:
            data = json.load(f)
        s = data.get('status', '')
        if s in ('no_video_found',) or 'ip_blocked' in s:
            todo.append({'label': data['label'], 'status': s, 'path': path})
    print(f'Manual attention needed: {len(todo)}')
    for t in todo[:30]:
        print(f'  [{t["status"]}] {t["label"]}')
    if len(todo) > 30:
        print(f'  ... and {len(todo) - 30} more')
    return todo


def manual_add(video_id: str, label: str = None) -> dict:
    if label is None:
        print('Provide a label from show_manual_todo() output.')
        show_manual_todo()
        return {}

    safe_name = make_safe_name(label)
    out_path  = RAW_DIR / f'{safe_name}.json'

    # Load existing metadata if available
    race_name, race_date, stage = label, '2017-01-01', None
    if out_path.exists():
        with open(out_path, encoding='utf-8') as f:
            existing = json.load(f)
        race_name = existing.get('race_name', label)
        race_date = existing.get('race_date', '2017-01-01')
        stage     = existing.get('stage')
    else:
        # Try to find in race_index
        for _, row in race_index.iterrows():
            rn, st = parse_race_name_and_stage(row['Race Name'])
            rd     = str(row['Date'])[:10]
            if make_label(rn, rd, st) == label:
                race_name, race_date, stage = rn, rd, st
                break

    print(f'Fetching transcript for: {label}')
    print(f'  Video ID: {video_id}')
    transcript = fetch_transcript(video_id)

    if transcript['success']:
        record = {
            'label':     label,
            'race_name': race_name,
            'race_date': race_date,
            'stage':     stage,
            'video': {
                'video_id':    video_id,
                'title':       f'Manual add: {video_id}',
                'url':         f'https://youtube.com/watch?v={video_id}',
                'channel':     'manual',
                'published':   '',
                'match_score': 999,
            },
            'transcript': {
                'snippet_count': transcript['snippet_count'],
                'raw_chars':     transcript['raw_chars'],
                'clean_chars':   transcript['clean_chars'],
                'duration_mins': transcript['duration_mins'],
                'clean_text':    transcript['clean_text'],
                'preview_start': transcript['preview_start'],
                'preview_end':   transcript['preview_end'],
            },
            'status': 'transcript_saved',
        }
        with open(out_path, 'w', encoding='utf-8') as f:
            json.dump(record, f, indent=2, ensure_ascii=False)
        print(f'  ✓ Saved: {transcript["clean_chars"]:,} chars')
        return record
    else:
        print(f'  ✗ Failed: {transcript["error"]}')
        return {'error': transcript['error']}


def retry_ip_blocked(delay_seconds: float = 90.0) -> dict:
    """
    Retry videos currently marked ip_blocked.

    NOTE: this retries GENUINE blocks going forward. For a one-time
    cleanup of files mislabeled by the OLD substring-matching bug (where
    "no_transcript" was misclassified as ip_blocked), this corrected
    version will now naturally relabel them correctly on first retry —
    a video that's really just missing captions will come back as
    no_transcript/transcripts_disabled instead of re-triggering the
    ip_blocked label, since the exact-match check now runs first.
    """
    blocked = []
    for path in sorted(RAW_DIR.glob('*.json')):
        with open(path, encoding='utf-8') as f:
            data = json.load(f)
        status   = data.get('status', '')
        video    = data.get('video') or {}
        video_id = video.get('video_id')
        if 'ip_blocked' in status and video_id:
            blocked.append((path, data))

    print(f'IP blocked races: {len(blocked)}')
    success = still_blocked = relabeled = errors = 0

    for path, data in blocked:
        label    = data.get('label', path.stem)
        video_id = data['video']['video_id']
        print(f'\nRetrying: {label}')
        transcript = fetch_transcript(video_id)

        if transcript['success']:
            data['transcript'] = {k: transcript[k] for k in
                ['snippet_count','raw_chars','clean_chars','duration_mins',
                 'clean_text','preview_start','preview_end']}
            data['status'] = 'transcript_saved'
            with open(path, 'w', encoding='utf-8') as f:
                json.dump(data, f, indent=2, ensure_ascii=False)
            print(f'  ✓ {transcript["clean_chars"]:,} chars')
            success += 1

        else:
            err = transcript['error']

            if err in KNOWN_NON_BLOCK_ERRORS:
                # This was never really an IP block — relabel correctly
                # and move on, do NOT halt the retry batch for it.
                data['status'] = f'transcript_error:{err}'
                with open(path, 'w', encoding='utf-8') as f:
                    json.dump(data, f, indent=2)
                relabeled += 1
                print(f'  – relabeled (actually {err}, not an IP block)')
            else:
                is_real_ip_block = bool(re.search(
                    r'\b(ip|blocked|429|too many requests)\b', err.lower()
                ))
                if is_real_ip_block:
                    print('  ✗ Still IP blocked — stopping')
                    still_blocked += 1
                    break
                data['status'] = f'transcript_error:{err[:100]}'
                with open(path, 'w', encoding='utf-8') as f:
                    json.dump(data, f, indent=2)
                errors += 1
                print(f'  ✗ {err[:80]}')

        actual = max(30, delay_seconds + random.uniform(-10, 15))
        print(f'  Waiting {actual:.0f}s...')
        time.sleep(actual)

    print(f'\nRetry done — saved: {success}, relabeled (false positives): {relabeled}, '
          f'still blocked: {still_blocked}, errors: {errors}')
    return {'success': success, 'relabeled': relabeled, 'still_blocked': still_blocked, 'errors': errors}


# Uncomment to use:
# show_manual_todo()
# manual_add('VIDEO_ID_HERE', '2019 Paris-Roubaix')
# retry_ip_blocked()

In [10]:
# ── CELL 8: Status dashboard ─────────────────────────────────────────────────

def show_status() -> None:
    raw_files = list(RAW_DIR.glob('*.json'))
    statuses  = {}
    for path in raw_files:
        with open(path, encoding='utf-8') as f:
            data = json.load(f)
        s = data.get('status', 'unknown')
        bucket = (s if s in ('transcript_saved', 'no_video_found', 'video_found')
                  else ('ip_blocked' if 'ip_blocked' in s else 'other_error'))
        statuses[bucket] = statuses.get(bucket, 0) + 1

    extracted_count = len(list(EXTRACTED_DIR.glob('*.json')))
    remaining       = len(race_index) - len(raw_files)
    cache_size      = len(all_videos) if all_videos is not None else 0

    print('══════ Commentary Pipeline Status ══════')
    print(f'Local cache:             {cache_size:,} videos')
    print(f'Total races:             {len(race_index):,}')
    print(f'Processed:               {len(raw_files):,}')
    print(f'  ✓ Transcript saved:    {statuses.get("transcript_saved", 0)}')
    print(f'  → Video found (queue): {statuses.get("video_found", 0)}')
    print(f'  ✗ No video:            {statuses.get("no_video_found", 0)}')
    print(f'  ⚠ IP blocked:          {statuses.get("ip_blocked", 0)}')
    print(f'  ? Other errors:        {statuses.get("other_error", 0)}')
    print(f'Remaining:               {remaining:,}')
    print(f'Extracted (Claude):      {extracted_count}')


show_status()


══════ Commentary Pipeline Status ══════
Local cache:             66,370 videos
Total races:             896
Processed:               897
  ✓ Transcript saved:    538
  → Video found (queue): 38
  ✗ No video:            78
  ⚠ IP blocked:          1
  ? Other errors:        242
Remaining:               -1
Extracted (Claude):      345


In [11]:
# ── CELL 9: Audit transcript quality (v3) ─────────────────────────────────
import re
import unicodedata

BAD_KEYWORDS = [
    'preview', 'riders to watch', 'ones to watch',
    'team time trial preparation', 'guide to', 'how to watch',
    'what to expect', 'top 10 riders', 'race preview',
    'stage preview', 'who to watch',
]

NON_RACE_KEYWORDS = [
    'truck tour', 'bus tour', 'kitchen truck', 'mechanics truck',
    'dutch corner', 'asks the pros', 'key climbs', 'most important climbs',
    'talking points', 'gcn show ep', 'french phrases',
    'mvps', 'super star riders', 'what we ve learnt', 'things we ve learnt',
    'custom tech', 'new bikes', 'bike tech', 'prototype bike',
    'cycling phrases', 'neutral service', 'aero tech',
    'spin vs grind', 'can pros fix', 'coffee do pro',
    'suitcase', 'leader s jersey', 'leaders jersey',
    'dirty kanza', 'cobbled classics', 'what are the',
    'how did', 'how to', 'explained',
    # Women's races
    'giro rosa', 'giro donne', 'giro women',
    # Wrong race patterns
    'tour de la provence', 'tour down under', 'tour of britain',
    'vuelta a san juan', 'santos tour',
]

STAGE_RACES = {
    "tour de france", "giro d'italia", "giro d italia",
    "vuelta a espana", "la vuelta", "criterium du dauphine",
    "tour de suisse", "tour de romandie", "paris-nice",
    "tirreno-adriatico", "volta a catalunya", "tour of the basque country",
    "itzulia basque country", "tour de pologne", "binckbank tour",
    "benelux tour", "tour of guangxi",
}

RACE_TITLE_ALIASES = {
    "vuelta a espana":               ["vuelta a espana", "la vuelta", "vuelta espana"],
    "la vuelta ciclista a espana":   ["vuelta a espana", "la vuelta", "vuelta espana"],
    "giro ditalia":                  ["giro d italia", "giro ditalia"],
    "giro d'italia":                 ["giro d italia", "giro ditalia"],
    "liege-bastogne-liege":          ["liege bastogne liege", "liege bastogne"],
    "liege - bastogne - liege":      ["liege bastogne liege", "liege bastogne"],
    "la fleche wallonne":            ["fleche wallonne"],
    "ronde van vlaanderen":          ["tour of flanders", "ronde", "vlaanderen"],
    "paris-roubaix":                 ["paris roubaix", "roubaix"],
    "milan-san remo":                ["milan san remo", "milan sanremo", "la primavera"],
    "il lombardia":                  ["lombardia", "tour of lombardy"],
    "strade bianche":                ["strade bianche"],
    "amstel gold race":              ["amstel gold", "amstel"],
    "criterium du dauphine":         ["dauphine", "dauphin"],
    "tour de suisse":                ["tour de suisse", "tour of switzerland"],
    "clasica san sebastian":         ["san sebastian", "clasica"],
    "eschborn-frankfurt":            ["eschborn frankfurt", "frankfurt"],
    "omloop het nieuwsblad":         ["omloop", "nieuwsblad"],
    "dwars door vlaanderen":         ["dwars door"],
    "e3 saxo bank classic":          ["e3 saxo", "e3 harelbeke", "e3 binckbank"],
    "gent-wevelgem":                 ["gent wevelgem"],
    "classic brugge-de panne":       ["brugge de panne", "bruges de panne"],
    "bretagne classic":              ["bretagne classic", "ouest france"],
    "tour de pologne":               ["tour de pologne"],
    "volta a catalunya":             ["volta a catalunya", "volta ciclista"],
    "paris-nice":                    ["paris nice"],
    "tirreno-adriatico":             ["tirreno adriatico"],
    "itzulia basque country":        ["itzulia", "basque country", "tour of the basque"],
    "tour of the basque country":    ["itzulia", "basque country"],
    "binckbank tour":                ["binckbank"],
    "benelux tour":                  ["benelux"],
}

MIN_SCORE = 75

# Files to always keep regardless of audit flags
# score=999 means manually provided; Paris-Nice 2017 stages 3-5 are
# good matches where NBC just didn't include the year in the title
ALWAYS_KEEP = {
    '2017_amstel_gold_race.json',
    '2017_strade_bianche.json',
    '2017_tour_de_pologne_stage_1.json',
    '2017_paris_nice_stage_3.json',
    '2017_paris_nice_stage_4.json',
    '2017_paris_nice_stage_5.json',
    '2023_la_vuelta_ciclista_a_espana_stage_15.json',
}


def _normalize_for_match(text: str) -> str:
    text = text.lower()
    text = ''.join(c for c in unicodedata.normalize('NFD', text) if unicodedata.category(c) != 'Mn')
    text = re.sub(r"[''`´'\"&;]", ' ', text)
    text = re.sub(r'[^a-z0-9 ]', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()


def _race_in_title(race_name: str, title: str) -> bool:
    race_norm  = _normalize_for_match(race_name)
    title_norm = _normalize_for_match(title)

    if race_norm in title_norm:
        return True

    words = [w for w in race_norm.split() if len(w) > 3]
    if words and sum(1 for w in words if w in title_norm) >= min(2, len(words)):
        return True

    for canonical, aliases in RACE_TITLE_ALIASES.items():
        canonical_norm = _normalize_for_match(canonical)
        if canonical_norm in race_norm or race_norm in canonical_norm:
            for alias in aliases:
                if _normalize_for_match(alias) in title_norm:
                    return True

    return False


def _stage_in_title(stage: int, title: str) -> bool:
    title_lower = title.lower()
    return any(p in title_lower for p in [
        f'stage {stage}', f'stage{stage}',
        f'étape {stage}', f'etape {stage}',
        f'tappa {stage}', f'etapa {stage}',
        f'st {stage} ',   f'st{stage} ',
    ])


def _is_stage_race(race_name: str) -> bool:
    return any(sr in race_name.lower() for sr in STAGE_RACES)


def audit_transcript_quality(
    year_filter: int = None,
    verbose: bool = True,
    require_year_in_title: bool = True,
    require_stage_in_title: bool = True,
) -> list:
    flagged = []
    
    for path in sorted(RAW_DIR.glob('*.json')):
        try:
            with open(path, encoding='utf-8') as f:
                data = json.load(f)
        except json.JSONDecodeError as e:
            print(f'Skipping corrupted file {path.name}: {e}')
            continue
 
        if data.get('status') != 'transcript_saved':
            continue

        label     = data.get('label', '')
        race_year = label[:4]
        race_name = data.get('race_name', '')
        stage     = data.get('stage')

        if year_filter and not label.startswith(str(year_filter)):
            continue

        video   = data.get('video') or {}
        title   = video.get('title', '')
        score   = video.get('match_score', 999)
        channel = video.get('channel', '')

        reasons = []

        # 1 — Bad keyword
        if any(kw in title.lower() for kw in BAD_KEYWORDS):
            reasons.append('preview_keyword_in_title')

        # 2 — Non-race content or wrong race
        if any(kw in title.lower() for kw in NON_RACE_KEYWORDS):
            reasons.append('non_race_content_or_wrong_race')

        # 3 — Score too low
        if score < MIN_SCORE:
            reasons.append(f'low_score:{score:.0f}')

        # 4 — Year not in title
        if require_year_in_title and race_year not in title:
            reasons.append(f'year_{race_year}_missing_from_title')

        # 5 — Race name not in title
        if not _race_in_title(race_name, title):
            reasons.append('race_name_not_in_title')

        # 6 — Stage number missing
        if stage and _is_stage_race(race_name) and require_stage_in_title:
            if not _stage_in_title(stage, title):
                reasons.append(f'stage_{stage}_missing_from_title')

        if reasons:
            flagged.append({
                'label':   label,
                'title':   title,
                'score':   score,
                'channel': channel,
                'reasons': reasons,
                'path':    path,
            })

    print(f'Flagged as likely bad matches: {len(flagged)} / checked')
    if verbose:
        for f in flagged:
            print(f'\n  [{", ".join(f["reasons"])}]')
            print(f'  Race:  {f["label"]}')
            print(f'  Video: {f["title"]}')
            print(f'  Score: {f["score"]:.0f}  Channel: {f["channel"]}')
    return flagged


def delete_bad_matches(flagged: list, dry_run: bool = True) -> None:
    for f in flagged:
        if dry_run:
            print(f'Would reset: {f["path"].name}')
        else:
            with open(f['path'], encoding='utf-8') as fh:
                data = json.load(fh)
            clean = {
                'label':      data['label'],
                'race_name':  data.get('race_name', ''),
                'race_date':  data.get('race_date', ''),
                'stage':      data.get('stage'),
                'video':      None,
                'transcript': None,
                'status':     'no_video_found',
            }
            with open(f['path'], 'w', encoding='utf-8') as out:
                json.dump(clean, out, indent=2)
            print(f'Reset: {f["path"].name}')

In [12]:
# ── Run audit ─────────────────────────────────────────────────────────────────
bad = audit_transcript_quality(verbose=True)

# Preview what would be reset
delete_bad_matches(bad, dry_run=True)

# # Uncomment to execute:
# delete_bad_matches(bad, dry_run=False)

Flagged as likely bad matches: 66 / checked

  [year_2017_missing_from_title, race_name_not_in_title]
  Race:  2017 Amstel Gold Race
  Video: Manual add: dlfHFC8he18
  Score: 999  Channel: manual

  [year_2017_missing_from_title]
  Race:  2017 Paris-Nice Stage 3
  Video: Cyclist Sam Bennett wins the Paris-Nice Stage 3
  Score: 115  Channel: NBC Sports

  [year_2017_missing_from_title]
  Race:  2017 Paris-Nice Stage 4
  Video: Julian Alaphilippe secures the yellow jersey at Stage 4 of Paris-Nice
  Score: 115  Channel: NBC Sports

  [year_2017_missing_from_title]
  Race:  2017 Paris-Nice Stage 5
  Video: Greipel sprints to victory at Stage 5 of the Paris-Nice
  Score: 115  Channel: NBC Sports

  [year_2017_missing_from_title, race_name_not_in_title]
  Race:  2017 Strade Bianche
  Video: Manual add: pCTdlonFsxs
  Score: 999  Channel: manual

  [year_2017_missing_from_title, race_name_not_in_title, stage_1_missing_from_title]
  Race:  2017 Tour de Pologne Stage 1
  Video: manually provided

In [13]:
# ── CELL 8b: Gap analysis — what needs manual attention ──────────────────────
# NEW — no module equivalent (this is interactive exploration tooling, not
# logic the live agent needs). Complements CELL 9's audit_transcript_quality:
# that flags individual bad matches; this groups EVERYTHING needing attention
# (bad matches + no_video_found + ip_blocked) by race series and year, so you
# can see at a glance where the systematic gaps are — e.g. "Giro d'Italia is
# missing 23 stages across 4 years" rather than scrolling through 99 individual
# filenames one at a time.
#
# Run CELL 9's audit first if you want this to include bad-match flags —
# this cell will call audit_transcript_quality(verbose=False) itself if you
# haven't already, so it's safe to run standalone too.

import re as _re
from collections import defaultdict


def _extract_race_series(race_name: str) -> str:
    """
    Collapse a race_name down to its series for grouping purposes.
    e.g. "Giro d'Italia" -> "Giro d'Italia" (already a series name in
    this dataset — race_name doesn't include "Stage N", that's separate).
    Kept as a thin pass-through with normalization in case race_name
    formatting varies (extra whitespace, case).
    """
    return race_name.strip()


def gap_analysis_summary(min_group_size: int = 1, flagged: list = None) -> dict:
    """
    Group every race needing attention — no_video_found, ip_blocked, and
    flagged bad matches — by race series and year, so systematic gaps
    (e.g. "Giro d'Italia 2017-2021 missing most stages") are visible at
    a glance instead of buried in a flat list of 100+ filenames.

    Args:
        min_group_size: only show race-series groups with at least this
                         many affected races (filters out one-off noise
                         so you can focus on the patterns worth a playlist
                         search rather than a single missing stage).
        flagged:         output of audit_transcript_quality() — pass this
                         explicitly so the bad-match group is always
                         included regardless of cell execution order.
                         If None, bad-match group is skipped (no_video and
                         ip_blocked groups are unaffected either way).
    """
    flagged_paths = {f['path'].name for f in flagged} if flagged else set()
    if flagged is None:
        print('(No `flagged` passed — run CELL 9\'s audit_transcript_quality() '
              'first and pass its result here to include bad-match flags)')

    no_video:   dict[str, dict[int, list[str]]] = defaultdict(lambda: defaultdict(list))
    ip_blocked: dict[str, dict[int, list[str]]] = defaultdict(lambda: defaultdict(list))
    bad_match:  dict[str, dict[int, list[str]]] = defaultdict(lambda: defaultdict(list))

    for path in sorted(RAW_DIR.glob('*.json')):
        with open(path, encoding='utf-8') as f:
            data = json.load(f)

        status    = data.get('status', '')
        race_name = _extract_race_series(data.get('race_name', 'Unknown'))
        label     = data.get('label', path.stem)
        year_str  = label[:4]
        try:
            year = int(year_str)
        except ValueError:
            year = 0

        if status == 'no_video_found':
            no_video[race_name][year].append(label)
        elif 'ip_blocked' in status:
            ip_blocked[race_name][year].append(label)
        elif path.name in flagged_paths:
            bad_match[race_name][year].append(label)

    def _print_group(title: str, data: dict, emoji: str):
        # Sort by total count descending — biggest gaps first
        sorted_races = sorted(
            data.items(),
            key=lambda kv: -sum(len(v) for v in kv[1].values())
        )
        total = sum(len(v) for race in data.values() for v in race.values())
        print(f'\n{emoji} {title} — {total} total')
        print('─' * 60)
        for race_name, by_year in sorted_races:
            race_total = sum(len(v) for v in by_year.values())
            if race_total < min_group_size:
                continue
            years_str = ', '.join(
                f'{y}({len(labels)})' for y, labels in sorted(by_year.items())
            )
            print(f'  {race_name:<40} {race_total:>3}   [{years_str}]')

    print('══════════════ GAP ANALYSIS SUMMARY ══════════════')
    _print_group('NO VIDEO FOUND — needs manual search / playlist import', no_video, '✗')
    _print_group('IP BLOCKED — needs retry (check CELL 7 retry_ip_blocked)', ip_blocked, '⚠')
    _print_group('FLAGGED BAD MATCH — wrong video, needs re-match or manual fix', bad_match, '⚑')

    print('\n' + '═' * 60)
    print('Use this to prioritize:')
    print('  - Race series with the HIGHEST no-video count are your best')
    print('    candidates for a bulk playlist import (CELL 12 below).')
    print('  - Race series with only 1-2 affected races are faster to')
    print('    fix one at a time with manual_add() in CELL 7.')

    return {'no_video': dict(no_video), 'ip_blocked': dict(ip_blocked), 'bad_match': dict(bad_match)}


# ── RUN ───────────────────────────────────────────────────────────────────────
# Run CELL 9's audit first, then pass its result in here explicitly —
# this makes the dependency visible instead of relying on kernel state /
# cell execution order.
bad = audit_transcript_quality(verbose=False)
gaps = gap_analysis_summary(min_group_size=1, flagged=bad)

Flagged as likely bad matches: 66 / checked
══════════════ GAP ANALYSIS SUMMARY ══════════════

✗ NO VIDEO FOUND — needs manual search / playlist import — 78 total
────────────────────────────────────────────────────────────
  Criterium du Dauphine                     36   [2017(8), 2018(7), 2019(8), 2020(5), 2023(8)]
  Presidential Cycling Tour of Turkey        9   [2017(6), 2019(3)]
  Volta Ciclista a Catalunya                 7   [2017(6), 2019(1)]
  Vuelta a Espana                            5   [2020(5)]
  BinckBank Tour                             3   [2017(3)]
  Tour de Pologne                            3   [2017(2), 2019(1)]
  Eschborn-Frankfurt                         3   [2018(1), 2019(1), 2023(1)]
  Itzulia Basque Country                     3   [2019(3)]
  Bretagne Classic - Ouest France            2   [2017(1), 2019(1)]
  Bretagne Classic - Ouest-France            2   [2018(1), 2021(1)]
  Eschborn-Frankfurt 'Rund um den Finanzplatz'   1   [2017(1)]
  Tour de Romandie     

In [14]:
# ── CELL 8c: Find races with a matched video but no transcript ──────────────
# NEW — answers "which races have a video_id saved, but we never got a
# transcript out of it?" — split into the two distinct reasons this happens:
#   1. Still queued (status == 'video_found') — never attempted yet, or
#      attempt was interrupted. Just needs run_transcript_fetch() to run.
#   2. Attempted and failed for a real reason (no_transcript /
#      transcripts_disabled / video_unavailable) — the matched video itself
#      has no captions. THESE are the ones worth manually replacing, since
#      you said you can probably find a better video with real captions.

def find_videos_without_transcript(verbose: bool = True) -> dict:
    queued  = []   # video_found — never attempted, or interrupted
    failed  = []   # transcript_error:no_transcript / transcripts_disabled / video_unavailable

    KNOWN_NON_BLOCK_ERRORS = {'no_transcript', 'transcripts_disabled', 'video_unavailable'}

    for path in sorted(RAW_DIR.glob('*.json')):
        try:
            with open(path, encoding='utf-8') as f:
                data = json.load(f)
        except json.JSONDecodeError:
            continue

        status = data.get('status', '')
        video  = data.get('video') or {}

        if status == 'video_found' and video.get('video_id'):
            queued.append({
                'label':     data.get('label', path.stem),
                'video_id':  video.get('video_id'),
                'title':     video.get('title', ''),
                'channel':   video.get('channel', ''),
                'path':      path,
            })
        elif status.startswith('transcript_error:'):
            error_reason = status.split(':', 1)[1]
            if error_reason in KNOWN_NON_BLOCK_ERRORS and video.get('video_id'):
                failed.append({
                    'label':     data.get('label', path.stem),
                    'video_id':  video.get('video_id'),
                    'title':     video.get('title', ''),
                    'channel':   video.get('channel', ''),
                    'reason':    error_reason,
                    'path':      path,
                })

    if verbose:
        print(f'══════ Videos matched but no transcript ══════')
        print(f'\n📋 QUEUED (status=video_found, never successfully fetched): {len(queued)}')
        for q in queued[:30]:
            print(f'  [{q["channel"]}] {q["label"]}: {q["title"][:55]}')
        if len(queued) > 30:
            print(f'  ... and {len(queued) - 30} more')

        print(f'\n🚫 FAILED — matched video genuinely has no transcript: {len(failed)}')
        # Group by reason for a quick breakdown
        by_reason: dict[str, int] = {}
        for f in failed:
            by_reason[f['reason']] = by_reason.get(f['reason'], 0) + 1
        for reason, count in sorted(by_reason.items(), key=lambda x: -x[1]):
            print(f'  {reason}: {count}')
        print()
        for f in failed:
            print(f'  [{f["reason"]}] [{f["channel"]}] {f["label"]}: {f["title"][:55]}')

    return {'queued': queued, 'failed': failed}


# ── RUN ───────────────────────────────────────────────────────────────────────
no_transcript_results = find_videos_without_transcript(verbose=True)

══════ Videos matched but no transcript ══════

📋 QUEUED (status=video_found, never successfully fetched): 38
  [Giro d'Italia] 2021 Giro d'Italia Stage 11: Giro d’Italia 2021 | Stage 11 | Highlights
  [Giro d'Italia] 2021 Giro d'Italia Stage 12: Giro d’Italia 2021 | Tappa 12 | Highlights
  [Giro d'Italia] 2021 Giro d'Italia Stage 13: Giro d’Italia 2021 | Stage 13 | Highlights
  [Giro d'Italia] 2021 Giro d'Italia Stage 14: Giro d’Italia 2021 | Stage 14 | Highlights
  [Giro d'Italia] 2021 Giro d'Italia Stage 15: Giro d’Italia 2021 | Stage 15 | Highlights
  [Giro d'Italia] 2021 Giro d'Italia Stage 16: Giro d’Italia 2021 | Stage 16 | Highlights
  [Giro d'Italia] 2021 Giro d'Italia Stage 7: Giro d’Italia 2021 | Stage 7 | Highlights
  [Giro d'Italia] 2021 Giro d'Italia Stage 8: Giro d’Italia 2021 | Stage 8 | Highlights
  [Giro d'Italia] 2021 Giro d'Italia Stage 9: Giro d’Italia 2021 | Stage 9 | Highlights
  [TNT Sports] 2021 Tour de Pologne Stage 1: Tour de Luxembourg 2021 - Stage 1 | Highl

In [15]:
# Pass 1 — reset only the genuinely bad ones (exclude manual + false positives)
SKIP_FILES = {
    '2017_amstel_gold_race.json',
    '2017_strade_bianche.json',
    '2017_tour_de_pologne_stage_1.json',
    '2017_paris_nice_stage_3.json',
    '2017_paris_nice_stage_4.json',
    '2017_paris_nice_stage_5.json',
    '2023_la_vuelta_ciclista_a_espana_stage_15.json',
}

def delete_bad_matches_filtered(flagged: list, dry_run: bool = True) -> None:
    to_reset  = [f for f in flagged if f['path'].name not in SKIP_FILES]
    to_keep   = [f for f in flagged if f['path'].name in SKIP_FILES]

    print(f'Will reset:  {len(to_reset)}')
    print(f'Will keep:   {len(to_keep)} (manually skipped)')
    print()
    for f in to_keep:
        print(f'  KEEPING: {f["path"].name}  [{", ".join(f["reasons"])}]')

    print()
    for f in to_reset:
        if dry_run:
            print(f'Would reset: {f["path"].name}')
        else:
            with open(f['path'], encoding='utf-8') as fh:
                data = json.load(fh)
            clean = {
                'label':      data['label'],
                'race_name':  data.get('race_name', ''),
                'race_date':  data.get('race_date', ''),
                'stage':      data.get('stage'),
                'video':      None,
                'transcript': None,
                'status':     'no_video_found',
            }
            with open(f['path'], 'w', encoding='utf-8') as out:
                json.dump(clean, out, indent=2)
            print(f'Reset: {f["path"].name}')


# Preview
# delete_bad_matches_filtered(bad, dry_run=True)

# Execute when ready
delete_bad_matches_filtered(bad, dry_run=False)

Will reset:  60
Will keep:   6 (manually skipped)

  KEEPING: 2017_amstel_gold_race.json  [year_2017_missing_from_title, race_name_not_in_title]
  KEEPING: 2017_paris_nice_stage_3.json  [year_2017_missing_from_title]
  KEEPING: 2017_paris_nice_stage_4.json  [year_2017_missing_from_title]
  KEEPING: 2017_paris_nice_stage_5.json  [year_2017_missing_from_title]
  KEEPING: 2017_strade_bianche.json  [year_2017_missing_from_title, race_name_not_in_title]
  KEEPING: 2017_tour_de_pologne_stage_1.json  [year_2017_missing_from_title, race_name_not_in_title, stage_1_missing_from_title]

Reset: 2017_tour_de_pologne_stage_3.json
Reset: 2017_tour_de_pologne_stage_5.json
Reset: 2017_tour_de_pologne_stage_6.json
Reset: 2017_tour_de_pologne_stage_7.json
Reset: 2017_tour_de_romandie_stage_1.json
Reset: 2017_tour_de_romandie_stage_3.json
Reset: 2017_tour_de_suisse_stage_3.json
Reset: 2017_tour_de_suisse_stage_6.json
Reset: 2018_binckbank_tour_stage_1.json
Reset: 2018_binckbank_tour_stage_3.json
Reset: 20

In [17]:
# ── CELL 13: Channel search bulk import (no playlist required) ──────────────
# NEW — no module equivalent. Complements CELL 12's playlist importer for
# races that DON'T have a dedicated playlist but ARE on a known channel —
# e.g. Criterium du Dauphine videos sitting on the Tour de France channel
# with no separate Dauphine playlist. Searches the LOCAL video cache
# (all_videos) by channel + title keywords + year, then reuses the same
# stage-number extraction and JSON-writing logic as CELL 12.
#
# Zero quota cost — searches the already-downloaded local cache, same as
# find_best_video_local(). No YouTube API calls at all.
#
# Usage:
#   preview_channel_search_import(
#       channel_name='Tour de France',
#       title_must_contain=['dauphine'],   # or ['dauphiné'] / ['criterium']
#       race_name='Criterium du Dauphine',
#       year=2017,
#   )
#   apply_channel_search_import(
#       channel_name='Tour de France',
#       title_must_contain=['dauphine'],
#       race_name='Criterium du Dauphine',
#       year=2017,
#   )


def _search_channel_cache(
    channel_name: str,
    title_must_contain: list[str],
    year: int,
) -> list[dict]:
    """
    Search the local video cache (all_videos) for videos on a given
    channel whose title contains ALL of the given keywords AND the
    target year. Case-insensitive.
    """
    if all_videos is None or all_videos.empty:
        print('No video cache loaded. Run CELL 3 first.')
        return []

    candidates = all_videos[all_videos['channel'] == channel_name].copy()
    if candidates.empty:
        print(f'No videos found for channel: {channel_name}')
        print(f'Available channels: {sorted(all_videos["channel"].unique())}')
        return []

    title_lower = candidates['title'].str.lower()
    for kw in title_must_contain:
        candidates = candidates[title_lower.str.contains(kw.lower(), na=False)]
        title_lower = candidates['title'].str.lower()  # re-narrow each pass

    candidates = candidates[candidates['title'].str.contains(str(year), na=False)]

    return candidates.to_dict('records')


def _build_channel_search_plan(
    channel_name: str,
    title_must_contain: list[str],
    race_name: str,
    year: int,
) -> list[dict]:
    """Same plan-building approach as CELL 12's playlist importer, but
    sourced from a channel keyword search instead of a playlist fetch."""
    videos = _search_channel_cache(channel_name, title_must_contain, year)
    print(f'Found {len(videos)} candidate videos on {channel_name} matching {title_must_contain} + {year}')

    stage_lookup: dict[int, dict] = {}
    for _, row in race_index.iterrows():
        rn, stage = parse_race_name_and_stage(row['Race Name'])
        race_year = str(row['Date'])[:4]
        if rn.lower() == race_name.lower() and race_year == str(year) and stage is not None:
            stage_lookup[stage] = row

    plan = []
    for v in videos:
        stage_num = _extract_stage_number(v['title'])
        row = stage_lookup.get(stage_num) if stage_num is not None else None
        plan.append({
            'video_id':    v['video_id'],
            'title':       v['title'],
            'published':   str(v['published']),
            'stage_num':   stage_num,
            'matched_row': row,
        })
    return plan


def preview_channel_search_import(
    channel_name: str,
    title_must_contain: list[str],
    race_name: str,
    year: int,
) -> list[dict]:
    """Dry run — same shape as preview_playlist_import(). Run this first."""
    plan = _build_channel_search_plan(channel_name, title_must_contain, race_name, year)
    if not plan:
        return []

    matched   = [p for p in plan if p['matched_row'] is not None]
    unmatched = [p for p in plan if p['matched_row'] is None]

    # De-dupe: if multiple candidate videos map to the same stage number,
    # flag it rather than silently picking one — channel search is noisier
    # than a curated playlist, so collisions are more likely here.
    by_stage: dict[int, list] = {}
    for p in matched:
        by_stage.setdefault(p['stage_num'], []).append(p)

    print(f'\n{chr(8212)*60}')
    print(f'PREVIEW — {race_name} {year} (channel search: {channel_name})')
    print(f'{chr(8212)*60}')

    print(f'\n✓ Matched stages: {len(by_stage)}')
    for stage_num in sorted(by_stage):
        candidates_for_stage = by_stage[stage_num]
        if len(candidates_for_stage) > 1:
            print(f'  Stage {stage_num:>2}: ⚠ {len(candidates_for_stage)} CANDIDATES — pick manually:')
            for c in candidates_for_stage:
                print(f'      {c["title"][:70]}  ({c["video_id"]})')
        else:
            c = candidates_for_stage[0]
            label     = make_label(race_name, f'{year}-01-01', stage_num)
            safe_name = make_safe_name(label)
            out_path  = RAW_DIR / f'{safe_name}.json'
            existing_status = ''
            if out_path.exists():
                with open(out_path, encoding='utf-8') as f:
                    existing = json.load(f)
                existing_status = f"  [WOULD OVERWRITE existing: {existing.get('status')}]"
            print(f'  Stage {stage_num:>2}: {c["title"][:65]}{existing_status}')

    if unmatched:
        print(f'\n✗ {len(unmatched)} candidate videos could not be matched to a stage:')
        for p in unmatched:
            reason = 'no stage number found in title' if p['stage_num'] is None else f'stage {p["stage_num"]} not in race_index'
            print(f'  [{reason}] {p["title"][:65]}')

    all_stages_in_index = {
        parse_race_name_and_stage(row['Race Name'])[1]
        for _, row in race_index.iterrows()
        if parse_race_name_and_stage(row['Race Name'])[0].lower() == race_name.lower()
        and str(row['Date'])[:4] == str(year)
    }
    missing_stages = sorted(s for s in all_stages_in_index if s and s not in by_stage)
    if missing_stages:
        print(f'\n⚠ Stages in race_index with NO candidate video found: {missing_stages}')
        print(f'  Try broadening title_must_contain, or these may need manual_add().')

    print(f'\nRun apply_channel_search_import(...) with the same arguments to commit.')
    return plan


def apply_channel_search_import(
    channel_name: str,
    title_must_contain: list[str],
    race_name: str,
    year: int,
    overwrite_existing: bool = False,
    delay_seconds: float = 20.0,
) -> dict:
    """
    Same semantics as CELL 12's apply_playlist_import(): fetches transcripts
    for every matched (non-ambiguous) stage and saves them in the standard
    JSON shape. Stages with MULTIPLE candidate videos are skipped — those
    need a manual_add() call with the specific video_id you choose, since
    auto-picking among ambiguous channel-search results is unsafe.
    """
    plan = _build_channel_search_plan(channel_name, title_must_contain, race_name, year)
    matched = [p for p in plan if p['matched_row'] is not None]

    by_stage: dict[int, list] = {}
    for p in matched:
        by_stage.setdefault(p['stage_num'], []).append(p)

    unambiguous = {s: c[0] for s, c in by_stage.items() if len(c) == 1}
    ambiguous_stages = [s for s, c in by_stage.items() if len(c) > 1]

    if ambiguous_stages:
        print(f'⚠ Skipping ambiguous stages (multiple candidates): {sorted(ambiguous_stages)}')
        print(f'  Use manual_add() for these with your chosen video_id.')

    if not unambiguous:
        print('No unambiguous matched stages to import.')
        return {'success': 0, 'skipped': 0, 'errors': 0}

    print(f'Importing {len(unambiguous)} stages for {race_name} {year}...')
    print(f'{chr(8212)*55}')

    success = skipped = errors = 0

    for stage_num in sorted(unambiguous):
        p         = unambiguous[stage_num]
        row       = p['matched_row']
        race_date = str(row['Date'])[:10]
        label     = make_label(race_name, race_date, stage_num)
        safe_name = make_safe_name(label)
        out_path  = RAW_DIR / f'{safe_name}.json'

        if out_path.exists() and not overwrite_existing:
            with open(out_path, encoding='utf-8') as f:
                existing = json.load(f)
            if existing.get('status') == 'transcript_saved':
                print(f'Stage {stage_num}: skipped (already has transcript_saved)')
                skipped += 1
                continue

        print(f'\nStage {stage_num}: {p["title"][:60]}')
        transcript = fetch_transcript(p['video_id'])

        if transcript['success']:
            record = {
                'label':     label,
                'race_name': race_name,
                'race_date': race_date,
                'stage':     stage_num,
                'video': {
                    'video_id':    p['video_id'],
                    'title':       p['title'],
                    'url':         f'https://youtube.com/watch?v={p["video_id"]}',
                    'channel':     channel_name,
                    'published':   p['published'],
                    'match_score': 999,  # manual channel-search import — trusted
                },
                'transcript': {
                    'snippet_count': transcript['snippet_count'],
                    'raw_chars':     transcript['raw_chars'],
                    'clean_chars':   transcript['clean_chars'],
                    'duration_mins': transcript['duration_mins'],
                    'clean_text':    transcript['clean_text'],
                    'preview_start': transcript['preview_start'],
                    'preview_end':   transcript['preview_end'],
                },
                'status': 'transcript_saved',
            }
            with open(out_path, 'w', encoding='utf-8') as f:
                json.dump(record, f, indent=2, ensure_ascii=False)
            print(f'  ✓ {transcript["clean_chars"]:,} chars | {transcript["duration_mins"]:.1f} min')
            success += 1
        else:
            print(f'  ✗ {transcript["error"]}')
            errors += 1


        time.sleep(delay_seconds)

    print(f'\n{chr(8212)*55}')
    print(f'Channel search import complete')
    print(f'  Saved:   {success}')
    print(f'  Skipped: {skipped}')
    print(f'  Errors:  {errors}')
    print(f'  Ambiguous (need manual_add): {len(ambiguous_stages)}')
    return {'success': success, 'skipped': skipped, 'errors': errors}

In [16]:
# ── CELL 14: Targeted date-range channel fetch ───────────────────────────────
# NEW — solves the problem fetch_nbc_older() can't: reaching a SPECIFIC old
# date range (e.g. July 2018) without walking the entire channel history
# backward page by page. fetch_nbc_older(skip_pages=400) only reached 2024 —
# getting to 2018 would mean skip_pages in the thousands, many chained calls,
# and a lot of wasted quota paging through years of irrelevant content.
#
# This instead uses the YouTube Data API's search.list with publishedAfter/
# publishedBefore — which queries by date directly rather than paging through
# the uploads playlist in chronological order. Costs MORE quota per video
# than playlistItems.list (search.list is 100 units/call vs ~1 unit for
# playlistItems), but for a single race's date window (a handful of calls)
# that's still cheap, and it's the only way to reach an arbitrary old date
# without walking the whole intervening history.
#
# Usage:
#   videos = fetch_channel_videos_in_range(
#       channel_id='UCqZQlzSHbVJrwrn5XvzrzcA',  # NBC Sports — see CHANNELS in CELL 1
#       start_date='2018-07-05',
#       end_date='2018-07-30',
#   )
#   merge_into_cache(videos)


def fetch_channel_videos_in_range(
    channel_id: str,
    start_date: str,
    end_date: str,
) -> pd.DataFrame:
    """
    Fetch all videos from a channel published within [start_date, end_date]
    using search.list's publishedAfter/publishedBefore — reaches an
    arbitrary old date range directly, without paging through the channel's
    entire upload history in between.

    Cost: 100 quota units per page of up to 50 results. For one race's
    ~3-4 week window this is typically 1-3 pages (100-300 units) — check
    your channel's typical upload volume if quota is a concern.
    """
    start_iso = pd.Timestamp(start_date, tz='UTC').isoformat().replace('+00:00', 'Z')
    end_iso   = pd.Timestamp(end_date, tz='UTC').isoformat().replace('+00:00', 'Z')

    rows, page_token = [], None
    while True:
        kwargs = dict(
            part='snippet',
            channelId=channel_id,
            publishedAfter=start_iso,
            publishedBefore=end_iso,
            maxResults=50,
            type='video',
            order='date',
        )
        if page_token:
            kwargs['pageToken'] = page_token

        resp = youtube.search().list(**kwargs).execute()

        for item in resp.get('items', []):
            vid_id = item.get('id', {}).get('videoId')
            s      = item.get('snippet', {})
            if not vid_id:
                continue
            rows.append({
                'video_id':   vid_id,
                'title':      s.get('title', ''),
                'published':  s.get('publishedAt', ''),
                'channel':    s.get('channelTitle', ''),
                'channel_id': channel_id,
            })

        page_token = resp.get('nextPageToken')
        if not page_token:
            break
        time.sleep(0.2)

    df = pd.DataFrame(rows)
    if not df.empty:
        df['published'] = pd.to_datetime(df['published'], utc=True)

    print(f'Found {len(df)} videos from {channel_id} between {start_date} and {end_date}')
    if not df.empty:
        print('Titles:')
        for t in df['title']:
            print(f'  {t}')

    return df


def merge_into_cache(new_videos: pd.DataFrame) -> pd.DataFrame:
    """
    Merge newly fetched videos into the in-memory all_videos cache AND
    persist to disk, same dedup pattern as merge_nbc_older_into_cache().
    """
    global all_videos

    if new_videos.empty:
        print('No new videos to merge.')
        return all_videos

    before = len(all_videos)
    all_videos = (
        pd.concat([all_videos, new_videos], ignore_index=True)
        .drop_duplicates('video_id')
        .reset_index(drop=True)
    )
    all_videos.to_parquet(CACHE_FILE)
    print(f'Cache updated: {before:,} → {len(all_videos):,} (+{len(all_videos) - before} new)')
    return all_videos

In [40]:
# ── DIAGNOSTIC: is NBC's 2018 Tour de France content even in the cache? ─────
# Checks the two known-good video IDs directly, plus a broader date-range
# check, to confirm whether fetch_nbc_older() needs to run further back.

known_good_ids = ['YNHzCmSKgj0', 'SP12v7BgGrU']  # 2018 TdF Stage 1 / Stage 2 NBC recaps

print('Checking if known-good video IDs are in the cache:')
for vid in known_good_ids:
    match = all_videos[all_videos['video_id'] == vid]
    if match.empty:
        print(f'  {vid}: NOT in cache')
    else:
        row = match.iloc[0]
        print(f'  {vid}: IN CACHE — "{row["title"]}" ({row["channel"]}, {row["published"]})')

print('\nNBC Sports date range currently in cache:')
nbc = all_videos[all_videos['channel'] == 'NBC Sports']
if not nbc.empty:
    print(f'  Oldest: {nbc["published"].min()}')
    print(f'  Newest: {nbc["published"].max()}')
    print(f'  Total NBC videos: {len(nbc):,}')

    # Specifically check July 2018 (Tour de France 2018 dates)
    july_2018 = nbc[
        (nbc['published'] >= '2018-07-01') & (nbc['published'] <= '2018-07-31')
    ]
    print(f'\n  NBC videos published in July 2018: {len(july_2018)}')
    if not july_2018.empty:
        print('  Sample titles:')
        for t in july_2018['title'].head(5):
            print(f'    {t}')
else:
    print('  No NBC Sports videos in cache at all.')

Checking if known-good video IDs are in the cache:
  YNHzCmSKgj0: NOT in cache
  SP12v7BgGrU: NOT in cache

NBC Sports date range currently in cache:
  Oldest: 2024-04-04 21:45:07+00:00
  Newest: 2026-06-24 18:00:36+00:00
  Total NBC videos: 20,135

  NBC videos published in July 2018: 0


In [42]:
nbc_id = "UCqZQlzSHbVJrwrn5XvzrzcA"

# Tour de France typically runs late June through late July
tdf_2018 = fetch_channel_videos_in_range(nbc_id, '2018-06-25', '2018-08-05')

Found 124 videos from UCqZQlzSHbVJrwrn5XvzrzcA between 2018-06-25 and 2018-08-05
Titles:
  Carolina Panthers Training Camp 2018: Three Things to Know I NFL I NBC Sports
  Carolina Panthers Training Camp 2018: Luke Kuechly ready for big year post surgery
  Houston Texans Training Camp 2018: Three Things to Know I NFL I NBC Sports
  Houston Texans Training Camp 2018: J.J. Watt shows off knowledge in Peter King quiz
  Houston Texans Training Camp 2018: Deshaun Watson feeling strong after ACL tear
  Buffalo Bills Training Camp 2018: Three Things to Know I NFL I NBC Sports
  Cleveland Browns Training Camp 2018: Three things to know I NFL I NBC Sports
  Cleveland Browns Training Camp 2018: Myles Garrett dominates trivia with Peter King
  Cleveland Browns Training Camp: Jarvis Landry says Browns building something special
  &quot;Glory Road&quot; premieres August 7th on NBCSN
  &quot;Racing Roots: Featuring Martin Truex Jr.&quot; debuts August 4th
  Pittsburgh Steelers Training Camp 2018: Why

In [44]:
merge_into_cache(tdf_2018)

Cache updated: 66,246 → 66,370 (+124 new)


,video_id,title,published,channel,channel_id
0,qU5VmuVgvO8,Queen Anne Stakes 2026 (FULL RACE) | NBC Sports,2026-06-16 14:58:43+00:00,NBC Sports,UCqZQlzSHbVJrwrn5XvzrzcA
1,6s2VvVvsedo,MLB Fantasy Baseball Q&A w/ Eric Samulski & Ja...,2026-06-15 18:07:15+00:00,NBC Sports,UCqZQlzSHbVJrwrn5XvzrzcA
2,3CzhrFX9FFA,"Sha'Carri Richardson: first 100m race, first p...",2026-06-15 15:17:10+00:00,NBC Sports,UCqZQlzSHbVJrwrn5XvzrzcA
3,P_yK-u9p9Kk,Texas Rangers' pitching can keep them alive in...,2026-06-15 03:26:25+00:00,NBC Sports,UCqZQlzSHbVJrwrn5XvzrzcA
4,qxi0ACRezoc,"Brandon Nimmo: Rangers' coaches, Skip Schumake...",2026-06-15 03:08:34+00:00,NBC Sports,UCqZQlzSHbVJrwrn5XvzrzcA
...,...,...,...,...,...
66365,T7KnBswH92E,"Meet Navy reserve officer, K&amp;N driver Jess...",2018-06-27 17:29:09+00:00,NBC Sports,UCqZQlzSHbVJrwrn5XvzrzcA
66366,0hga6DJtVtE,2018 Tour De France on NBCSN,2018-06-26 21:39:13+00:00,NBC Sports,UCqZQlzSHbVJrwrn5XvzrzcA
66367,xtbt957dqfk,World Cup 2018 I Iranians party outside Portug...,2018-06-26 21:13:39+00:00,NBC Sports,UCqZQlzSHbVJrwrn5XvzrzcA
66368,kQ37LqgvlEQ,2018 NASCAR on NBC: Show Open Preview | NASCAR...,2018-06-25 20:40:42+00:00,NBC Sports,UCqZQlzSHbVJrwrn5XvzrzcA


In [47]:
preview_channel_search_import(
    channel_name='NBC Sports',
    title_must_contain=['recap'],
    race_name='Tour de France',
    year=2018,
)

Found 22 candidate videos on NBC Sports matching ['recap'] + 2018

————————————————————————————————————————————————————————————
PREVIEW — Tour de France 2018 (channel search: NBC Sports)
————————————————————————————————————————————————————————————

✓ Matched stages: 21
  Stage  1: Tour de France 2018: Stage 1 Recap I NBC Sports  [WOULD OVERWRITE existing: transcript_saved]
  Stage  2: Tour de France 2018: Stage 2 Recap I NBC Sports  [WOULD OVERWRITE existing: transcript_saved]
  Stage  3: Tour de France 2018: Stage 3 Recap I NBC Sports  [WOULD OVERWRITE existing: transcript_saved]
  Stage  4: Tour de France 2018: Stage 4 Recap I NBC Sports  [WOULD OVERWRITE existing: transcript_saved]
  Stage  5: Tour de France 2018: Stage 5 Recap I NBC Sports  [WOULD OVERWRITE existing: transcript_saved]
  Stage  6: Tour de France 2018: Stage 6 Recap I NBC Sports  [WOULD OVERWRITE existing: transcript_saved]
  Stage  7: Tour de France 2018: Stage 7 Recap I NBC Sports  [WOULD OVERWRITE existing: transc

[{'video_id': '22MMMVwUQVw',
  'title': 'Tour de France 2018: Stage 21 Recap I NBC Sports',
  'published': '2018-07-29 23:15:27+00:00',
  'stage_num': 21,
  'matched_row': Race Name        2018 Tour de France Stage 21
  Race_results                   Tour de France
  Date                      2018-07-29 00:00:00
  Year_results                             2018
  Stage_results                            21.0
  Name: 661, dtype: object},
 {'video_id': 'Sr_shEr08W4',
  'title': 'Tour de France 2018: Stage 20 Recap I NBC Sports',
  'published': '2018-07-28 20:13:11+00:00',
  'stage_num': 20,
  'matched_row': Race Name        2018 Tour de France Stage 20
  Race_results                   Tour de France
  Date                      2018-07-28 00:00:00
  Year_results                             2018
  Stage_results                            20.0
  Name: 662, dtype: object},
 {'video_id': 'Fbz4l-77oU8',
  'title': 'Tour de France 2018: Stage 19 Recap I NBC Sports',
  'published': '2018-07-27 19:

In [ ]:
apply_channel_search_import(
    channel_name='NBC Sports',
    title_must_contain=['recap'],
    race_name='Tour de France',
    year=2018,
    overwrite_existing=True,
)

Found 22 candidate videos on NBC Sports matching ['recap'] + 2018
Importing 21 stages for Tour de France 2018...
———————————————————————————————————————————————————————

Stage 1: Tour de France 2018: Stage 1 Recap I NBC Sports
  ✓ 14,125 chars | 14.4 min

Stage 2: Tour de France 2018: Stage 2 Recap I NBC Sports
  ✓ 4,540 chars | 5.5 min

Stage 3: Tour de France 2018: Stage 3 Recap I NBC Sports
  ✓ 12,019 chars | 12.6 min

Stage 4: Tour de France 2018: Stage 4 Recap I NBC Sports
  ✓ 11,998 chars | 12.1 min

Stage 5: Tour de France 2018: Stage 5 Recap I NBC Sports
  ✓ 12,489 chars | 13.3 min

Stage 6: Tour de France 2018: Stage 6 Recap I NBC Sports
  ✗ 
Could not retrieve a transcript for the video https://www.youtube.com/watch?v=rkMnVOlvRCs! This is most likely caused by:

YouTube is blocking requests from your IP. This usually is due to one of the following reasons:
- You have done too many requests and your IP has been blocked by YouTube
- You are doing requests from an IP belonging t

In [18]:
for race in ["Tour de France", "Vuelta a Espana", "Giro d'Italia"]:
       for _, row in race_index.iterrows():
           rn, _ = parse_race_name_and_stage(row['Race Name'])
           if rn == race:
               print(race, row['Date'])
               break  # just show the first hit per race/year combo if you want all years, drop this break

Tour de France 2023-07-23 00:00:00
Vuelta a Espana 2020-11-08 00:00:00
Giro d'Italia 2022-05-28 00:00:00


In [19]:
# Auto-generate accurate job windows directly from race_index instead of guessing
def build_job_list(race_names: list[str]) -> list[tuple]:
    jobs = []
    for race in race_names:
        years_seen = {}
        for _, row in race_index.iterrows():
            rn, _ = parse_race_name_and_stage(row['Race Name'])
            if rn == race:
                year = row['Date'].year
                years_seen.setdefault(year, []).append(row['Date'])
        for year, dates in sorted(years_seen.items()):
            start = (min(dates) - pd.Timedelta(days=10)).strftime('%Y-%m-%d')
            end   = (max(dates) + pd.Timedelta(days=10)).strftime('%Y-%m-%d')
            jobs.append((race, year, start, end))
    return jobs

STAGE_RACE_JOBS = build_job_list(['Tour de France', 'Vuelta a Espana', "Giro d'Italia"])
print(f'{len(STAGE_RACE_JOBS)} jobs generated from race_index:')
for j in STAGE_RACE_JOBS:
    print(' ', j)

17 jobs generated from race_index:
  ('Tour de France', 2017, '2017-06-21', '2017-08-02')
  ('Tour de France', 2018, '2018-06-27', '2018-08-08')
  ('Tour de France', 2019, '2019-06-26', '2019-08-07')
  ('Tour de France', 2020, '2020-08-19', '2020-09-30')
  ('Tour de France', 2021, '2021-06-16', '2021-07-28')
  ('Tour de France', 2022, '2022-06-22', '2022-08-03')
  ('Tour de France', 2023, '2023-06-21', '2023-08-02')
  ('Vuelta a Espana', 2017, '2017-08-09', '2017-09-20')
  ('Vuelta a Espana', 2018, '2018-08-15', '2018-09-25')
  ('Vuelta a Espana', 2019, '2019-08-15', '2019-09-25')
  ('Vuelta a Espana', 2020, '2020-10-10', '2020-11-18')
  ("Giro d'Italia", 2017, '2017-04-25', '2017-06-07')
  ("Giro d'Italia", 2018, '2018-04-24', '2018-06-06')
  ("Giro d'Italia", 2019, '2019-05-02', '2019-06-11')
  ("Giro d'Italia", 2020, '2020-09-24', '2020-11-04')
  ("Giro d'Italia", 2021, '2021-04-28', '2021-06-09')
  ("Giro d'Italia", 2022, '2022-04-26', '2022-06-07')


In [ ]:
#### END TROUBLESHOOTING

In [22]:
# ── CELL 12: Playlist bulk import ─────────────────────────────────────────────
# NEW — no module equivalent (interactive sourcing tool, not agent-runtime
# logic). Lets you paste ONE playlist URL (e.g. an official channel's full
# "Stage Highlights" playlist for a Grand Tour) and automatically matches
# every video in it to the corresponding race in race_index by parsing the
# stage number out of each video's title — instead of one manual_add() call
# per stage.
#
# Costs 1 YouTube quota unit per 50 videos in the playlist (playlistItems.list)
# — trivially cheap even for a 21-stage Grand Tour playlist.
#
# Usage:
#   preview_playlist_import(
#       playlist_url='https://www.youtube.com/watch?v=6-UUvSbxSqI&list=PL26vj4XQfSgpGIWJcSsAnRZwFEJE8m_SB',
#       race_name="Giro d'Italia",
#       year=2017,
#   )
#   # review the preview output, then:
#   apply_playlist_import(
#       playlist_url='...', race_name="Giro d'Italia", year=2017,
#   )

import urllib.parse


def _extract_playlist_id(url: str) -> str | None:
    """Pull the `list=` query param out of any YouTube URL shape."""
    parsed = urllib.parse.urlparse(url)
    qs     = urllib.parse.parse_qs(parsed.query)
    if 'list' in qs:
        return qs['list'][0]
    return None


def _fetch_playlist_videos(playlist_id: str) -> list[dict]:
    """Fetch all videos in a playlist, in playlist order. Costs 1 quota unit per page."""
    videos, page_token = [], None
    while True:
        kwargs = dict(part='snippet', playlistId=playlist_id, maxResults=50)
        if page_token:
            kwargs['pageToken'] = page_token
        resp = youtube.playlistItems().list(**kwargs).execute()
        for item in resp.get('items', []):
            s      = item['snippet']
            vid_id = s.get('resourceId', {}).get('videoId')
            if not vid_id or s.get('title') == 'Private video':
                continue
            videos.append({
                'video_id':       vid_id,
                'title':          s['title'],
                'published':      s.get('publishedAt', ''),
                'playlist_index': s.get('position', 0),
            })
        page_token = resp.get('nextPageToken')
        if not page_token:
            break
        time.sleep(0.1)
    return videos


# Stage-number extraction patterns, in priority order. Tries each in turn
# and returns the first match — handles "Stage 5", "Stage5", "St 5",
# "Étape 5", "Tappa 5", "Etapa 5", and a bare ordinal pattern like
# "Giro d'Italia 2017 | 5" as a last resort.
_STAGE_NUM_PATTERNS = [
    r'stage\s*(\d+)\b',
    r'st(?:age)?\.?\s*(\d+)\b',
    r'étape\s*(\d+)\b',
    r'etape\s*(\d+)\b',
    r'tappa\s*(\d+)\b',
    r'etapa\s*(\d+)\b',
]


def _extract_stage_number(title: str) -> int | None:
    """Try each stage-number pattern in turn; return the first match found."""
    title_lower = title.lower()
    for pattern in _STAGE_NUM_PATTERNS:
        m = _re.search(pattern, title_lower)
        if m:
            return int(m.group(1))
    return None


def _build_playlist_match_plan(
    playlist_url: str,
    race_name: str,
    year: int,
) -> list[dict]:
    """
    Core matching logic shared by preview and apply. Returns a list of
    plan entries — one per playlist video that successfully maps to a
    stage number, with the corresponding race_index row attached (or
    None if that stage isn't in race_index, e.g. a rest-day video).
    """
    playlist_id = _extract_playlist_id(playlist_url)
    if not playlist_id:
        print(f'Could not extract a playlist ID from: {playlist_url}')
        print('Expected a URL containing "list=PLxxxxx"')
        return []

    print(f'Fetching playlist {playlist_id}...')
    videos = _fetch_playlist_videos(playlist_id)
    print(f'Found {len(videos)} videos in playlist')

    # Build a lookup: stage number -> race_index row for this race/year
    stage_lookup: dict[int, dict] = {}
    for _, row in race_index.iterrows():
        rn, stage = parse_race_name_and_stage(row['Race Name'])
        race_year = str(row['Date'])[:4]
        if rn.lower() == race_name.lower() and race_year == str(year) and stage is not None:
            stage_lookup[stage] = row

    plan = []
    for v in videos:
        stage_num = _extract_stage_number(v['title'])
        row = stage_lookup.get(stage_num) if stage_num is not None else None
        plan.append({
            'video_id':    v['video_id'],
            'title':       v['title'],
            'published':   v['published'],
            'stage_num':   stage_num,
            'matched_row': row,
        })
    return plan


def preview_playlist_import(playlist_url: str, race_name: str, year: int) -> list[dict]:
    """
    Dry run — shows what WOULD be imported without writing anything.
    Always run this before apply_playlist_import().
    """
    plan = _build_playlist_match_plan(playlist_url, race_name, year)
    if not plan:
        return []

    matched   = [p for p in plan if p['matched_row'] is not None]
    unmatched = [p for p in plan if p['matched_row'] is None]

    print(f'\n{chr(8212)*60}')
    print(f'PREVIEW — {race_name} {year}')
    print(f'{chr(8212)*60}')
    print(f'\n✓ Would import {len(matched)} stage videos:')
    for p in sorted(matched, key=lambda x: x['stage_num']):
        # Flag if this race already has a transcript_saved — applying
        # would overwrite it, worth knowing before you commit.
        label     = make_label(race_name, f'{year}-01-01', p['stage_num'])
        safe_name = make_safe_name(label)
        out_path  = RAW_DIR / f'{safe_name}.json'
        existing_status = ''
        if out_path.exists():
            with open(out_path, encoding='utf-8') as f:
                existing = json.load(f)
            existing_status = f"  [WOULD OVERWRITE existing: {existing.get('status')}]"
        print(f'  Stage {p["stage_num"]:>2}: {p["title"][:65]}{existing_status}')

    if unmatched:
        print(f'\n✗ {len(unmatched)} playlist videos could not be matched to a stage:')
        for p in unmatched:
            reason = 'no stage number found in title' if p['stage_num'] is None else f'stage {p["stage_num"]} not in race_index (rest day / not in dataset?)'
            print(f'  [{reason}] {p["title"][:65]}')

    print(f'\nRun apply_playlist_import(...) with the same arguments to commit.')
    return plan


def apply_playlist_import(
    playlist_url: str,
    race_name: str,
    year: int,
    overwrite_existing: bool = False,
    delay_seconds: float = 20.0,
) -> dict:
    """
    Fetch transcripts for every matched stage in the playlist and save
    them as if they'd been found by the normal matching pipeline —
    same JSON shape, so everything downstream (audit, gap analysis,
    the agent's commentary retrieval) treats them identically.

    Args:
        overwrite_existing: if False (default), skips any stage that
            already has status transcript_saved — protects good existing
            matches from being clobbered. Sset True to force re-fetch
            (e.g. you know the existing match is one of the bad ones
            flagged by CELL 9's audit).
        delay_seconds: delay between transcript fetches — same purpose
            as run_transcript_fetch's delay, avoids hammering YouTube.
    """
    plan = _build_playlist_match_plan(playlist_url, race_name, year)
    matched = [p for p in plan if p['matched_row'] is not None]

    if not matched:
        print('No matched stages to import. Run preview_playlist_import() first to debug.')
        return {'success': 0, 'skipped': 0, 'errors': 0}

    print(f'Importing {len(matched)} stages for {race_name} {year}...')
    print(f'{chr(8212)*55}')

    success = skipped = errors = 0

    for p in sorted(matched, key=lambda x: x['stage_num']):
        row       = p['matched_row']
        stage_num = p['stage_num']
        race_date = str(row['Date'])[:10]
        label     = make_label(race_name, race_date, stage_num)
        safe_name = make_safe_name(label)
        out_path  = RAW_DIR / f'{safe_name}.json'

        if out_path.exists() and not overwrite_existing:
            with open(out_path, encoding='utf-8') as f:
                existing = json.load(f)
            if existing.get('status') == 'transcript_saved':
                print(f'Stage {stage_num}: skipped (already has transcript_saved, '
                      f'pass overwrite_existing=True to replace)')
                skipped += 1
                continue

        print(f'\nStage {stage_num}: {p["title"][:60]}')
        transcript = fetch_transcript(p['video_id'])

        if transcript['success']:
            record = {
                'label':     label,
                'race_name': race_name,
                'race_date': race_date,
                'stage':     stage_num,
                'video': {
                    'video_id':    p['video_id'],
                    'title':       p['title'],
                    'url':         f'https://youtube.com/watch?v={p["video_id"]}',
                    'channel':     'manual_playlist_import',
                    'published':   p['published'],
                    'match_score': 999,  # manual import — trusted, skip scoring
                },
                'transcript': {
                    'snippet_count': transcript['snippet_count'],
                    'raw_chars':     transcript['raw_chars'],
                    'clean_chars':   transcript['clean_chars'],
                    'duration_mins': transcript['duration_mins'],
                    'clean_text':    transcript['clean_text'],
                    'preview_start': transcript['preview_start'],
                    'preview_end':   transcript['preview_end'],
                },
                'status': 'transcript_saved',
            }
            with open(out_path, 'w', encoding='utf-8') as f:
                json.dump(record, f, indent=2, ensure_ascii=False)
            print(f'  ✓ {transcript["clean_chars"]:,} chars | {transcript["duration_mins"]:.1f} min')
            success += 1
        else:
            print(f'  ✗ {transcript["error"]}')
            errors += 1

        time.sleep(delay_seconds)

    print(f'\n{chr(8212)*55}')
    print(f'Playlist import complete')
    print(f'  Saved:   {success}')
    print(f'  Skipped: {skipped}')
    print(f'  Errors:  {errors}')
    return {'success': success, 'skipped': skipped, 'errors': errors}

In [20]:
# ── CELL 15 (SIMPLIFIED): NBC bulk match — Tour de France + Vuelta only ─────
# Does exactly ONE thing: writes 'video_found' records matching NBC recap
# videos to TdF/Vuelta stages, for every year present in race_index.
# Does NOT call fetch_transcript() anywhere — zero YouTube transcript
# traffic. Safe to run anytime, including while the CLI pipeline is
# fetching transcripts elsewhere (this only does search.list + local
# cache writes, not youtube_transcript_api calls).
#
# After this finishes, run the CLI in a separate terminal to fetch
# transcripts for everything this writes:
#   python scripts/run_commentary.py --max-transcripts 300
#
# Giro d'Italia is EXCLUDED — confirmed NBC doesn't cover it, handled
# separately later.

NBC_CHANNEL_ID = 'UCqZQlzSHbVJrwrn5XvzrzcA'  # confirm against your CELL 1 channel registry

# ── Build job list from race_index, with a wider buffer (±14 days instead
# of ±10) in case NBC posts recaps later than the stage date itself ──────────
def build_job_list(race_names: list[str], buffer_days: int = 14) -> list[tuple]:
    jobs = []
    for race in race_names:
        years_seen = {}
        for _, row in race_index.iterrows():
            rn, _ = parse_race_name_and_stage(row['Race Name'])
            if rn == race:
                year = row['Date'].year
                years_seen.setdefault(year, []).append(row['Date'])
        for year, dates in sorted(years_seen.items()):
            start = (min(dates) - pd.Timedelta(days=buffer_days)).strftime('%Y-%m-%d')
            end   = (max(dates) + pd.Timedelta(days=buffer_days)).strftime('%Y-%m-%d')
            jobs.append((race, year, start, end))
    return jobs


JOBS = build_job_list(['Tour de France', 'Vuelta a Espana'], buffer_days=14)

print(f'{len(JOBS)} jobs to run:')
for j in JOBS:
    print(f'  {j[0]:<20} {j[1]}   {j[2]} to {j[3]}')

# Stage-number extraction (self-contained, no dependency on other cells)
_STAGE_PATTERNS = [
    r'stage\s*(\d+)\b', r'st(?:age)?\.?\s*(\d+)\b',
    r'étape\s*(\d+)\b', r'etape\s*(\d+)\b',
    r'tappa\s*(\d+)\b', r'etapa\s*(\d+)\b',
]

def _extract_stage(title: str) -> int | None:
    t = title.lower()
    for p in _STAGE_PATTERNS:
        m = _re.search(p, t)
        if m:
            return int(m.group(1))
    return None


# ── Run all jobs ──────────────────────────────────────────────────────────────
all_results = []   # (race_name, year, stage_num, title, video_id, written_or_reason)

for race_name, year, start, end in JOBS:
    print(f'\n{"="*70}')
    print(f'{race_name} {year}  ({start} to {end})')
    print(f'{"="*70}')

    # 1. Fetch from YouTube (search.list — costs quota, no transcript calls)
    videos = fetch_channel_videos_in_range(NBC_CHANNEL_ID, start, end)
    merge_into_cache(videos)

    # 2. Filter to "recap" titles only
    recap_videos = [v for _, v in videos.iterrows() if 'recap' in v['title'].lower()] if not videos.empty else []

    # 3. Build stage lookup for this race/year from race_index
    stage_lookup = {}
    for _, row in race_index.iterrows():
        rn, stage = parse_race_name_and_stage(row['Race Name'])
        if rn == race_name and row['Date'].year == year and stage is not None:
            stage_lookup[stage] = row

    # 4. Match recap videos to stages, write video_found records
    by_stage = {}
    for v in recap_videos:
        s = _extract_stage(v['title'])
        if s is not None and s in stage_lookup:
            by_stage.setdefault(s, []).append(v)

    for stage_num in sorted(stage_lookup.keys()):
        candidates = by_stage.get(stage_num, [])

        if not candidates:
            all_results.append((race_name, year, stage_num, None, None, 'NO_MATCH'))
            continue
        if len(candidates) > 1:
            titles = '; '.join(c['title'][:40] for c in candidates)
            all_results.append((race_name, year, stage_num, titles, None, 'AMBIGUOUS'))
            print(f'  Stage {stage_num}: \u26a0 AMBIGUOUS ({len(candidates)} candidates) - skipped')
            continue

        v   = candidates[0]
        row = stage_lookup[stage_num]
        race_date = str(row['Date'])[:10]
        label     = make_label(race_name, race_date, stage_num)
        out_path  = RAW_DIR / f'{make_safe_name(label)}.json'

        record = {
            'label': label, 'race_name': race_name, 'race_date': race_date,
            'stage': stage_num,
            'video': {
                'video_id': v['video_id'], 'title': v['title'],
                'url': f'https://youtube.com/watch?v={v["video_id"]}',
                'channel': 'NBC Sports', 'published': str(v['published']),
                'match_score': 999,
            },
            'transcript': None,
            'status': 'video_found',
        }
        with open(out_path, 'w', encoding='utf-8') as f:
            json.dump(record, f, indent=2, ensure_ascii=False)

        all_results.append((race_name, year, stage_num, v['title'], v['video_id'], 'WRITTEN'))
        print(f'  Stage {stage_num}: OK {v["title"][:55]}')

    time.sleep(0.5)


# ── Summary - compare this against what you expect to see ───────────────────
print(f'\n\n{"="*70}')
print('SUMMARY - review this before running the CLI fetch')
print(f'{"="*70}')

written   = [r for r in all_results if r[5] == 'WRITTEN']
no_match  = [r for r in all_results if r[5] == 'NO_MATCH']
ambiguous = [r for r in all_results if r[5] == 'AMBIGUOUS']

print(f'\nWRITTEN (video_found, ready for transcript fetch): {len(written)}')
print(f'NO MATCH (NBC has no recap for this stage):          {len(no_match)}')
print(f'AMBIGUOUS (multiple candidates, needs manual pick):   {len(ambiguous)}')

print(f'\nBy race/year:')
from collections import Counter
counts = Counter((r[0], r[1]) for r in written)
for (race, year), count in sorted(counts.items()):
    total_stages = len([r for r in all_results if r[0] == race and r[1] == year])
    print(f'  {race} {year}: {count}/{total_stages} stages matched')

if no_match:
    print(f'\nNO MATCH details:')
    for r in no_match:
        print(f'  {r[0]} {r[1]} Stage {r[2]}')

if ambiguous:
    print(f'\nAMBIGUOUS details (resolve with manual_add):')
    for r in ambiguous:
        print(f'  {r[0]} {r[1]} Stage {r[2]}: {r[3]}')

print(f'\n{"="*70}')
print(f'NEXT STEP - run this in a SEPARATE TERMINAL (not in this notebook):')
print(f'  python scripts/run_commentary.py --max-transcripts {len(written) + 20}')
print(f'Do not run any transcript-fetching cell in this notebook at the same time.')

11 jobs to run:
  Tour de France       2017   2017-06-17 to 2017-08-06
  Tour de France       2018   2018-06-23 to 2018-08-12
  Tour de France       2019   2019-06-22 to 2019-08-11
  Tour de France       2020   2020-08-15 to 2020-10-04
  Tour de France       2021   2021-06-12 to 2021-08-01
  Tour de France       2022   2022-06-18 to 2022-08-07
  Tour de France       2023   2023-06-17 to 2023-08-06
  Vuelta a Espana      2017   2017-08-05 to 2017-09-24
  Vuelta a Espana      2018   2018-08-11 to 2018-09-29
  Vuelta a Espana      2019   2019-08-11 to 2019-09-29
  Vuelta a Espana      2020   2020-10-06 to 2020-11-22

Tour de France 2017  (2017-06-17 to 2017-08-06)
Found 120 videos from UCqZQlzSHbVJrwrn5XvzrzcA between 2017-06-17 and 2017-08-06
Titles:
  1996-97 PL Season Review: Newcastle clashes with Man United
  1995-96 PL Season Review: Man United takes back the crown
  1994-95 PL Season Review: Blackburn takes down Man United
  1993-94 PL Season Review: Man United defends inaugural cr

In [23]:
# ── CELL 16: Run preview against all user-supplied playlists ────────────────
# Uses the EXISTING preview_playlist_import() / apply_playlist_import() from
# CELL 12 - no new matching logic needed, since every link below is a real
# playlist (or, for a few one-off cases, a single anchor video to manually
# add). This replaces the keyword-guessing approach entirely for these races.
#
# Two-phase, same safety pattern as before:
#   1. Run this cell - previews everything, writes nothing.
#   2. Review the printed output for each race/year.
#   3. Call apply_playlist_import(...) individually for ones that look right,
#      or adjust/skip ones that don't.

PLAYLIST_JOBS = [
    # -- Tour de France --
    ('Tour de France', 2017, 'https://www.youtube.com/playlist?list=PLbTJt6mn0mSVxsf8uhzSLiiHyxmpGMRL8'),
    ('Tour de France', 2019, 'https://www.youtube.com/watch?v=LdgBc18R5y8&list=PLXEMPXZ3PY1i2iluXFLGSplAOUzJO9Vq6'),
    ('Tour de France', 2020, 'https://www.youtube.com/playlist?list=PLXEMPXZ3PY1iW7nrpZRkjSDkvO5oqpYUZ'),
    ('Tour de France', 2021, 'https://www.youtube.com/playlist?list=PLRWKt1LuYZ7kp5HYnsUo0HU6JYqsxgqEt'),
    ('Tour de France', 2022, 'https://www.youtube.com/playlist?list=PL3rpHGErtVNEBDMwcg23DbkhbPXfdm1j7'),
    # 2023 TdF: no playlist found - single anchor video, handled separately below

    # -- Vuelta a Espana --
    ('Vuelta a Espana', 2017, 'https://www.youtube.com/watch?v=6JtwXrHdits&list=PL63AFsUKwdts7fiG-5Iqn9uI0LT0DRPl_'),
    ('Vuelta a Espana', 2018, 'https://www.youtube.com/watch?v=bTZGlgkeWSo&list=PLwuXMGd35KXUIzoIKIm7NCf-7O_CbYba6'),
    ('Vuelta a Espana', 2019, 'https://www.youtube.com/watch?v=7a14LIcGJmc&list=PLpfYJCdQDP3bmlBGhXpMzgNRWMSGG2qv0'),
    # 2020 Vuelta: no playlist found - single anchor video, handled separately below
    ('Vuelta a Espana', 2021, 'https://www.youtube.com/watch?v=1QjilwSdHMs&list=PLvXzEqP5BHoFgvAP4NijfiaRgaZ0n0X3E'),
    ('Vuelta a Espana', 2022, 'https://www.youtube.com/watch?v=S49OwsatVLg&list=PLlN2mu0LBKMEZY6histiBbdLY84x2VdYN'),
    ('Vuelta a Espana', 2023, 'https://www.youtube.com/watch?v=jabPJ5_9eSU&list=PL-jtQe550MnTTde2CUiiI7R5lIPgLGHkz'),

    # -- Giro d'Italia --
    ("Giro d'Italia", 2017, 'https://www.youtube.com/playlist?list=PL26vj4XQfSgpGIWJcSsAnRZwFEJE8m_SB'),
    ("Giro d'Italia", 2018, 'https://www.youtube.com/playlist?list=PLwuXMGd35KXWromGO4jOV6eCXHYyi0y-b'),
    ("Giro d'Italia", 2019, 'https://www.youtube.com/playlist?list=PLwuXMGd35KXVB7uxIj9B57ahvdZsBJ8sv'),
    ("Giro d'Italia", 2020, 'https://www.youtube.com/playlist?list=PLwuXMGd35KXXWrExGB3kWHBaN1YZy73fg'),
    ("Giro d'Italia", 2021, 'https://www.youtube.com/playlist?list=PL_IbDU0JIybRI7qcgoatw_1VYpeH75I_Z'),
    ("Giro d'Italia", 2022, 'https://www.youtube.com/playlist?list=PL_IbDU0JIybShTmLzWMPTQIqrnxClFdnf'),
    ("Giro d'Italia", 2023, 'https://www.youtube.com/playlist?list=PL_IbDU0JIybTipnk-cyWl5RdEBKIOUIdO'),
]

print(f'{len(PLAYLIST_JOBS)} playlist jobs to preview\n')

playlist_preview_results = {}
for race_name, year, url in PLAYLIST_JOBS:
    plan = preview_playlist_import(playlist_url=url, race_name=race_name, year=year)
    playlist_preview_results[(race_name, year)] = plan
    print()  # spacer between jobs

18 playlist jobs to preview

Fetching playlist PLbTJt6mn0mSVxsf8uhzSLiiHyxmpGMRL8...


Found 18 videos in playlist

————————————————————————————————————————————————————————————
PREVIEW — Tour de France 2017
————————————————————————————————————————————————————————————

✓ Would import 17 stage videos:
  Stage  1: 2017 Tour de France: Stage 1 Recap  [WOULD OVERWRITE existing: transcript_saved]
  Stage  3: 2017 Tour de France: Stage 3 Recap  [WOULD OVERWRITE existing: transcript_saved]
  Stage  4: 2017 Tour de France: Stage 4 Recap  [WOULD OVERWRITE existing: video_found]
  Stage  5: Tour de France 2017: Stage 5 Recap | NBC Sports  [WOULD OVERWRITE existing: video_found]
  Stage  6: 2017 Tour de France: Stage 6 Recap  [WOULD OVERWRITE existing: video_found]
  Stage  7: 2017 Tour de France: Stage 7 Recap  [WOULD OVERWRITE existing: video_found]
  Stage  8: 2017 Tour de France: Stage 8 Recap  [WOULD OVERWRITE existing: video_found]
  Stage  9: Tour de France 2017: Stage 9 Recap | NBC Sports  [WOULD OVERWRITE existing: video_found]
  Stage 10: Tour de France 2017: Stage 10 Reca

In [24]:
# ── CELL 17: Single-anchor-video races (no playlist available) ──────────────
# For 2023 Tour de France and 2020 Vuelta a Espana, no playlist was found -
# only one anchor video each, following a consistent URL/title pattern for
# the rest of the stages (per your observation). This uses the existing
# all_videos cache to find the sibling videos by matching the anchor's
# title pattern, rather than fetching a playlist that doesn't exist.

def find_siblings_by_title_pattern(
    anchor_video_id: str,
    race_name: str,
    year: int,
) -> list[dict]:
    """
    Given one known-good video_id, find its title in the cache, strip the
    stage number out, and search the cache for other videos matching the
    same title template (channel + everything except the stage number).
    """
    anchor_row = all_videos[all_videos['video_id'] == anchor_video_id]
    if anchor_row.empty:
        print(f'Anchor video {anchor_video_id} not found in cache.')
        print(f'Fetching it directly first...')
        resp = youtube.videos().list(part='snippet', id=anchor_video_id).execute()
        items = resp.get('items', [])
        if not items:
            print('Video not found on YouTube either - check the ID.')
            return []
        s = items[0]['snippet']
        anchor_title   = s['title']
        anchor_channel = s['channelTitle']
        merge_into_cache(pd.DataFrame([{
            'video_id': anchor_video_id, 'title': anchor_title,
            'published': pd.to_datetime(s['publishedAt'], utc=True),
            'channel': anchor_channel, 'channel_id': s.get('channelId', ''),
        }]))
    else:
        anchor_title   = anchor_row.iloc[0]['title']
        anchor_channel = anchor_row.iloc[0]['channel']

    print(f'Anchor title: "{anchor_title}"  (channel: {anchor_channel})')

    anchor_stage = _extract_stage(anchor_title)
    if anchor_stage is None:
        print('Could not extract a stage number from the anchor title - cannot build a pattern.')
        return []

    template = anchor_title.lower().replace(f'stage {anchor_stage}', 'stage X').replace(f'stage{anchor_stage}', 'stageX')
    print(f'Template: "{template}"')

    same_channel = all_videos[all_videos['channel'] == anchor_channel].copy()
    same_channel['stage_num'] = same_channel['title'].apply(_extract_stage)
    same_channel = same_channel[same_channel['stage_num'].notna()]

    def _matches_template(title: str, stage_num: int) -> bool:
        candidate_template = title.lower().replace(f'stage {stage_num}', 'stage X').replace(f'stage{stage_num}', 'stageX')
        return candidate_template == template

    matches = same_channel[
        same_channel.apply(lambda r: _matches_template(r['title'], int(r['stage_num'])), axis=1)
    ]

    print(f'Found {len(matches)} videos matching the same template (including the anchor)')
    return matches.to_dict('records')


def preview_sibling_import(anchor_video_id: str, race_name: str, year: int) -> list[dict]:
    """Dry run for the sibling-pattern approach - review before applying."""
    videos = find_siblings_by_title_pattern(anchor_video_id, race_name, year)
    if not videos:
        return []

    stage_lookup = {}
    for _, row in race_index.iterrows():
        rn, stage = parse_race_name_and_stage(row['Race Name'])
        if rn == race_name and row['Date'].year == year and stage is not None:
            stage_lookup[stage] = row

    print(f'\n{"="*60}')
    print(f'PREVIEW (sibling pattern) - {race_name} {year}')
    print(f'{"="*60}')

    by_stage = {}
    for v in videos:
        s = _extract_stage(v['title'])
        if s is not None and s in stage_lookup:
            by_stage.setdefault(s, []).append(v)

    for stage_num in sorted(stage_lookup.keys()):
        candidates = by_stage.get(stage_num, [])
        if not candidates:
            print(f'  Stage {stage_num:>2}: no match found')
        elif len(candidates) > 1:
            print(f'  Stage {stage_num:>2}: AMBIGUOUS {len(candidates)} candidates: {[c["video_id"] for c in candidates]}')
        else:
            print(f'  Stage {stage_num:>2}: OK {candidates[0]["title"][:60]}  ({candidates[0]["video_id"]})')

    return videos


def apply_sibling_import(
    anchor_video_id: str,
    race_name: str,
    year: int,
    overwrite_existing: bool = True,
) -> dict:
    """Apply the sibling-pattern matches - writes video_found records, NO transcript fetch."""
    videos = find_siblings_by_title_pattern(anchor_video_id, race_name, year)
    if not videos:
        return {'written': 0}

    stage_lookup = {}
    for _, row in race_index.iterrows():
        rn, stage = parse_race_name_and_stage(row['Race Name'])
        if rn == race_name and row['Date'].year == year and stage is not None:
            stage_lookup[stage] = row

    by_stage = {}
    for v in videos:
        s = _extract_stage(v['title'])
        if s is not None and s in stage_lookup:
            by_stage.setdefault(s, []).append(v)

    written = 0
    for stage_num, candidates in sorted(by_stage.items()):
        if len(candidates) != 1:
            print(f'Stage {stage_num}: skipped (0 or multiple candidates)')
            continue

        v   = candidates[0]
        row = stage_lookup[stage_num]
        race_date = str(row['Date'])[:10]
        label     = make_label(race_name, race_date, stage_num)
        out_path  = RAW_DIR / f'{make_safe_name(label)}.json'

        if out_path.exists() and not overwrite_existing:
            with open(out_path, encoding='utf-8') as f:
                existing = json.load(f)
            if existing.get('status') == 'transcript_saved':
                continue

        record = {
            'label': label, 'race_name': race_name, 'race_date': race_date,
            'stage': stage_num,
            'video': {
                'video_id': v['video_id'], 'title': v['title'],
                'url': f'https://youtube.com/watch?v={v["video_id"]}',
                'channel': v['channel'], 'published': str(v['published']),
                'match_score': 999,
            },
            'transcript': None,
            'status': 'video_found',
        }
        with open(out_path, 'w', encoding='utf-8') as f:
            json.dump(record, f, indent=2, ensure_ascii=False)
        written += 1
        print(f'Stage {stage_num}: written')

    print(f'\nTotal written: {written}')
    return {'written': written}


print('Sibling-pattern tools ready: preview_sibling_import(), apply_sibling_import()')
print()
print('Run previews:')
print("  preview_sibling_import('7WlQpnR70zQ', 'Tour de France', 2023)")
print("  preview_sibling_import('7HEYkokitbk', 'Vuelta a Espana', 2020)")

Sibling-pattern tools ready: preview_sibling_import(), apply_sibling_import()

Run previews:
  preview_sibling_import('7WlQpnR70zQ', 'Tour de France', 2023)
  preview_sibling_import('7HEYkokitbk', 'Vuelta a Espana', 2020)


In [25]:
preview_sibling_import('7WlQpnR70zQ', 'Tour de France', 2023)
preview_sibling_import('7HEYkokitbk', 'Vuelta a Espana', 2020)

Anchor video 7WlQpnR70zQ not found in cache.
Fetching it directly first...
Cache updated: 67,350 → 67,351 (+1 new)
Anchor title: "Tour de France 2023: Stage 1 | EXTENDED HIGHLIGHTS | 7/1/2023 | Cycling on NBC Sports"  (channel: NBC Sports)
Template: "tour de france 2023: stage X | extended highlights | 7/1/2023 | cycling on nbc sports"
Found 1 videos matching the same template (including the anchor)

PREVIEW (sibling pattern) - Tour de France 2023
  Stage  1: OK Tour de France 2023: Stage 1 | EXTENDED HIGHLIGHTS | 7/1/202  (7WlQpnR70zQ)
  Stage  2: no match found
  Stage  3: no match found
  Stage  4: no match found
  Stage  5: no match found
  Stage  6: no match found
  Stage  7: no match found
  Stage  8: no match found
  Stage  9: no match found
  Stage 10: no match found
  Stage 11: no match found
  Stage 12: no match found
  Stage 13: no match found
  Stage 14: no match found
  Stage 15: no match found
  Stage 16: no match found
  Stage 17: no match found
  Stage 18: no match foun

[{'video_id': '3HyLSzARKhk',
  'title': 'Vuelta a España 2020: Stage 6 | EXTENDED HIGHLIGHTS | NBC Sports',
  'published': Timestamp('2020-10-25 20:10:21+0000', tz='UTC'),
  'channel': 'NBC Sports',
  'channel_id': 'UCqZQlzSHbVJrwrn5XvzrzcA',
  'stage_num': 6.0},
 {'video_id': '7HEYkokitbk',
  'title': 'Vuelta a España 2020: Stage 1 | EXTENDED HIGHLIGHTS | NBC Sports',
  'published': Timestamp('2020-10-20 20:33:44+0000', tz='UTC'),
  'channel': 'NBC Sports',
  'channel_id': 'UCqZQlzSHbVJrwrn5XvzrzcA',
  'stage_num': 1.0}]

In [26]:
# Check actual stage numbers race_index has for one of the all-zero races
for _, row in race_index.iterrows():
    rn, stage = parse_race_name_and_stage(row['Race Name'])
    if rn == 'Vuelta a Espana' and row['Date'].year == 2021:
        print(stage, row['Race Name'], row['Date'])

In [31]:
# ── CELL 18 (v2): Simple playlist write — with content-type scoring ─────────
# Same as before (extract stage from title, write video_found, no transcript
# fetch, no race_index gate) — but now scores candidates per stage and keeps
# only the BEST one, instead of letting the last-processed video silently
# win. Without this, playlists with multiple videos per stage (recap +
# incident clips + finish clips) ended up keeping whichever was processed
# last — not necessarily the substantive one.
#
# Score preference (highest wins): "extended highlights" > "highlights" >
# "recap" > "summary" > anything else (incident/crash/reaction clips, etc).

_CONTENT_TYPE_SCORES = [
    ('extended highlights', 100),
    ('highlights', 90),
    ('recap', 80),
    ('summary', 70),
]


def _content_type_score(title: str) -> int:
    """Higher = more likely to be the substantive recap/highlights video,
    rather than an incident clip, reaction video, or finish-only snippet."""
    t = title.lower()
    for keyword, score in _CONTENT_TYPE_SCORES:
        if keyword in t:
            return score
    return 0   # no recognized content-type keyword — lowest preference


def write_playlist_simple(
    playlist_url: str,
    race_name: str,
    year: int,
    overwrite_existing: bool = True,
) -> dict:
    """
    Fetch every video in a playlist, extract its stage number from the
    title, score each by content-type keyword, and write ONLY the
    highest-scoring video per stage — no race_index validation, no
    transcript fetch.
    """
    playlist_id = _extract_playlist_id(playlist_url)
    if not playlist_id:
        print(f'Could not extract playlist ID from: {playlist_url}')
        return {'written': 0, 'skipped': 0, 'unmatched': 0}

    videos = _fetch_playlist_videos(playlist_id)
    print(f'Found {len(videos)} videos in playlist for {race_name} {year}')

    # Group all candidates by stage number first, so we can pick the best
    # one per stage rather than overwriting in arbitrary playlist order.
    by_stage: dict[int, list[dict]] = {}
    unmatched = 0
    for v in videos:
        stage_num = _extract_stage(v['title'])
        if stage_num is None:
            unmatched += 1
            print(f'  [no stage found] {v["title"][:60]}')
            continue
        by_stage.setdefault(stage_num, []).append(v)

    written = skipped = 0

    for stage_num in sorted(by_stage.keys()):
        candidates = by_stage[stage_num]

        # Pick the best-scoring candidate; ties broken by playlist order
        # (first one seen at the top score wins).
        best = max(candidates, key=lambda v: _content_type_score(v['title']))

        if len(candidates) > 1:
            other_titles = [c['title'][:40] for c in candidates if c is not best]
            print(f'  Stage {stage_num}: {len(candidates)} candidates, picked best '
                  f'(score={_content_type_score(best["title"])}): "{best["title"][:55]}"')
            print(f'    (passed over: {other_titles})')

        race_date = f'{year}-01-01'
        for _, row in race_index.iterrows():
            rn, st = parse_race_name_and_stage(row['Race Name'])
            if rn == race_name and row['Date'].year == year and st == stage_num:
                race_date = str(row['Date'])[:10]
                break

        label    = make_label(race_name, race_date, stage_num)
        out_path = RAW_DIR / f'{make_safe_name(label)}.json'

        if out_path.exists() and not overwrite_existing:
            with open(out_path, encoding='utf-8') as f:
                existing = json.load(f)
            if existing.get('status') == 'transcript_saved':
                skipped += 1
                print(f'  Stage {stage_num}: skipped (already has transcript_saved)')
                continue

        record = {
            'label': label, 'race_name': race_name, 'race_date': race_date,
            'stage': stage_num,
            'video': {
                'video_id': best['video_id'], 'title': best['title'],
                'url': f'https://youtube.com/watch?v={best["video_id"]}',
                'channel': 'playlist_import', 'published': str(best.get('published', '')),
                'match_score': 999,
            },
            'transcript': None,
            'status': 'video_found',
        }
        with open(out_path, 'w', encoding='utf-8') as f:
            json.dump(record, f, indent=2, ensure_ascii=False)
        written += 1
        if len(candidates) == 1:
            print(f'  Stage {stage_num}: written ({best["title"][:55]})')

    print(f'\n{race_name} {year}: written={written} skipped={skipped} unmatched={unmatched}')
    return {'written': written, 'skipped': skipped, 'unmatched': unmatched}


print('Updated: write_playlist_simple() now scores and picks the best video per stage')
print('(extended highlights > highlights > recap > summary > unscored)')

Updated: write_playlist_simple() now scores and picks the best video per stage
(extended highlights > highlights > recap > summary > unscored)


In [28]:
for race_name, year, url in PLAYLIST_JOBS:
    write_playlist_simple(url, race_name, year)
    print()

Found 18 videos in playlist for Tour de France 2017
  Stage 1: written (2017 Tour de France: Stage 1 Recap)
  Stage 3: written (2017 Tour de France: Stage 3 Recap)
  Stage 4: written (2017 Tour de France: Stage 4 Recap)
  Stage 5: written (Tour de France 2017: Stage 5 Recap | NBC Sports)
  Stage 6: written (2017 Tour de France: Stage 6 Recap)
  Stage 7: written (2017 Tour de France: Stage 7 Recap)
  Stage 8: written (2017 Tour de France: Stage 8 Recap)
  Stage 9: written (Tour de France 2017: Stage 9 Recap | NBC Sports)
  Stage 10: written (Tour de France 2017: Stage 10 Recap | NBC Sports)
  Stage 11: written (2017 Tour de France: Stage 11 Recap)
  Stage 12: written (Tour de France 2017: Stage 12 Recap | NBC Sports)
  Stage 13: written (2017 Tour de France: Stage 13 Recap)
  Stage 14: written (2017 Tour de France: Stage 14 Recap)
  Stage 16: written (Tour de France 2017: Stage 16 Recap | NBC Sports)
  Stage 17: written (Tour de France 2017: Stage 17 Recap | NBC Sports)
  Stage 18: writ

In [32]:
rerun_jobs = [
    ('Tour de France', 2019, 'https://www.youtube.com/watch?v=LdgBc18R5y8&list=PLXEMPXZ3PY1i2iluXFLGSplAOUzJO9Vq6'),
    ('Tour de France', 2020, 'https://www.youtube.com/playlist?list=PLXEMPXZ3PY1iW7nrpZRkjSDkvO5oqpYUZ'),
    ("Giro d'Italia", 2017, 'https://www.youtube.com/playlist?list=PL26vj4XQfSgpGIWJcSsAnRZwFEJE8m_SB'),
    ("Giro d'Italia", 2018, 'https://www.youtube.com/playlist?list=PLwuXMGd35KXWromGO4jOV6eCXHYyi0y-b'),
]

for race_name, year, url in rerun_jobs:
    write_playlist_simple(url, race_name, year)
    print()

Found 56 videos in playlist for Tour de France 2019
  [no stage found] Tour de France 2019: Egan Bernal's top 10 moments | NBC Spor
  [no stage found] Tour de France 2019: Top 10 moments | NBC Sports
  [no stage found] Tour de France 2019: Best commentator calls from third and f
  [no stage found] Final ceremonies from the 2019 Tour de France | NBC Sports
  [no stage found] Tour de France 2019: Julian Alaphilippe cries after losing y
  [no stage found] Thibaut Pinot forced to abandon Tour de France | NBC Sports
  [no stage found] Tour de France 2019: Should Luke Rowe, Tony Martin have been
  [no stage found] Tour de France 2019: Best commentator calls from the second 
  [no stage found] Lance Armstrong: Geraint Thomas will take yellow jersey | 20
  [no stage found] Tour de France 2019: Scariest crashes from week one | NBC Sp
  [no stage found] Tour de France 2019: Wout Van-Aert opens up about amazing st
  [no stage found] Tejay van Garderen out of 2019 Tour de France after crash | 
  [

In [29]:
# ── CELL 19: Simple channel-search write — for races with no playlist ───────
# Same philosophy as CELL 18's write_playlist_simple(): extract stage number
# from title, write video_found record, NO transcript fetch, NO race_index
# gate blocking the write. Only difference: sources candidate videos from a
# channel+keyword search of the already-cached all_videos, rather than a
# playlist fetch — for races where no playlist exists (2023 Tour de France,
# 2020 Vuelta a España, per your findings).

def write_channel_search_simple(
    channel_name: str,
    title_must_contain: list[str],
    race_name: str,
    year: int,
    overwrite_existing: bool = True,
) -> dict:
    """
    Search the local cache for videos on `channel_name` whose title contains
    ALL of `title_must_contain` and mentions `year`, extract each match's
    stage number, and write a video_found record. No transcript fetch.
    """
    candidates = all_videos[all_videos['channel'] == channel_name].copy()
    if candidates.empty:
        print(f'No videos cached for channel: {channel_name}')
        return {'written': 0, 'skipped': 0, 'unmatched': 0}

    title_lower = candidates['title'].str.lower()
    for kw in title_must_contain:
        candidates = candidates[title_lower.str.contains(kw.lower(), na=False)]
        title_lower = candidates['title'].str.lower()

    candidates = candidates[candidates['title'].str.contains(str(year), na=False)]

    print(f'Found {len(candidates)} candidate videos for {race_name} {year} on {channel_name}')

    written = skipped = unmatched = 0
    seen_stages = {}

    for _, v in candidates.iterrows():
        stage_num = _extract_stage(v['title'])
        if stage_num is None:
            unmatched += 1
            print(f'  [no stage found] {v["title"][:60]}')
            continue

        if stage_num in seen_stages:
            print(f'  Stage {stage_num}: AMBIGUOUS - multiple candidates, keeping first seen, skipping duplicate')
            print(f'    kept:    {seen_stages[stage_num][:55]}')
            print(f'    skipped: {v["title"][:55]}')
            continue
        seen_stages[stage_num] = v['title']

        race_date = f'{year}-01-01'
        for _, row in race_index.iterrows():
            rn, st = parse_race_name_and_stage(row['Race Name'])
            if rn == race_name and row['Date'].year == year and st == stage_num:
                race_date = str(row['Date'])[:10]
                break

        label    = make_label(race_name, race_date, stage_num)
        out_path = RAW_DIR / f'{make_safe_name(label)}.json'

        if out_path.exists() and not overwrite_existing:
            with open(out_path, encoding='utf-8') as f:
                existing = json.load(f)
            if existing.get('status') == 'transcript_saved':
                skipped += 1
                print(f'  Stage {stage_num}: skipped (already has transcript_saved)')
                continue

        record = {
            'label': label, 'race_name': race_name, 'race_date': race_date,
            'stage': stage_num,
            'video': {
                'video_id': v['video_id'], 'title': v['title'],
                'url': f'https://youtube.com/watch?v={v["video_id"]}',
                'channel': channel_name, 'published': str(v.get('published', '')),
                'match_score': 999,
            },
            'transcript': None,
            'status': 'video_found',
        }
        with open(out_path, 'w', encoding='utf-8') as f:
            json.dump(record, f, indent=2, ensure_ascii=False)
        written += 1
        print(f'  Stage {stage_num}: written ({v["title"][:55]})')

    print(f'\n{race_name} {year}: written={written} skipped={skipped} unmatched={unmatched}')
    return {'written': written, 'skipped': skipped, 'unmatched': unmatched}


print('Ready: write_channel_search_simple(channel_name, title_must_contain, race_name, year)')
print()
print('Run for the two no-playlist races:')
print("  write_channel_search_simple('NBC Sports', ['tour de france 2023', 'extended highlights'], 'Tour de France', 2023)")
print("  write_channel_search_simple('NBC Sports', ['vuelta', 'extended highlights'], 'Vuelta a Espana', 2020)")

Ready: write_channel_search_simple(channel_name, title_must_contain, race_name, year)

Run for the two no-playlist races:
  write_channel_search_simple('NBC Sports', ['tour de france 2023', 'extended highlights'], 'Tour de France', 2023)
  write_channel_search_simple('NBC Sports', ['vuelta', 'extended highlights'], 'Vuelta a Espana', 2020)


In [30]:
write_channel_search_simple('NBC Sports', ['tour de france 2023', 'extended highlights'], 'Tour de France', 2023)
write_channel_search_simple('NBC Sports', ['vuelta', 'extended highlights'], 'Vuelta a Espana', 2020)

Found 9 candidate videos for Tour de France 2023 on NBC Sports
  Stage 20: written (Tour de France 2023: Stage 20 | EXTENDED HIGHLIGHTS | 7)
  Stage 19: written (Tour de France 2023: Stage 19 | EXTENDED HIGHLIGHTS | 7)
  Stage 15: written (Tour de France 2023: Stage 15 | EXTENDED HIGHLIGHTS | 7)
  Stage 11: written (Tour de France 2023: Stage 11 | EXTENDED HIGHLIGHTS | 7)
  Stage 9: written (Tour de France 2023: Stage 9 | EXTENDED HIGHLIGHTS | 7/)
  Stage 7: written (Tour de France 2023: Stage 7 | EXTENDED HIGHLIGHTS | 7/)
  Stage 4: written (Tour de France 2023: Stage 4 | EXTENDED HIGHLIGHTS | 7/)
  Stage 3: written (Tour de France 2023: Stage 3 | EXTENDED HIGHLIGHTS | 7/)
  Stage 1: written (Tour de France 2023: Stage 1 | EXTENDED HIGHLIGHTS | 7/)

Tour de France 2023: written=9 skipped=0 unmatched=0
Found 3 candidate videos for Vuelta a Espana 2020 on NBC Sports
  Stage 14: written (Vuelta a Espana 2020: Stage 14 | EXTENDED HIGHLIGHTS | )
  Stage 6: written (Vuelta a España 2020: St

{'written': 3, 'skipped': 0, 'unmatched': 0}

In [33]:
for _, row in race_index.iterrows():
    rn, st = parse_race_name_and_stage(row['Race Name'])
    if rn == "Giro d'Italia" and row['Date'].year == 2018 and st == 16:
        print(make_label(rn, str(row['Date'])[:10], st))

2018 Giro d'Italia Stage 16


In [51]:
# 2018 giro stage 16
manual_add("3I6jQsY8o_8", "2018 Giro d'Italia Stage 16")

Fetching transcript for: 2018 Giro d'Italia Stage 16
  Video ID: 3I6jQsY8o_8
  ✓ Saved: 2,478 chars


{'label': "2018 Giro d'Italia Stage 16",
 'race_name': "Giro d'Italia",
 'race_date': '2018-05-22',
 'stage': 16,
 'video': {'video_id': '3I6jQsY8o_8',
  'title': 'Manual add: 3I6jQsY8o_8',
  'url': 'https://youtube.com/watch?v=3I6jQsY8o_8',
  'channel': 'manual',
  'published': '',
  'match_score': 999},
 'transcript': {'snippet_count': 71,
  'raw_chars': 2508,
  'clean_chars': 2478,
  'duration_mins': 4.6,
  'clean_text': "this is the Thanh Trung that the rods are facing today stage 16 the day after the rest day of course a big big day for thunder rule an even bigger day of course for Simon gates it's two minutes and 11 seconds of drift of the race leader Simon gates we know this is his favorite disciplines almost and he is off as brilliant a moment of truth everything else doesn't really matter at the moment cubed himself great position seconds bumper in the colours of champion of Australians that is a little bit of a surprise slower in fact that Fabio aru proved driving powering th

In [53]:
# ── DIAGNOSTIC: other_error files with race name + full error message ───────

from collections import Counter

other_error_entries = []
for path in sorted(RAW_DIR.glob('*.json')):
    with open(path, encoding='utf-8') as f:
        data = json.load(f)
    s = data.get('status', '')
    if s not in ('transcript_saved', 'no_video_found', 'video_found') and 'ip_blocked' not in s:
        other_error_entries.append({
            'path': path.name,
            'label': data.get('label', ''),
            'race_name': data.get('race_name', ''),
            'video_id': (data.get('video') or {}).get('video_id', ''),
            'status': s,
        })

exact_statuses = Counter(e['status'] for e in other_error_entries if not e['status'].startswith('transcript_error:\n'))
print('Exact-match statuses:')
for status, count in exact_statuses.most_common():
    print(f'  {count}  {status}')

print('\nUnclassified (generic exception) entries, with race name:')
for e in other_error_entries:
    if e['status'].startswith('transcript_error:\n') or 'Could not retrieve' in e['status']:
        print(f"  {e['video_id']:<13} | {e['label']:<45} | {e['status'][:120]!r}")

Exact-match statuses:
  54  transcript_error:transcripts_disabled
  15  transcript_error:no_transcript

Unclassified (generic exception) entries, with race name:
  H16xBJlEGVw   | 2018 La Fleche Wallonne                       | 'transcript_error:\nCould not retrieve a transcript for the video https://www.youtube.com/watch?v=H'
  Lw8apg9oXes   | 2018 Presidential Cycling Tour of Turkey Stage 1 | 'transcript_error:\nCould not retrieve a transcript for the video https://www.youtube.com/watch?v=L'
  knGb9v4dfy8   | 2018 Presidential Cycling Tour of Turkey Stage 2 | 'transcript_error:\nCould not retrieve a transcript for the video https://www.youtube.com/watch?v=k'
  uVtGaxGKdi0   | 2018 Presidential Cycling Tour of Turkey Stage 3 | 'transcript_error:\nCould not retrieve a transcript for the video https://www.youtube.com/watch?v=u'
  VG2Usdo6D2I   | 2018 Presidential Cycling Tour of Turkey Stage 4 | 'transcript_error:\nCould not retrieve a transcript for the video https://www.youtube.com/wa

In [54]:
# ── DIAGNOSTIC: duplicate race/stage files ───────────────────────────────────
# Finds cases where the same (race_name, year, stage) is represented by
# MORE THAN ONE json file - usually caused by one copy having a real date
# from race_index and another having a placeholder "{year}-01-01" date
# (different date -> different make_label() output -> different filename),
# or by race_name spelling drift (e.g. "Tour of Flanders" vs "Ronde van
# Vlaanderen" written separately for the same actual race).

from collections import defaultdict

by_race_stage = defaultdict(list)
for path in sorted(RAW_DIR.glob('*.json')):
    with open(path, encoding='utf-8') as f:
        data = json.load(f)
    key = (data.get('race_name', ''), str(data.get('race_date', ''))[:4], data.get('stage'))
    by_race_stage[key].append({
        'path': path.name,
        'race_date': data.get('race_date', ''),
        'status': data.get('status', ''),
        'video_id': (data.get('video') or {}).get('video_id', ''),
    })

dupes = {k: v for k, v in by_race_stage.items() if len(v) > 1}

print(f'{len(dupes)} (race_name, year, stage) combos with multiple files\n')
for (race_name, year, stage), files in sorted(dupes.items(), key=lambda x: (x[0][0], x[0][1] or '', x[0][2] or 0)):
    print(f'{race_name!r}  year={year}  stage={stage}')
    for f in files:
        print(f"    {f['path']:<55}  date={f['race_date']:<12}  status={f['status']:<35}  video={f['video_id']}")
    print()

0 (race_name, year, stage) combos with multiple files



In [55]:
# Confirm the VideoUnplayable except branch actually exists in your live file
import inspect
from peloton_iq.commentary.transcript import TranscriptFetcher
print(inspect.getsource(TranscriptFetcher._fetch_transcript_raw))

    def _fetch_transcript_raw(self, video_id: str) -> dict:
        """Raw transcript fetch — returns a plain dict."""
        try:
            transcript = self._ytt.fetch(video_id)
            raw_text   = " ".join([s.text for s in transcript])
            clean_text = re.sub(r"\[.*?\]", "", raw_text)
            clean_text = re.sub(r"\(.*?\)", "", clean_text)
            clean_text = re.sub(r"\s+", " ", clean_text).strip()
            duration   = round(transcript[-1].start, 0) if transcript else 0
            return {
                "success":       True,
                "video_id":      video_id,
                "snippet_count": len(transcript),
                "raw_chars":     len(raw_text),
                "clean_chars":   len(clean_text),
                "duration_mins": round(duration / 60, 1),
                "clean_text":    clean_text,
                "preview_start": clean_text[:500],
                "preview_end":   clean_text[-500:],
                "error":         Non

In [56]:
# Step 2 — reset all 73 unclassified-exception files to no_video_found
reset_count = 0
for path in sorted(RAW_DIR.glob('*.json')):
    with open(path, encoding='utf-8') as f:
        data = json.load(f)
    s = data.get('status', '')
    if s.startswith('transcript_error:') and ('Could not retrieve' in s):
        data['status'] = 'no_video_found'
        data['video'] = None
        with open(path, 'w', encoding='utf-8') as f:
            json.dump(data, f, indent=2, ensure_ascii=False)
        reset_count += 1
print(f'Reset {reset_count} files to no_video_found')

Reset 76 files to no_video_found


In [57]:
new_run = run_automated_search(max_races=None, verbose=True, threshold=75.0)

Local cache: 67,352 videos across 10 channels
Races to process: 111 (limit: 111)
———————————————————————————————————————————————————————
  [0 new | 0 found | 0 no_video]
  ✓ [145] [Velon] Inside the breakaway | Tour de Suisse 2022 Stage 2 On-B
  ✓ [145] [TNT Sports] 2022 Tour de Suisse - Stage 1 Highlights | Cycling | Eu
  ✓ [112] [TNT Sports] Eschborn - Frankfurt 2021 | Highlights | Cycling | Euro
  ✓ [78] [TNT Sports] Tour de Romandie 2021 - Stage 5 Highlights | Cycling | 
  ✓ [145] [TNT Sports] 2021 Tour de Suisse - Stage 8 | Highlights | Cycling | 
  ✓ [145] [TNT Sports] 2021 Tour de Suisse - Stage 7 | Highlights | Cycling | 
  ✓ [145] [TNT Sports] 2021 Tour de Suisse - Stage 6 | Highlights | Cycling | 
  ✓ [145] [TNT Sports] 2021 Tour de Suisse - Stage 5 | Highlights | Cycling | 
  ✓ [145] [TNT Sports] 2021 Tour de Suisse - Stage 4 | Highlights | Cycling | 
  ✓ [145] [TNT Sports] 2021 Tour de Suisse - Stage 3 | Highlights | Cycling | 
  ✓ [145] [TNT Sports] 2021 Tour de Suisse - S

In [36]:
# ── CELL 20: Write manually-identified videos for remaining gaps ────────────
# User found these by hand on YouTube - writes them as video_found records
# (no transcript fetch), same as every other tool in this notebook.

import re as _re_local


def _video_id_from_url(url: str) -> str:
    """Extract the video ID from any YouTube URL shape, including ones
    with extra query params like &t=1612s."""
    m = _re_local.search(r'[?&]v=([A-Za-z0-9_-]{11})', url)
    if m:
        return m.group(1)
    m = _re_local.search(r'youtu\.be/([A-Za-z0-9_-]{11})', url)
    if m:
        return m.group(1)
    raise ValueError(f'Could not extract video ID from: {url}')


def write_manual_stage(
    race_name: str,
    year: int,
    stage_num: int,
    url: str,
    overwrite_existing: bool = True,
) -> bool:
    """Write a single manually-identified video as a video_found record."""
    video_id = _video_id_from_url(url)

    race_date = f'{year}-01-01'
    for _, row in race_index.iterrows():
        rn, st = parse_race_name_and_stage(row['Race Name'])
        if rn == race_name and row['Date'].year == year and st == stage_num:
            race_date = str(row['Date'])[:10]
            break

    label    = make_label(race_name, race_date, stage_num)
    out_path = RAW_DIR / f'{make_safe_name(label)}.json'

    if out_path.exists() and not overwrite_existing:
        with open(out_path, encoding='utf-8') as f:
            existing = json.load(f)
        if existing.get('status') == 'transcript_saved':
            print(f'  Stage {stage_num}: skipped (already has transcript_saved)')
            return False

    record = {
        'label': label, 'race_name': race_name, 'race_date': race_date,
        'stage': stage_num,
        'video': {
            'video_id': video_id, 'title': 'manually provided',
            'url': f'https://youtube.com/watch?v={video_id}',
            'channel': 'manual', 'published': '',
            'match_score': 999,
        },
        'transcript': None,
        'status': 'video_found',
    }
    with open(out_path, 'w', encoding='utf-8') as f:
        json.dump(record, f, indent=2, ensure_ascii=False)
    print(f'  Stage {stage_num}: written ({video_id})')
    return True


# -- Tour de France 2023 - missing stages --
TDF_2023_MANUAL = {
    2:  'https://www.youtube.com/watch?v=IqSBMFGsQUg',
    5:  'https://www.youtube.com/watch?v=i-_rL0Cubxg',
    6:  'https://www.youtube.com/watch?v=pYYVF2sJzBo',
    8:  'https://www.youtube.com/watch?v=1U7SOvaobYk',
    10: 'https://www.youtube.com/watch?v=PLnbKwc9vSE',
    12: 'https://www.youtube.com/watch?v=0meC9h54VOg&t=1612s',
    13: 'https://www.youtube.com/watch?v=4HgnRUrO1l4',
    14: 'https://www.youtube.com/watch?v=4HAk-7k-PcY',
    16: 'https://www.youtube.com/watch?v=VCF9N8tdlOI',
    17: 'https://www.youtube.com/watch?v=NoT0f4DBQlY',
    18: 'https://www.youtube.com/watch?v=KS1WVQ4V-bA',
    21: 'https://www.youtube.com/watch?v=LehgoikZzW8',
}

# -- Vuelta a Espana 2020 - missing stages --
VUELTA_2020_MANUAL = {
    2:  'https://www.youtube.com/watch?v=bLVCVwlowb0',
    3:  'https://www.youtube.com/watch?v=h6uHxK-bnl0',
    4:  'https://www.youtube.com/watch?v=Z8mAGmqK-Gs',
    5:  'https://www.youtube.com/watch?v=ceSdjpH8LVA',
    7:  'https://www.youtube.com/watch?v=IC2UI_GgOJI',
    8:  'https://www.youtube.com/watch?v=LHRMIzGVasU',
    9:  'https://www.youtube.com/watch?v=18_7XGu4Qcg',
    10: 'https://www.youtube.com/watch?v=r8O_uuqw164',
    11: 'https://www.youtube.com/watch?v=Cgyy1DAJ7EI',
    12: 'https://www.youtube.com/watch?v=hSIXc3iiMKQ',
    13: 'https://www.youtube.com/watch?v=xj8cWZtj_sg',
    15: 'https://www.youtube.com/watch?v=JxJuyLW8Nuc',
    16: 'https://www.youtube.com/watch?v=WQYNvD4QBk8',
    17: 'https://www.youtube.com/watch?v=0WVHGYYuz3A',
    18: 'https://www.youtube.com/watch?v=oXOt6PLaJoY',
}

print(f'Writing {len(TDF_2023_MANUAL)} Tour de France 2023 stages...')
tdf_written = 0
for stage_num, url in sorted(TDF_2023_MANUAL.items()):
    if write_manual_stage('Tour de France', 2023, stage_num, url):
        tdf_written += 1

print(f'\nWriting {len(VUELTA_2020_MANUAL)} Vuelta a Espana 2020 stages...')
vuelta_written = 0
for stage_num, url in sorted(VUELTA_2020_MANUAL.items()):
    if write_manual_stage('Vuelta a Espana', 2020, stage_num, url):
        vuelta_written += 1

print(f'\n{"="*60}')
print(f'Tour de France 2023: {tdf_written}/{len(TDF_2023_MANUAL)} written')
print(f'Vuelta a Espana 2020: {vuelta_written}/{len(VUELTA_2020_MANUAL)} written')

Writing 12 Tour de France 2023 stages...
  Stage 2: written (IqSBMFGsQUg)
  Stage 5: written (i-_rL0Cubxg)
  Stage 6: written (pYYVF2sJzBo)
  Stage 8: written (1U7SOvaobYk)
  Stage 10: written (PLnbKwc9vSE)
  Stage 12: written (0meC9h54VOg)
  Stage 13: written (4HgnRUrO1l4)
  Stage 14: written (4HAk-7k-PcY)
  Stage 16: written (VCF9N8tdlOI)
  Stage 17: written (NoT0f4DBQlY)
  Stage 18: written (KS1WVQ4V-bA)
  Stage 21: written (LehgoikZzW8)

Writing 15 Vuelta a Espana 2020 stages...
  Stage 2: written (bLVCVwlowb0)
  Stage 3: written (h6uHxK-bnl0)
  Stage 4: written (Z8mAGmqK-Gs)
  Stage 5: written (ceSdjpH8LVA)
  Stage 7: written (IC2UI_GgOJI)
  Stage 8: written (LHRMIzGVasU)
  Stage 9: written (18_7XGu4Qcg)
  Stage 10: written (r8O_uuqw164)
  Stage 11: written (Cgyy1DAJ7EI)
  Stage 12: written (hSIXc3iiMKQ)
  Stage 13: written (xj8cWZtj_sg)
  Stage 15: written (JxJuyLW8Nuc)
  Stage 16: written (WQYNvD4QBk8)
  Stage 17: written (0WVHGYYuz3A)
  Stage 18: written (oXOt6PLaJoY)

Tour de 

In [37]:
for _, row in race_index.iterrows():
    rn, st = parse_race_name_and_stage(row['Race Name'])
    if "Ronde" in rn:
        print(rn)
        print(make_label(rn, str(row['Date'])[:10], st))

Ronde van Vlaanderen - Tour des Flandres
2023 Ronde van Vlaanderen - Tour des Flandres
Ronde van Vlaanderen - Tour des Flandres
2022 Ronde van Vlaanderen - Tour des Flandres
Ronde van Vlaanderen - Tour des Flandres
2021 Ronde van Vlaanderen - Tour des Flandres
Ronde van Vlaanderen
2020 Ronde van Vlaanderen
Ronde Van Vlaanderen
2019 Ronde Van Vlaanderen
Ronde Van Vlaanderen
2018 Ronde Van Vlaanderen
Ronde Van Vlaanderen
2017 Ronde Van Vlaanderen


In [41]:
for _, row in race_index.iterrows():
    rn, st = parse_race_name_and_stage(row['Race Name'])
    if "Liege" in rn:
        print(rn)
        print(make_label(rn, str(row['Date'])[:10], st))

Liege-Bastogne-Liege
2023 Liege-Bastogne-Liege
Liege - Bastogne - Liege
2022 Liege - Bastogne - Liege
Liege - Bastogne - Liege
2021 Liege - Bastogne - Liege
Liege-Bastogne-Liege
2020 Liege-Bastogne-Liege
Liege - Bastogne - Liege
2017 Liege - Bastogne - Liege


In [42]:
# ── CELL 21 (v2): One-day race writer — per-year race name lookup ───────────
# CORRECTED based on actual race_index contents:
#   - "Ronde van Vlaanderen" / Tour of Flanders changes name+case by year:
#       2017-2019: "Ronde Van Vlaanderen"  (capital V)
#       2020:      "Ronde van Vlaanderen"  (lowercase v, no subtitle)
#       2021-2023: "Ronde van Vlaanderen - Tour des Flandres"
#   - "Liege-Bastogne-Liege" only exists in race_index for 2020 and 2023.
#     2017/2018/2019/2021/2022 have NO row at all - writing these still
#     works (creates a record with a placeholder date) but there's no
#     race_index entry to attach a real date to.
#   - Paris-Roubaix is consistently "Paris-Roubaix" across all years.

ONEDAY_NAME_BY_YEAR = {
    'Tour of Flanders': {
        2017: 'Ronde Van Vlaanderen',
        2018: 'Ronde Van Vlaanderen',
        2019: 'Ronde Van Vlaanderen',
        2020: 'Ronde van Vlaanderen',
        2021: 'Ronde van Vlaanderen - Tour des Flandres',
        2022: 'Ronde van Vlaanderen - Tour des Flandres',
        2023: 'Ronde van Vlaanderen - Tour des Flandres',
    },
    'Liege-Bastogne-Liege': {
        2017: 'Liege-Bastogne-Liege',
        2018: 'Liege-Bastogne-Liege',
        2019: 'Liege-Bastogne-Liege',
        2020: 'Liege-Bastogne-Liege',
        2021: 'Liege-Bastogne-Liege',
        2022: 'Liege-Bastogne-Liege',
        2023: 'Liege-Bastogne-Liege',
    },
}


def write_oneday_manual(
    canonical_race_name: str,
    year: int,
    url: str,
    overwrite_existing: bool = True,
) -> bool:
    """
    Write a single manually-identified one-day race video as video_found.
    canonical_race_name is looked up against ONEDAY_NAME_BY_YEAR to get
    the actual race_index spelling for that year, if registered. Falls
    back to canonical_race_name as-is for races with no name variance.
    """
    video_id = _video_id_from_url(url)

    actual_race_name = ONEDAY_NAME_BY_YEAR.get(canonical_race_name, {}).get(year, canonical_race_name)

    race_date = f'{year}-01-01'
    found_in_index = False
    for _, row in race_index.iterrows():
        rn, st = parse_race_name_and_stage(row['Race Name'])
        if rn == actual_race_name and row['Date'].year == year and st is None:
            race_date = str(row['Date'])[:10]
            found_in_index = True
            break

    label    = make_label(actual_race_name, race_date, None)
    out_path = RAW_DIR / f'{make_safe_name(label)}.json'

    if out_path.exists() and not overwrite_existing:
        with open(out_path, encoding='utf-8') as f:
            existing = json.load(f)
        if existing.get('status') == 'transcript_saved':
            print(f'  {year}: skipped (already has transcript_saved)')
            return False

    record = {
        'label': label, 'race_name': actual_race_name, 'race_date': race_date,
        'stage': None,
        'video': {
            'video_id': video_id, 'title': 'manually provided',
            'url': f'https://youtube.com/watch?v={video_id}',
            'channel': 'manual', 'published': '',
            'match_score': 999,
        },
        'transcript': None,
        'status': 'video_found',
    }
    with open(out_path, 'w', encoding='utf-8') as f:
        json.dump(record, f, indent=2, ensure_ascii=False)

    flag = '' if found_in_index else '  [NOT IN race_index - placeholder date used]'
    print(f'  {year}: written ({video_id})  -> label="{label}"{flag}')
    return True


def write_oneday_race_batch(canonical_race_name: str, year_to_url: dict) -> int:
    print(f'Writing {len(year_to_url)} years for {canonical_race_name}...')
    count = 0
    for year, url in sorted(year_to_url.items()):
        if write_oneday_manual(canonical_race_name, year, url):
            count += 1
    print(f'{canonical_race_name}: {count}/{len(year_to_url)} written\n')
    return count


# -- Paris-Roubaix --
PARIS_ROUBAIX = {
    2017: 'https://www.youtube.com/watch?v=PO683yZBISc',
    2018: 'https://www.youtube.com/watch?v=8ygpaL0xfbA',
    2019: 'https://www.youtube.com/watch?v=-cfIjaCZ9NM',
    2021: 'https://www.youtube.com/watch?v=1qMivd4Wyeo',
    2022: 'https://www.youtube.com/watch?v=R9vh215-wsI',
    2023: 'https://www.youtube.com/watch?v=L10sAuX4pP4&t=4s',
}

# -- Tour of Flanders --
TOUR_OF_FLANDERS = {
    2017: 'https://www.youtube.com/watch?v=t6rWbpRdpeQ',
    2018: 'https://www.youtube.com/watch?v=5EryZIoVTR8',
    2019: 'https://www.youtube.com/watch?v=-EimKtU_xAY',
    2020: 'https://www.youtube.com/watch?v=-aF5kkmWG1Y',
    2021: 'https://www.youtube.com/watch?v=eh6i6YVBCm0',
    2022: 'https://www.youtube.com/watch?v=wwoG2ls_r4Y',
    2023: 'https://www.youtube.com/watch?v=QrrlLDybH6Y',
}

# -- Liege-Bastogne-Liege --
LIEGE_BASTOGNE_LIEGE = {
    2017: 'https://www.youtube.com/watch?v=qOxOQKZXlMU',
    2018: 'https://www.youtube.com/watch?v=9FffrlwNAa8',
    2019: 'https://www.youtube.com/watch?v=xGWzyg3FFj0',
    2020: 'https://www.youtube.com/watch?v=xgCWnAqjIAU',
    2021: 'https://www.youtube.com/watch?v=wUekxHVf13E',
    2022: 'https://www.youtube.com/watch?v=mqLmGRu7FaA',
    2023: 'https://www.youtube.com/watch?v=_JXI_8rxZqI',
}

write_oneday_race_batch('Paris-Roubaix', PARIS_ROUBAIX)
write_oneday_race_batch('Tour of Flanders', TOUR_OF_FLANDERS)
write_oneday_race_batch('Liege-Bastogne-Liege', LIEGE_BASTOGNE_LIEGE)

Writing 6 years for Paris-Roubaix...
  2017: written (PO683yZBISc)  -> label="2017 Paris-Roubaix"  [NOT IN race_index - placeholder date used]
  2018: written (8ygpaL0xfbA)  -> label="2018 Paris-Roubaix"  [NOT IN race_index - placeholder date used]
  2019: written (-cfIjaCZ9NM)  -> label="2019 Paris-Roubaix"
  2021: written (1qMivd4Wyeo)  -> label="2021 Paris-Roubaix"
  2022: written (R9vh215-wsI)  -> label="2022 Paris-Roubaix"
  2023: written (L10sAuX4pP4)  -> label="2023 Paris-Roubaix"
Paris-Roubaix: 6/6 written

Writing 7 years for Tour of Flanders...
  2017: written (t6rWbpRdpeQ)  -> label="2017 Ronde Van Vlaanderen"
  2018: written (5EryZIoVTR8)  -> label="2018 Ronde Van Vlaanderen"
  2019: written (-EimKtU_xAY)  -> label="2019 Ronde Van Vlaanderen"
  2020: written (-aF5kkmWG1Y)  -> label="2020 Ronde van Vlaanderen"
  2021: written (eh6i6YVBCm0)  -> label="2021 Ronde van Vlaanderen - Tour des Flandres"
  2022: written (wwoG2ls_r4Y)  -> label="2022 Ronde van Vlaanderen - Tour des F

7

In [39]:
for name in ['Paris-Roubaix', 'Tour of Flanders', 'Ronde van Vlaanderen', 'Liege-Bastogne-Liege', "Liege-Bastogne-Liege"]:
      matches = race_index[race_index['Race Name'].str.contains(name, case=False, na=False, regex=False)]
      print(name, '->', matches['Race Name'].unique()[:3])

Paris-Roubaix -> <ArrowStringArray>
['2023 Paris-Roubaix', '2022 Paris-Roubaix', '2021 Paris-Roubaix']
Length: 3, dtype: str
Tour of Flanders -> <ArrowStringArray>
[]
Length: 0, dtype: str
Ronde van Vlaanderen -> <ArrowStringArray>
['2023 Ronde van Vlaanderen - Tour des Flandres',
 '2022 Ronde van Vlaanderen - Tour des Flandres',
 '2021 Ronde van Vlaanderen - Tour des Flandres']
Length: 3, dtype: str
Liege-Bastogne-Liege -> <ArrowStringArray>
['2023 Liege-Bastogne-Liege', '2020 Liege-Bastogne-Liege']
Length: 2, dtype: str
Liege-Bastogne-Liege -> <ArrowStringArray>
['2023 Liege-Bastogne-Liege', '2020 Liege-Bastogne-Liege']
Length: 2, dtype: str


In [40]:
for _, row in race_index.iterrows():
    rn, st = parse_race_name_and_stage(row['Race Name'])
    if 'dauphine' in rn.lower() and row['Date'].year == 2018:
        print(st, row['Race Name'], row['Date'])

7 2018 Criterium du Dauphine Stage 7 2018-06-10 00:00:00
6 2018 Criterium du Dauphine Stage 6 2018-06-09 00:00:00
5 2018 Criterium du Dauphine Stage 5 2018-06-08 00:00:00
4 2018 Criterium du Dauphine Stage 4 2018-06-07 00:00:00
3 2018 Criterium du Dauphine Stage 3 2018-06-06 00:00:00
2 2018 Criterium du Dauphine Stage 2 2018-06-05 00:00:00
1 2018 Criterium du Dauphine Stage 1 2018-06-04 00:00:00


In [43]:
# ── CELL 22: Criterium du Dauphine 2017/2018 — individual videos ────────────
# 2017 and 2018 are individual stage videos (not playlists) - use the same
# write_manual_stage() from CELL 20. 2018's prologue has NO matching row in
# race_index (confirmed: only stages 1-7 exist there) - written anyway with
# stage=0 and a placeholder date, per instruction.

DAUPHINE_2017 = {
    1: 'https://www.youtube.com/watch?v=QrfmDk_OU1E',
    2: 'https://www.youtube.com/watch?v=eJpcXEACn78',
    3: 'https://www.youtube.com/watch?v=39iu6Q0-yrg',
    4: 'https://www.youtube.com/watch?v=Puze4oCx4ks',
    5: 'https://www.youtube.com/watch?v=e7Xr-EPMtjo',
    6: 'https://www.youtube.com/watch?v=_9C4av45oSk',
    7: 'https://www.youtube.com/watch?v=wpz9JVudLUo',
    8: 'https://www.youtube.com/watch?v=MjQDdchItlA',
}

DAUPHINE_2018 = {
    0: 'https://www.youtube.com/watch?v=soAyZMf2COI',
    1: 'https://www.youtube.com/watch?v=jJHwK4dgntI',
    2: 'https://www.youtube.com/watch?v=tUAzxuG8Euk',
    3: 'https://www.youtube.com/watch?v=Kp7ZFNSHxKM',
    4: 'https://www.youtube.com/watch?v=pOVFpPa99Xw',
    5: 'https://www.youtube.com/watch?v=fetCrttnw6g',
    6: 'https://www.youtube.com/watch?v=nlE4grpyfP8',
    7: 'https://www.youtube.com/watch?v=H_9enhZILDU',
}

print('Writing Criterium du Dauphine 2017...')
written_2017 = 0
for stage_num, url in sorted(DAUPHINE_2017.items()):
    if write_manual_stage('Criterium du Dauphine', 2017, stage_num, url):
        written_2017 += 1

print(f'\nWriting Criterium du Dauphine 2018 (including prologue, stage=0)...')
written_2018 = 0
for stage_num, url in sorted(DAUPHINE_2018.items()):
    if write_manual_stage('Criterium du Dauphine', 2018, stage_num, url):
        written_2018 += 1
    if stage_num == 0:
        print('    ^ prologue - no race_index match, placeholder date used')

print(f'\n2017: {written_2017}/{len(DAUPHINE_2017)} written')
print(f'2018: {written_2018}/{len(DAUPHINE_2018)} written')

Writing Criterium du Dauphine 2017...
  Stage 1: written (QrfmDk_OU1E)
  Stage 2: written (eJpcXEACn78)
  Stage 3: written (39iu6Q0-yrg)
  Stage 4: written (Puze4oCx4ks)
  Stage 5: written (e7Xr-EPMtjo)
  Stage 6: written (_9C4av45oSk)
  Stage 7: written (wpz9JVudLUo)
  Stage 8: written (MjQDdchItlA)

Writing Criterium du Dauphine 2018 (including prologue, stage=0)...
  Stage 0: written (soAyZMf2COI)
    ^ prologue - no race_index match, placeholder date used
  Stage 1: written (jJHwK4dgntI)
  Stage 2: written (tUAzxuG8Euk)
  Stage 3: written (Kp7ZFNSHxKM)
  Stage 4: written (pOVFpPa99Xw)
  Stage 5: written (fetCrttnw6g)
  Stage 6: written (nlE4grpyfP8)
  Stage 7: written (H_9enhZILDU)

2017: 8/8 written
2018: 8/8 written


In [44]:
write_playlist_simple('https://www.youtube.com/playlist?list=PL-IWSyN8d4qXd7i7HJvWFpo4ohR18vRoo', 'Criterium du Dauphine', 2019)
write_playlist_simple('https://www.youtube.com/playlist?list=PL15dcRLXc3-xZTp4HclxzVbnz0Ee0wPmC', 'Criterium du Dauphine', 2020)
write_playlist_simple('https://www.youtube.com/playlist?list=PL2HVrp8_RMdBT1XH72Wq3bw-rC5n2WcZA', 'Criterium du Dauphine', 2021)
write_playlist_simple('https://www.youtube.com/playlist?list=PLwHDQSzvvLgV2lBNuRAV6fNf_uiKWQwzP', 'Criterium du Dauphine', 2022)
write_playlist_simple('https://www.youtube.com/playlist?list=PLscKx95e0gBT2LEZjicGoyAcJRJds_6rL', 'Criterium du Dauphine', 2023)

Found 8 videos in playlist for Criterium du Dauphine 2019
  Stage 1: written (Criterium du Dauphine 2019: Stage 1 highlights | NBC Sp)
  Stage 2: written (Criterium du Dauphine 2019: Stage 2 highlights | NBC Sp)
  Stage 3: written (Critérium du Dauphiné 2019: Stage 3 | EXTENDED HIGHLIGH)
  Stage 4: written (Critérium du Dauphiné 2019: Stage 4 | EXTENDED HIGHLIGH)
  Stage 5: written (Criterium du Dauphine 2019: Stage 5 | EXTENDED HIGHLIGH)
  Stage 6: written (Critérium du Dauphiné 2019: Stage 6 | EXTENDED HIGHLIGH)
  Stage 7: written (Criterium du Dauphine Stage 7 2019: Stage 7 | EXTENDED )
  Stage 8: written (Criterium du Dauphine 2019: Stage 8 | EXTENDED HIGHLIGH)

Criterium du Dauphine 2019: written=8 skipped=0 unmatched=0
Found 6 videos in playlist for Criterium du Dauphine 2020
  [no stage found] We have no idea what we're doing | SWEET BOYS #1
  Stage 1: written (Criterium du Dauphine 2020: Stage 1 | EXTENDED HIGHLIGH)
  Stage 2: written (Criterium du Dauphine 2020: Stage 2 | EXTE

{'written': 8, 'skipped': 0, 'unmatched': 0}

In [46]:
for path in sorted(RAW_DIR.glob('*.json')):
    with open(path, encoding='utf-8') as f:
        data = json.load(f)
    s = data.get('status', '')
    if s.startswith('transcript_error:') and 'Could not retrieve' in s:
        print(data['video']['video_id'], '|', data.get('label'))

H16xBJlEGVw | 2018 La Fleche Wallonne
Lw8apg9oXes | 2018 Presidential Cycling Tour of Turkey Stage 1
knGb9v4dfy8 | 2018 Presidential Cycling Tour of Turkey Stage 2
uVtGaxGKdi0 | 2018 Presidential Cycling Tour of Turkey Stage 3
VG2Usdo6D2I | 2018 Presidential Cycling Tour of Turkey Stage 4
6XaFazvzAfA | 2018 Presidential Cycling Tour of Turkey Stage 5
wcoHI8nDEC0 | 2018 Presidential Cycling Tour of Turkey Stage 6
oc1B9EJMnTs | 2018 Volta Ciclista a Catalunya Stage 1
4W05rb7dP_0 | 2018 Volta Ciclista a Catalunya Stage 2
xYcODR8ZtsE | 2018 Volta Ciclista a Catalunya Stage 3
ZL8pLoRoeC0 | 2018 Volta Ciclista a Catalunya Stage 4
qiSFxuq56OQ | 2018 Volta Ciclista a Catalunya Stage 5
VcqvJvchQTA | 2018 Volta Ciclista a Catalunya Stage 6
itF2j35lWeY | 2018 Volta Ciclista a Catalunya Stage 7
Yi4opDanurU | 2019 Amstel Gold Race
caNGrKBe42c | 2019 Itzulia Basque Country Stage 1
caNGrKBe42c | 2019 Itzulia Basque Country Stage 2
caNGrKBe42c | 2019 Itzulia Basque Country Stage 3
5j2pGuw1qxs | 2019 L

In [47]:
from youtube_transcript_api import YouTubeTranscriptApi
ytt = YouTubeTranscriptApi()
try:
    ytt.fetch("H16xBJlEGVw")
except Exception as e:
    print(type(e).__name__)
    print(e)

VideoUnplayable

Could not retrieve a transcript for the video https://www.youtube.com/watch?v=H16xBJlEGVw! This is most likely caused by:

The video is unplayable for the following reason: The uploader has not made this video available in your country

If you are sure that the described cause is not responsible for this error and that a transcript should be retrievable, please create an issue at https://github.com/jdepoix/youtube-transcript-api/issues. Please add which version of youtube_transcript_api you are using and provide the information needed to replicate the error. Also make sure that there are no open issues which already describe your problem!


In [58]:
from collections import Counter

other_error_entries = []
for path in sorted(RAW_DIR.glob('*.json')):
    with open(path, encoding='utf-8') as f:
        data = json.load(f)
    s = data.get('status', '')
    if s not in ('transcript_saved', 'no_video_found', 'video_found') and 'ip_blocked' not in s:
        other_error_entries.append({
            'path': path.name,
            'label': data.get('label', ''),
            'video_id': (data.get('video') or {}).get('video_id', ''),
            'status': s,
        })

exact_statuses = Counter(e['status'] for e in other_error_entries if not e['status'].startswith('transcript_error:\n'))
print('Exact-match statuses:')
for status, count in exact_statuses.most_common():
    print(f'  {count}  {status}')

print('\nStill-unclassified entries, with label:')
for e in other_error_entries:
    if e['status'].startswith('transcript_error:\n') or 'Could not retrieve' in e['status']:
        print(f"  {e['video_id']:<13} | {e['label']:<45} | {e['status'][:120]!r}")

Exact-match statuses:
  54  transcript_error:transcripts_disabled
  15  transcript_error:no_transcript
  1  transcript_error:video_unplayable_geo_restricted

Still-unclassified entries, with label:


In [87]:
# ── DIAGNOSTIC: full list of races/stages with no successful transcript ─────
# Groups everything that ISN'T transcript_saved into clear buckets so you
# can see exactly what needs a playlist hunt vs. a manual single-video add.
#
#   NO_VIDEO       - never matched anything at all. Needs a playlist or
#                     manual video search from scratch.
#   NO_TRANSCRIPT  - a video WAS matched, but it has no captions
#                     (no_transcript / transcripts_disabled). The matched
#                     video itself is unusable - needs a DIFFERENT video
#                     for this race/stage.
#   GEO_RESTRICTED - a video WAS matched, but it's blocked in your region.
#                     Needs a different, non-region-locked video.
#   STILL_QUEUED   - matched, not yet attempted (video_found). Not a
#                     problem, just hasn't been fetched yet - informational.
#   UNCLASSIFIED   - anything else (shouldn't be much/any of this left).

from collections import defaultdict

NO_VIDEO       = defaultdict(list)
NO_TRANSCRIPT  = defaultdict(list)
GEO_RESTRICTED = defaultdict(list)
STILL_QUEUED   = defaultdict(list)
UNCLASSIFIED   = defaultdict(list)

for path in sorted(RAW_DIR.glob('*.json')):
    with open(path, encoding='utf-8') as f:
        data = json.load(f)

    status    = data.get('status', '')
    label     = data.get('label', path.stem)
    race_name = data.get('race_name', 'Unknown') or 'Unknown'

    if status == 'transcript_saved':
        continue
    elif status == 'no_video_found':
        NO_VIDEO[race_name].append(label)
    elif status == 'transcript_error:no_transcript':
        NO_TRANSCRIPT[race_name].append(label)
    elif status == 'transcript_error:transcripts_disabled':
        NO_TRANSCRIPT[race_name].append(label)
    elif status == 'transcript_error:video_unplayable_geo_restricted':
        GEO_RESTRICTED[race_name].append(label)
    elif status == 'video_found':
        STILL_QUEUED[race_name].append(label)
    else:
        UNCLASSIFIED[race_name].append((label, status))


def _print_group(title: str, group: dict, show_status: bool = False):
    total = sum(len(v) for v in group.values())
    print(f'\n{"="*70}')
    print(f'{title} - {total} total')
    print(f'{"="*70}')
    for race_name, items in sorted(group.items(), key=lambda kv: -len(kv[1])):
        print(f'\n  {race_name} ({len(items)})')
        for item in sorted(items, key=lambda x: x[0] if isinstance(x, tuple) else x):
            if show_status:
                label, status = item
                print(f'    {label}  [{status[:60]}]')
            else:
                print(f'    {item}')


_print_group('NO VIDEO FOUND - needs a playlist or manual video search', NO_VIDEO)
_print_group('MATCHED BUT NO TRANSCRIPT - needs a DIFFERENT video', NO_TRANSCRIPT)
_print_group('GEO-RESTRICTED - needs a non-region-locked video', GEO_RESTRICTED)
_print_group('STILL QUEUED (not yet fetched - informational only)', STILL_QUEUED)
_print_group('UNCLASSIFIED (unexpected - investigate)', UNCLASSIFIED, show_status=True)

print(f'\n\n{"="*70}')
print('SUMMARY')
print(f'{"="*70}')
print(f'No video found:            {sum(len(v) for v in NO_VIDEO.values())}')
print(f'No transcript (bad video): {sum(len(v) for v in NO_TRANSCRIPT.values())}')
print(f'Geo-restricted:            {sum(len(v) for v in GEO_RESTRICTED.values())}')
print(f'Still queued:              {sum(len(v) for v in STILL_QUEUED.values())}')
print(f'Unclassified:              {sum(len(v) for v in UNCLASSIFIED.values())}')
print(f'\nTOTAL needing attention: '
      f'{sum(len(v) for v in NO_VIDEO.values()) + sum(len(v) for v in NO_TRANSCRIPT.values()) + sum(len(v) for v in GEO_RESTRICTED.values()) + sum(len(v) for v in UNCLASSIFIED.values())}')
print('(excludes "still queued" since those just need a normal fetch run, not manual sourcing)')


NO VIDEO FOUND - needs a playlist or manual video search - 23 total

  BinckBank Tour (3)
    2017 BinckBank Tour Stage 4
    2017 BinckBank Tour Stage 5
    2017 BinckBank Tour Stage 6

  Eschborn-Frankfurt (3)
    2018 Eschborn-Frankfurt
    2019 Eschborn-Frankfurt
    2023 Eschborn-Frankfurt

  Itzulia Basque Country (3)
    2019 Itzulia Basque Country Stage 4
    2019 Itzulia Basque Country Stage 5
    2019 Itzulia Basque Country Stage 6

  Presidential Cycling Tour of Turkey (3)
    2019 Presidential Cycling Tour of Turkey Stage 4
    2019 Presidential Cycling Tour of Turkey Stage 5
    2019 Presidential Cycling Tour of Turkey Stage 6

  Bretagne Classic - Ouest France (2)
    2017 Bretagne Classic - Ouest France
    2019 Bretagne Classic - Ouest France

  Volta Ciclista a Catalunya (2)
    2017 Volta Ciclista a Catalunya Stage 7
    2019 Volta Ciclista a Catalunya Stage 7

  Bretagne Classic - Ouest-France (2)
    2018 Bretagne Classic - Ouest-France
    2021 Bretagne Classic - 

In [70]:
# ── DIAGNOSTIC: Tirreno-Adriatico spelling by year ───────────────────────────

for _, row in race_index.iterrows():
    rn, st = parse_race_name_and_stage(row['Race Name'])
    if 'tirreno' in rn.lower():
        print(f"{row['Date'].year}  {rn!r}  stage={st}")

2023  'Tirreno-Adriatico'  stage=7
2023  'Tirreno-Adriatico'  stage=6
2023  'Tirreno-Adriatico'  stage=5
2023  'Tirreno-Adriatico'  stage=4
2023  'Tirreno-Adriatico'  stage=3
2023  'Tirreno-Adriatico'  stage=2
2023  'Tirreno-Adriatico'  stage=1
2022  'Tirreno-Adriatico'  stage=7
2022  'Tirreno-Adriatico'  stage=6
2022  'Tirreno-Adriatico'  stage=5
2022  'Tirreno-Adriatico'  stage=4
2022  'Tirreno-Adriatico'  stage=3
2022  'Tirreno-Adriatico'  stage=2
2022  'Tirreno-Adriatico'  stage=1
2021  'Tirreno-Adriatico'  stage=6
2021  'Tirreno-Adriatico'  stage=5
2021  'Tirreno-Adriatico'  stage=4
2021  'Tirreno-Adriatico'  stage=3
2021  'Tirreno-Adriatico'  stage=2
2021  'Tirreno-Adriatico'  stage=1
2020  'Tirreno Adriatico'  stage=7
2020  'Tirreno Adriatico'  stage=6
2020  'Tirreno Adriatico'  stage=5
2020  'Tirreno Adriatico'  stage=4
2020  'Tirreno Adriatico'  stage=3
2020  'Tirreno Adriatico'  stage=2
2020  'Tirreno Adriatico'  stage=1
2019  'Tirreno-Adriatico'  stage=6
2019  'Tirreno-Adria

In [61]:
# ── DIAGNOSTIC: exact Paris-Nice spelling in race_index by year ─────────────

for _, row in race_index.iterrows():
    rn, st = parse_race_name_and_stage(row['Race Name'])
    if 'nice' in rn.lower() and 'paris' in rn.lower():
        print(f"{row['Date'].year}  {rn!r}  stage={st}")

2023  'Paris-Nice'  stage=8
2023  'Paris-Nice'  stage=7
2023  'Paris-Nice'  stage=5
2023  'Paris-Nice'  stage=4
2023  'Paris-Nice'  stage=3
2023  'Paris-Nice'  stage=2
2023  'Paris-Nice'  stage=1
2022  'Paris-Nice'  stage=8
2022  'Paris-Nice'  stage=7
2022  'Paris-Nice'  stage=6
2022  'Paris-Nice'  stage=5
2022  'Paris-Nice'  stage=4
2022  'Paris-Nice'  stage=3
2022  'Paris-Nice'  stage=2
2022  'Paris-Nice'  stage=1
2021  'Paris-Nice'  stage=8
2021  'Paris-Nice'  stage=7
2021  'Paris-Nice'  stage=6
2021  'Paris-Nice'  stage=5
2021  'Paris-Nice'  stage=4
2021  'Paris-Nice'  stage=3
2021  'Paris-Nice'  stage=2
2021  'Paris-Nice'  stage=1
2020  'Paris - Nice'  stage=7
2020  'Paris - Nice'  stage=6
2020  'Paris - Nice'  stage=5
2020  'Paris - Nice'  stage=4
2020  'Paris - Nice'  stage=3
2020  'Paris - Nice'  stage=2
2020  'Paris - Nice'  stage=1
2019  'Paris-Nice'  stage=8
2019  'Paris-Nice'  stage=7
2019  'Paris-Nice'  stage=6
2019  'Paris-Nice'  stage=5
2019  'Paris-Nice'  stage=4
2019  

In [64]:
# ── CELL 23: Batch fill — Paris-Nice, Turkey, Catalunya, Pologne, Suisse, Tirreno ──
# Uses existing tools: write_playlist_simple() for playlists, write_manual_stage()
# for individual videos. All write video_found, NO transcript fetch — hand off
# to the CLI afterward as usual.

# -- Paris-Nice -- playlists --
write_playlist_simple('https://www.youtube.com/playlist?list=PL0Q2W1AG8GGpBCdBIzlMB2-VXluHf0OyK', 'Paris-Nice', 2017)
write_playlist_simple('https://www.youtube.com/playlist?list=PLg3WzTG3LfYbovXCZbeFDssFwHZFLRIxQ', 'Paris-Nice', 2018)
write_playlist_simple('https://www.youtube.com/playlist?list=PLg3WzTG3LfYacMoVZL20gM5avt73LFQpn', 'Paris-Nice', 2019)
write_playlist_simple('https://www.youtube.com/playlist?list=PLdlaXyQVoS-_15DUdG30ZwJ1HYVs5oDk8', 'Paris - Nice', 2020)
# NOTE: race_index spells this "Paris - Nice" (spaces around hyphen) for
# 2020 ONLY — every other year (2017-2019, 2021-2023) uses "Paris-Nice"
# with no spaces. Confirmed via direct race_index inspection. Using the
# wrong spelling here would create a new orphaned file instead of fixing
# the actual 2020 gap.

# -- Paris-Nice 2021 -- individual stage videos (no playlist) --
PARIS_NICE_2021 = {
    1: 'https://www.youtube.com/watch?v=jLk0Nx9o8vg',
    2: 'https://www.youtube.com/watch?v=_GH8TZ-UEKc',
    3: 'https://www.youtube.com/watch?v=9VDjpXhRSJg',
    4: 'https://www.youtube.com/watch?v=lbw8rZSEx8Y',
    5: 'https://www.youtube.com/watch?v=UAXuHtZDICk',
    6: 'https://www.youtube.com/watch?v=uK1tO5jwZOE',
    7: 'https://www.youtube.com/watch?v=_8oxBHo4RYE',
    8: 'https://www.youtube.com/watch?v=htBH_RauYjs',
}
print('\nWriting Paris-Nice 2021...')
for stage_num, url in sorted(PARIS_NICE_2021.items()):
    write_manual_stage('Paris-Nice', 2021, stage_num, url)

# -- Presidential Cycling Tour of Turkey 2017 -- individual videos --
TURKEY_2017 = {
    1: 'https://www.youtube.com/watch?v=Nj3qj9RaMmM',
    2: 'https://www.youtube.com/watch?v=O36Ydh8yZOI',
    3: 'https://www.youtube.com/watch?v=cxnQ8OPxdkQ',
    4: 'https://www.youtube.com/watch?v=afZpsbEA65A',
    5: 'https://www.youtube.com/watch?v=yZpA4LOnwKY',
    6: 'https://www.youtube.com/watch?v=7dBQj-9RE-A',
}
print('\nWriting Presidential Cycling Tour of Turkey 2017...')
for stage_num, url in sorted(TURKEY_2017.items()):
    write_manual_stage('Presidential Cycling Tour of Turkey', 2017, stage_num, url)

# -- Volta Ciclista a Catalunya 2017 -- individual videos --
# NOTE: Stage 1 and 2 share the SAME video — written twice, once per stage,
# since the dataset tracks them as separate stage rows.
CATALUNYA_2017 = {
    1: 'https://www.youtube.com/watch?v=Xkdb4UbakqE',
    2: 'https://www.youtube.com/watch?v=Xkdb4UbakqE',
    3: 'https://www.youtube.com/watch?v=hS1PkHbsyiQ',
    5: 'https://www.youtube.com/watch?v=kGeYfMBeMnI',
    6: 'https://www.youtube.com/watch?v=S0Xg4WvfATI',
}
print('\nWriting Volta Ciclista a Catalunya 2017...')
for stage_num, url in sorted(CATALUNYA_2017.items()):
    write_manual_stage('Volta Ciclista a Catalunya', 2017, stage_num, url)

# -- Tour de Pologne 2017 -- individual videos --
POLOGNE_2017 = {
    2: 'https://www.youtube.com/watch?v=S2rFlaMj1pg',
    4: 'https://www.youtube.com/watch?v=VbgX64ri0zc',
}
print('\nWriting Tour de Pologne 2017...')
for stage_num, url in sorted(POLOGNE_2017.items()):
    write_manual_stage('Tour de Pologne', 2017, stage_num, url)

# -- Tour de Suisse 2018 -- individual videos --
SUISSE_2018 = {
    7: 'https://www.youtube.com/watch?v=-ZfrHdE3n0E',
    8: 'https://www.youtube.com/watch?v=XPn_IrIVKbo',
    9: 'https://www.youtube.com/watch?v=PCC214dxdGw',
}
print('\nWriting Tour de Suisse 2018...')
for stage_num, url in sorted(SUISSE_2018.items()):
    write_manual_stage('Tour de Suisse', 2018, stage_num, url)

# -- Tour de Suisse 2019 -- individual videos --
SUISSE_2019 = {
    3: 'https://www.youtube.com/watch?v=i4bmhimGYH0',
    4: 'https://www.youtube.com/watch?v=C35phk9YbBw',
    7: 'https://www.youtube.com/watch?v=IaObekNouTs',
    9: 'https://www.youtube.com/watch?v=EPnJFWK5XOc',
}
print('\nWriting Tour de Suisse 2019...')
for stage_num, url in sorted(SUISSE_2019.items()):
    write_manual_stage('Tour de Suisse', 2019, stage_num, url)

# -- Tirreno-Adriatico -- playlists --
write_playlist_simple('https://www.youtube.com/playlist?list=PLY_BzaA1TIAeBfhZPInoNdgGKYtnzCgRc', 'Tirreno-Adriatico', 2017)
write_playlist_simple('https://www.youtube.com/playlist?list=PL26vj4XQfSgq_1NqlklHw3VmU4ZRyFIJ3', 'Tirreno-Adriatico', 2021)

print('\n\nDone with this batch. Tour of Oman playlists are handled separately')
print('(need filtering to "Summary - ..." titles only - see next cell).')

Found 7 videos in playlist for Paris-Nice 2017
  Stage 1: written (2017 Paris-NIce Stage 1 & 2 Summary and Highlights)
  Stage 3: written (2017 Paris-Nice Stage 3 Summary and Highlights)
  Stage 4: written (2017 Paris-Nice Stage 4 Summary & Highlights)
  Stage 5: written (2017 Paris-NIce Stage 5 Summary and Highlights)
  Stage 6: written (2017 Paris-Nice Stage 6 Summary and Highlights)
  Stage 7: written (2017 Paris-Nice Stage 7 Summary and Highlights)
  Stage 8: written (2017 Paris-Nice Stage 8 Summary and Highlights)

Paris-Nice 2017: written=7 skipped=0 unmatched=0
Found 9 videos in playlist for Paris-Nice 2018
  [no stage found] Best of (English) - Paris-Nice 2018
  Stage 1: written (Summary - Stage 1 (Chatou / Meudon) - Paris-Nice 2018)
  Stage 2: written (Summary - Stage 2 - Paris-Nice 2018)
  Stage 3: written (Summary - Stage 3 - Paris-Nice 2018)
  Stage 4: written (Summary - Stage 4 - Paris-Nice 2018)
  Stage 5: written (Summary - Stage 5 - Paris-Nice 2018)
  Stage 6: written (

In [65]:
# ── CELL 24: Tour of Oman 2018/2019 — filtered to "Summary - ..." only ──────
# These playlists contain other content alongside the stage summaries, so
# write_playlist_simple() would pull in unwanted videos too. Filter to
# titles containing "Summary -" before writing.

def write_playlist_filtered(
    playlist_url: str,
    race_name: str,
    year: int,
    title_must_contain: str,
    overwrite_existing: bool = True,
) -> dict:
    """
    Same as write_playlist_simple(), but only considers videos whose title
    contains `title_must_contain` (case-insensitive).
    """
    playlist_id = _extract_playlist_id(playlist_url)
    if not playlist_id:
        print(f'Could not extract playlist ID from: {playlist_url}')
        return {'written': 0, 'skipped': 0, 'unmatched': 0}

    all_videos_in_playlist = _fetch_playlist_videos(playlist_id)
    videos = [v for v in all_videos_in_playlist if title_must_contain.lower() in v['title'].lower()]
    print(f'Found {len(all_videos_in_playlist)} videos in playlist, '
          f'{len(videos)} match "{title_must_contain}" for {race_name} {year}')

    by_stage: dict = {}
    unmatched = 0
    for v in videos:
        stage_num = _extract_stage(v['title'])
        if stage_num is None:
            unmatched += 1
            print(f'  [no stage found] {v["title"][:60]}')
            continue
        by_stage.setdefault(stage_num, []).append(v)

    written = skipped = 0
    for stage_num in sorted(by_stage.keys()):
        candidates = by_stage[stage_num]
        best = max(candidates, key=lambda v: _content_type_score(v['title']))

        if len(candidates) > 1:
            print(f'  Stage {stage_num}: {len(candidates)} candidates, picked best: "{best["title"][:55]}"')

        race_date = f'{year}-01-01'
        for _, row in race_index.iterrows():
            rn, st = parse_race_name_and_stage(row['Race Name'])
            if rn == race_name and row['Date'].year == year and st == stage_num:
                race_date = str(row['Date'])[:10]
                break

        label    = make_label(race_name, race_date, stage_num)
        out_path = RAW_DIR / f'{make_safe_name(label)}.json'

        if out_path.exists() and not overwrite_existing:
            with open(out_path, encoding='utf-8') as f:
                existing = json.load(f)
            if existing.get('status') == 'transcript_saved':
                skipped += 1
                print(f'  Stage {stage_num}: skipped (already has transcript_saved)')
                continue

        record = {
            'label': label, 'race_name': race_name, 'race_date': race_date,
            'stage': stage_num,
            'video': {
                'video_id': best['video_id'], 'title': best['title'],
                'url': f'https://youtube.com/watch?v={best["video_id"]}',
                'channel': 'playlist_import', 'published': str(best.get('published', '')),
                'match_score': 999,
            },
            'transcript': None,
            'status': 'video_found',
        }
        with open(out_path, 'w', encoding='utf-8') as f:
            json.dump(record, f, indent=2, ensure_ascii=False)
        written += 1
        if len(candidates) == 1:
            print(f'  Stage {stage_num}: written ({best["title"][:55]})')

    print(f'\n{race_name} {year}: written={written} skipped={skipped} unmatched={unmatched}')
    return {'written': written, 'skipped': skipped, 'unmatched': unmatched}


write_playlist_filtered(
    'https://www.youtube.com/playlist?list=PLg3WzTG3LfYaiZaRSEDeNzoe8_BK-4Cfh',
    'Tour of Oman', 2018, title_must_contain='Summary -',
)
write_playlist_filtered(
    'https://www.youtube.com/playlist?list=PLg3WzTG3LfYag9CYHUsfy0mLhG-qPBl3w',
    'Tour of Oman', 2019, title_must_contain='Summary -',
)

Found 20 videos in playlist, 6 match "Summary -" for Tour of Oman 2018
  Stage 1: written (Summary - Stage 1 - Tour of Oman 2018)
  Stage 2: written (Summary - Stage 2 - Tour of Oman 2018)
  Stage 3: written (Summary - Stage 3 - Tour of Oman 2018)
  Stage 4: written (Summary - Stage 4 - Tour of Oman 2018)
  Stage 5: written (Summary - Stage 5 - Tour of Oman 2018)
  Stage 6: written (Summary - Stage 6 - Tour of Oman 2018)

Tour of Oman 2018: written=6 skipped=0 unmatched=0
Found 18 videos in playlist, 6 match "Summary -" for Tour of Oman 2019
  Stage 1: written (Stage 1 - Summary - Tour of Oman 2019)
  Stage 2: written (Stage 2 - Summary - Tour of Oman 2019)
  Stage 3: written (Stage 3 - Summary - Tour of Oman 2019)
  Stage 4: written (Stage 4 - Summary - Tour of Oman 2019)
  Stage 5: written (Stage 5 - Summary - Tour of Oman 2019)
  Stage 6: written (Stage 6 - Summary - Tour of Oman 2019)

Tour of Oman 2019: written=6 skipped=0 unmatched=0


{'written': 6, 'skipped': 0, 'unmatched': 0}

In [67]:
# ── CELL 25: Vuelta a Espana 2018/2020/2021 + Criterium du Dauphine 2023 fix ──

# -- 2018 Vuelta a Espana -- full playlist --
write_playlist_simple(
    'https://www.youtube.com/playlist?list=PLwuXMGd35KXUIzoIKIm7NCf-7O_CbYba6',
    'Vuelta a Espana', 2018,
)

# -- 2020 Vuelta a Espana -- re-fix for stages that were reset --
write_manual_stage('Vuelta a Espana', 2020, 10, 'https://www.youtube.com/watch?v=r8O_uuqw164')
write_manual_stage('Vuelta a Espana', 2020, 17, 'https://www.youtube.com/watch?v=0WVHGYYuz3A')

# -- 2021 Vuelta a Espana -- new stages --
write_manual_stage('Vuelta a Espana', 2021, 12, 'https://www.youtube.com/watch?v=AfjCcdtZLs0')
write_manual_stage('Vuelta a Espana', 2021, 18, 'https://www.youtube.com/watch?v=ZRQdgRi3_R0')

# -- 2023 Criterium du Dauphine -- fix Stage 4 --
write_manual_stage('Criterium du Dauphine', 2023, 4, 'https://www.youtube.com/watch?v=pE2-ZNykEJg')

print('\nDone - re-run the full diagnostic to check remaining gaps.')

Found 20 videos in playlist for Vuelta a Espana 2018
  Stage 1: written (Dennis Claims Six Second Lead in Malaga | Vuelta a Espa)
  Stage 2: written (Kwiatkowski Steals Leader's Jersey From Dennis | Vuelta)
  Stage 3: written (Viviani Wins Career First, Kwiatkowski Retained Lead | )
  Stage 4: written (King Dominates Final Climb, Yates Moves to Third | Vuel)
  Stage 5: written (Clarke Prevails in Sprint, Molard Takes Leader's Jersey)
  Stage 6: written (Bouhanni Ends Four-Year Wait, Molard Retains Lead | Vue)
  Stage 7: written (Gallopin Rolls Back the Years to Steal Win | Vuelta a E)
  Stage 8: written (Valverde Victorious in Electrifying Sprint | Vuelta a E)
  Stage 9: written (King Wins Uphill Struggle, Yates Takes Overall Lead | V)
  Stage 10: written (Yates Sustains Puncture But Retains One-Second Lead | V)
  Stage 11: written (De Marchi Eases to Victory, Yates Clings On | Vuelta a )
  Stage 12: written (Geniez Wins in Nail-Biting Style, Yates Loses Lead | Vu)
  Stage 13: written 

In [71]:
# ── CELL 26: Strade Bianche, La Fleche Wallonne, Tirreno 2020, Suisse 2021, Romandie ──

# -- Strade Bianche (one-day race, no stage number) --
write_oneday_manual('Strade Bianche', 2019, 'https://www.youtube.com/watch?v=6XnfAypEnFM')
write_oneday_manual('Strade Bianche', 2020, 'https://www.youtube.com/watch?v=1nlJzufQeNc')
write_oneday_manual('Strade Bianche', 2021, 'https://www.youtube.com/watch?v=iBLDoE-5Kwc')

# -- La Fleche Wallonne (one-day race, no stage number) --
write_oneday_manual('La Fleche Wallonne', 2018, 'https://www.youtube.com/watch?v=_0H-VRfyFMM')
write_oneday_manual('La Fleche Wallonne', 2019, 'https://www.youtube.com/watch?v=ndQ4XsuFpb8')

# -- Tirreno-Adriatico 2020 -- playlist --
# -- Tirreno-Adriatico 2020 -- playlist --
# NOTE: race_index spells this "Tirreno Adriatico" (NO hyphen) for 2020
# ONLY — every other year (2017-2019, 2021-2023) uses "Tirreno-Adriatico"
# (hyphenated). Confirmed via direct race_index inspection, same pattern
# as the Paris-Nice 2020 spelling quirk found earlier.
write_playlist_simple(
    'https://www.youtube.com/playlist?list=PL26vj4XQfSgqia1bk1JNUEiI8OvtLf_f6',
    'Tirreno Adriatico', 2020,
)

# -- Tour de Suisse 2021 -- filtered playlist (lots of unrelated videos --
# require BOTH "stage" and "highlight" in the title, case-insensitive --
def write_playlist_filtered_multi(
    playlist_url: str,
    race_name: str,
    year: int,
    required_keywords: list[str],
    overwrite_existing: bool = True,
) -> dict:
    """
    Like write_playlist_filtered(), but requires ALL of `required_keywords`
    to appear in the title (case-insensitive), rather than one exact phrase.
    """
    playlist_id = _extract_playlist_id(playlist_url)
    if not playlist_id:
        print(f'Could not extract playlist ID from: {playlist_url}')
        return {'written': 0, 'skipped': 0, 'unmatched': 0}

    all_videos_in_playlist = _fetch_playlist_videos(playlist_id)
    videos = [
        v for v in all_videos_in_playlist
        if all(kw.lower() in v['title'].lower() for kw in required_keywords)
    ]
    print(f'Found {len(all_videos_in_playlist)} videos in playlist, '
          f'{len(videos)} match ALL of {required_keywords} for {race_name} {year}')
    for v in videos:
        print(f'  candidate: {v["title"][:70]}')

    by_stage: dict = {}
    unmatched = 0
    for v in videos:
        stage_num = _extract_stage(v['title'])
        if stage_num is None:
            unmatched += 1
            continue
        by_stage.setdefault(stage_num, []).append(v)

    written = skipped = 0
    for stage_num in sorted(by_stage.keys()):
        candidates = by_stage[stage_num]
        best = max(candidates, key=lambda v: _content_type_score(v['title']))

        if len(candidates) > 1:
            print(f'  Stage {stage_num}: {len(candidates)} candidates, picked best: "{best["title"][:55]}"')

        race_date = f'{year}-01-01'
        for _, row in race_index.iterrows():
            rn, st = parse_race_name_and_stage(row['Race Name'])
            if rn == race_name and row['Date'].year == year and st == stage_num:
                race_date = str(row['Date'])[:10]
                break

        label    = make_label(race_name, race_date, stage_num)
        out_path = RAW_DIR / f'{make_safe_name(label)}.json'

        if out_path.exists() and not overwrite_existing:
            with open(out_path, encoding='utf-8') as f:
                existing = json.load(f)
            if existing.get('status') == 'transcript_saved':
                skipped += 1
                continue

        record = {
            'label': label, 'race_name': race_name, 'race_date': race_date,
            'stage': stage_num,
            'video': {
                'video_id': best['video_id'], 'title': best['title'],
                'url': f'https://youtube.com/watch?v={best["video_id"]}',
                'channel': 'playlist_import', 'published': str(best.get('published', '')),
                'match_score': 999,
            },
            'transcript': None,
            'status': 'video_found',
        }
        with open(out_path, 'w', encoding='utf-8') as f:
            json.dump(record, f, indent=2, ensure_ascii=False)
        written += 1
        if len(candidates) == 1:
            print(f'  Stage {stage_num}: written ({best["title"][:55]})')

    print(f'\n{race_name} {year}: written={written} skipped={skipped} unmatched={unmatched}')
    return {'written': written, 'skipped': skipped, 'unmatched': unmatched}


write_playlist_filtered_multi(
    'https://www.youtube.com/playlist?list=PLw-e4qN8sebEt3dTDCR_Bx2Ii-JxzUM0N',
    'Tour de Suisse', 2021,
    required_keywords=['stage', 'highlight'],
)

# -- Tour de Romandie 2019 -- individual videos --
ROMANDIE_2019 = {
    1: 'https://www.youtube.com/watch?v=UzGZoMPOWao',
    2: 'https://www.youtube.com/watch?v=mRONUvJn2Nk',
    3: 'https://www.youtube.com/watch?v=X6F6suf4O3I',
    4: 'https://www.youtube.com/watch?v=rbaSyFhi1G8',
    5: 'https://www.youtube.com/watch?v=wwnFXRjKz90',
}
print('\nWriting Tour de Romandie 2019...')
for stage_num, url in sorted(ROMANDIE_2019.items()):
    write_manual_stage('Tour de Romandie', 2019, stage_num, url)

# -- Tour de Romandie 2021 -- individual videos --
ROMANDIE_2021 = {
    1: 'https://www.youtube.com/watch?v=AXqnkihrr_k',
    2: 'https://www.youtube.com/watch?v=BehS-z5RV88',
}
print('\nWriting Tour de Romandie 2021...')
for stage_num, url in sorted(ROMANDIE_2021.items()):
    write_manual_stage('Tour de Romandie', 2021, stage_num, url)

print('\nDone - re-run the full diagnostic to check remaining gaps.')

  2019: written (6XnfAypEnFM)  -> label="2019 Strade Bianche"
  2020: written (1nlJzufQeNc)  -> label="2020 Strade Bianche"
  2021: written (iBLDoE-5Kwc)  -> label="2021 Strade Bianche"
  2018: written (_0H-VRfyFMM)  -> label="2018 La Fleche Wallonne"
  2019: written (ndQ4XsuFpb8)  -> label="2019 La Fleche Wallonne"
Found 8 videos in playlist for Tirreno Adriatico 2020
  Stage 1: written (Tirreno-Adriatico 2020 | Stage 1 Highlights | inCycle)
  Stage 2: written (Tirreno-Adriatico 2020 | Stage 2 Highlights | inCycle)
  Stage 3: written (Tirreno-Adriatico 2020 | Stage 3 Highlights | inCycle)
  Stage 4: written (Tirreno-Adriatico 2020 | Stage 4 Highlights | inCycle)
  Stage 5: written (Tirreno-Adriatico 2020 | Stage 5 Highlights | inCycle)
  Stage 6: written (Tirreno-Adriatico 2020 | Stage 6 Highlights | inCycle)
  Stage 7: written (Tirreno-Adriatico 2020 | Stage 7 Highlights | inCycle)
  Stage 8: written (Tirreno-Adriatico 2020 | Stage 8 Highlights | inCycle)

Tirreno Adriatico 2020: wri

In [73]:
for name in ['catalunya', 'vuelta']:
    seen = set()
    for _, row in race_index.iterrows():
        rn, _ = parse_race_name_and_stage(row['Race Name'])
        if name in rn.lower():
            seen.add(rn)
    print(name, '->', seen)

catalunya -> {'Volta Ciclista a Catalunya'}
vuelta -> {'La Vuelta ciclista a Espana', 'La Vuelta Ciclista a Espana', 'Vuelta a Espana'}


In [74]:
# ── DIAGNOSTIC: Vuelta a Espana spelling by year ─────────────────────────────

for _, row in race_index.iterrows():
    rn, st = parse_race_name_and_stage(row['Race Name'])
    if 'vuelta' in rn.lower():
        print(f"{row['Date'].year}  {rn!r}  stage={st}")

2023  'La Vuelta Ciclista a Espana'  stage=21
2023  'La Vuelta Ciclista a Espana'  stage=20
2023  'La Vuelta Ciclista a Espana'  stage=19
2023  'La Vuelta Ciclista a Espana'  stage=18
2023  'La Vuelta Ciclista a Espana'  stage=17
2023  'La Vuelta Ciclista a Espana'  stage=16
2023  'La Vuelta Ciclista a Espana'  stage=15
2023  'La Vuelta Ciclista a Espana'  stage=14
2023  'La Vuelta Ciclista a Espana'  stage=13
2023  'La Vuelta Ciclista a Espana'  stage=12
2023  'La Vuelta Ciclista a Espana'  stage=11
2023  'La Vuelta Ciclista a Espana'  stage=10
2023  'La Vuelta Ciclista a Espana'  stage=9
2023  'La Vuelta Ciclista a Espana'  stage=8
2023  'La Vuelta Ciclista a Espana'  stage=7
2023  'La Vuelta Ciclista a Espana'  stage=6
2023  'La Vuelta Ciclista a Espana'  stage=5
2023  'La Vuelta Ciclista a Espana'  stage=4
2023  'La Vuelta Ciclista a Espana'  stage=3
2023  'La Vuelta Ciclista a Espana'  stage=2
2023  'La Vuelta Ciclista a Espana'  stage=1
2022  'La Vuelta ciclista a Espana'  stage=

In [75]:
# ── CELL 27: Volta a Catalunya 2018/2019/2021 + Vuelta/Dauphine primary writes ──
# For entries with a stated fallback video, this writes the PRIMARY video
# first. After the next CLI fetch run, check which of these labels still
# failed - THOSE are the ones to retry with their fallback video_id.

# -- Volta Ciclista a Catalunya 2018 -- individual videos --
CATALUNYA_2018 = {
    1: 'https://www.youtube.com/watch?v=NaEDxvoeNrY',
    2: 'https://www.youtube.com/watch?v=URoQ1-e3I5Y',
    3: 'https://www.youtube.com/watch?v=NhCAhKD4tL4',
    4: 'https://www.youtube.com/watch?v=r1h1t8yymMY',
}
print('Writing Volta Ciclista a Catalunya 2018...')
for stage_num, url in sorted(CATALUNYA_2018.items()):
    write_manual_stage('Volta Ciclista a Catalunya', 2018, stage_num, url)

# -- Volta Ciclista a Catalunya 2019 -- individual videos --
# NOTE: Stage 1 and 2 share the SAME video.
CATALUNYA_2019 = {
    1: 'https://www.youtube.com/watch?v=h_vRYP8c4Ys',
    2: 'https://www.youtube.com/watch?v=h_vRYP8c4Ys',
}
print('\nWriting Volta Ciclista a Catalunya 2019...')
for stage_num, url in sorted(CATALUNYA_2019.items()):
    write_manual_stage('Volta Ciclista a Catalunya', 2019, stage_num, url)

# -- Volta Ciclista a Catalunya 2021 -- playlist --
write_playlist_simple(
    'https://www.youtube.com/playlist?list=PL_NgtX8A_VP5pYNWMsoCjTcUvkI6rWt2b',
    'Volta Ciclista a Catalunya', 2021,
)

# -- Entries with a primary + fallback video - writing PRIMARY only for now --
PRIMARY_WITH_FALLBACK = {
    ('Criterium du Dauphine',        2023, 4):  'https://www.youtube.com/watch?v=pE2-ZNykEJg',
    ('Vuelta a Espana',              2018, 13): 'https://www.youtube.com/watch?v=ip-CP1jnHqg',
    ('Vuelta a Espana',              2018, 4):  'https://www.youtube.com/watch?v=5-paDAzeL2g',
    ('Vuelta a Espana',              2018, 8):  'https://www.youtube.com/watch?v=hmi5MOk3P0s',
    ('Vuelta a Espana',              2018, 15): 'https://www.youtube.com/watch?v=8uA5Lq8qwEo',
    ('Vuelta a Espana',              2020, 10): 'https://www.youtube.com/watch?v=r8O_uuqw164',
    ('Vuelta a Espana',              2020, 17): 'https://www.youtube.com/watch?v=0WVHGYYuz3A',
    # 2021 uses a DIFFERENT spelling in race_index than 2018/2020 —
    # confirmed via direct inspection: "La Vuelta ciclista a Espana"
    # (note lowercase "ciclista"), not "Vuelta a Espana". Same recurring
    # per-year spelling drift pattern as Paris-Nice/Tirreno-Adriatico.
    ('La Vuelta ciclista a Espana',  2021, 12): 'https://www.youtube.com/watch?v=AfjCcdtZLs0',
    ('La Vuelta ciclista a Espana',  2021, 18): 'https://www.youtube.com/watch?v=ZRQdgRi3_R0',
}

FALLBACKS = {
    ('Criterium du Dauphine',       2023, 4):  'https://www.youtube.com/watch?v=kqJAaIqU1VM',
    ('Vuelta a Espana',             2018, 15): 'https://www.youtube.com/watch?v=ANlLDAgj_CU',
    ('Vuelta a Espana',             2020, 10): 'https://www.youtube.com/watch?v=pctuy4N6dic',
    ('Vuelta a Espana',             2020, 17): 'https://www.youtube.com/watch?v=EPzdH5u6_b4',
    ('La Vuelta ciclista a Espana', 2021, 12): 'https://www.youtube.com/watch?v=bdDy52tZxp0',
    ('La Vuelta ciclista a Espana', 2021, 18): 'https://www.youtube.com/watch?v=73K5uHfeIOw',
}
# 2018 Vuelta Stage 13/4/8 have no stated fallback.

print('\nWriting primary videos for Vuelta/Dauphine fallback-pair entries...')
for (race_name, year, stage_num), url in PRIMARY_WITH_FALLBACK.items():
    write_manual_stage(race_name, year, stage_num, url)

print('\nDone. Run the CLI fetch next. After it finishes, check which of')
print('these specific labels still failed, then use retry_with_fallback()')
print('(CELL 28) to swap in the fallback for just those.')

Writing Volta Ciclista a Catalunya 2018...
  Stage 1: written (NaEDxvoeNrY)
  Stage 2: written (URoQ1-e3I5Y)
  Stage 3: written (NhCAhKD4tL4)
  Stage 4: written (r1h1t8yymMY)

Writing Volta Ciclista a Catalunya 2019...
  Stage 1: written (h_vRYP8c4Ys)
  Stage 2: written (h_vRYP8c4Ys)
Found 8 videos in playlist for Volta Ciclista a Catalunya 2021
  [no stage found] MEGA Volta a Catalunya Preview 2021| Hirschi, Froome, Sagan 
  Stage 1: written (What were Movistar THINKING?! | Volta a Catalunya Stage)
  Stage 2: written (João Almeida Takes the Leader's Jersey | Volta a Catalu)
  Stage 3: written (Adam Yates serves up the PAIN | Volta a Catalunya Stage)
  Stage 4: written (ICE COLD Esteban Chaves Attacks the INEOS Train | Volta)
  Stage 5: written (RIP the Supertuck | Volta a Catalunya Stage 5 2021)
  Stage 6: written (Peter Sagan IMPOSSIBLE Sprint Through TINY SPACE | Volt)
  Stage 7: written (INEOS Grenadiers CLEAN SWEEP the Podium | Volta a Catal)

Volta Ciclista a Catalunya 2021: writ

In [76]:
# ── CELL 28: Check fallback-pair results, retry failures with fallback video ──
# Run this AFTER the CLI transcript fetch has processed the primary videos
# from CELL 27's PRIMARY_WITH_FALLBACK.

FAILURE_STATUSES = {
    'no_video_found',
    'transcript_error:no_transcript',
    'transcript_error:transcripts_disabled',
    'transcript_error:video_unplayable_geo_restricted',
}


def check_and_queue_fallbacks(primary_with_fallback: dict, fallbacks: dict) -> list:
    """
    For every (race_name, year, stage) in `fallbacks`, check the current
    status. If it's a known failure status, write the fallback video as
    a fresh video_found record. Returns the list of keys re-queued.
    """
    requeued = []
    for key, fallback_url in fallbacks.items():
        race_name, year, stage_num = key

        race_date = f'{year}-01-01'
        for _, row in race_index.iterrows():
            rn, st = parse_race_name_and_stage(row['Race Name'])
            if rn == race_name and row['Date'].year == year and st == stage_num:
                race_date = str(row['Date'])[:10]
                break

        label    = make_label(race_name, race_date, stage_num)
        out_path = RAW_DIR / f'{make_safe_name(label)}.json'

        if not out_path.exists():
            print(f'{label}: no file found - skipping (was the primary ever written?)')
            continue

        with open(out_path, encoding='utf-8') as f:
            data = json.load(f)
        status = data.get('status', '')

        if status == 'transcript_saved':
            print(f'{label}: OK - primary video succeeded, no fallback needed')
            continue

        if status in FAILURE_STATUSES:
            print(f'{label}: primary failed ({status}) - queuing fallback')
            write_manual_stage(race_name, year, stage_num, fallback_url)
            requeued.append(key)
        else:
            print(f'{label}: unexpected status ({status}) - check manually')

    print(f'\n{len(requeued)} entries re-queued with their fallback video.')
    if requeued:
        print('Run the CLI fetch again to attempt these.')
    return requeued


# Usage (after running the CLI fetch on CELL 27's primaries):
#   check_and_queue_fallbacks(PRIMARY_WITH_FALLBACK, FALLBACKS)
print('Ready: check_and_queue_fallbacks(PRIMARY_WITH_FALLBACK, FALLBACKS)')

Ready: check_and_queue_fallbacks(PRIMARY_WITH_FALLBACK, FALLBACKS)


In [77]:
check_and_queue_fallbacks(PRIMARY_WITH_FALLBACK, FALLBACKS)

2023 Criterium du Dauphine Stage 4: primary failed (transcript_error:transcripts_disabled) - queuing fallback
  Stage 4: written (kqJAaIqU1VM)
2018 Vuelta a Espana Stage 15: primary failed (transcript_error:transcripts_disabled) - queuing fallback
  Stage 15: written (ANlLDAgj_CU)
2020 Vuelta a Espana Stage 10: primary failed (transcript_error:transcripts_disabled) - queuing fallback
  Stage 10: written (pctuy4N6dic)
2020 Vuelta a Espana Stage 17: primary failed (transcript_error:transcripts_disabled) - queuing fallback
  Stage 17: written (EPzdH5u6_b4)
2021 La Vuelta ciclista a Espana Stage 12: primary failed (transcript_error:no_transcript) - queuing fallback
  Stage 12: written (bdDy52tZxp0)
2021 La Vuelta ciclista a Espana Stage 18: primary failed (transcript_error:no_transcript) - queuing fallback
  Stage 18: written (73K5uHfeIOw)

6 entries re-queued with their fallback video.
Run the CLI fetch again to attempt these.


[('Criterium du Dauphine', 2023, 4),
 ('Vuelta a Espana', 2018, 15),
 ('Vuelta a Espana', 2020, 10),
 ('Vuelta a Espana', 2020, 17),
 ('La Vuelta ciclista a Espana', 2021, 12),
 ('La Vuelta ciclista a Espana', 2021, 18)]

In [79]:
# ── CELL 29: Amstel Gold Race 2019, Il Lombardia 2020, Paris-Nice 2017 fix ──

# -- Amstel Gold Race (one-day race, no stage number) --
write_oneday_manual('Amstel Gold Race', 2019, 'https://www.youtube.com/watch?v=J6rA9zou0PU')

# -- Il Lombardia (one-day race, no stage number) --
write_oneday_manual('Il Lombardia', 2020, 'https://www.youtube.com/watch?v=Ngw5jjHHvpk')

# -- Paris-Nice 2017 Stage 2 -- individual fix --
write_manual_stage('Paris-Nice', 2017, 2, 'https://www.youtube.com/watch?v=USUOZLdYqxk')

# -- Tour de Suisse 2021 Stage 1 -- PENDING: URL not yet provided --
# write_manual_stage('Tour de Suisse', 2021, 1, '<URL HERE>')

print('\nDone with this batch (3 of 4 - Tour de Suisse 2021 Stage 1 still needs its URL).')

  2019: written (J6rA9zou0PU)  -> label="2019 Amstel Gold Race"
  2020: written (Ngw5jjHHvpk)  -> label="2020 Il Lombardia"
  Stage 2: written (USUOZLdYqxk)

Done with this batch (3 of 4 - Tour de Suisse 2021 Stage 1 still needs its URL).


In [80]:
# ── CELL 30: Tour de Suisse 2021 individual stage fixes + 2022 Stage 2 ──

# -- Tour de Suisse 2021 -- individual videos --
SUISSE_2021_FIXES = {
    1: 'https://www.youtube.com/watch?v=xrVkzcmLNzU',
    3: 'https://www.youtube.com/watch?v=Zo8rYTZnNxI',
    4: 'https://www.youtube.com/watch?v=kXMFd9e-vhs',
    5: 'https://www.youtube.com/watch?v=On8T5aw6sNE',
    6: 'https://www.youtube.com/watch?v=b2wI44CF5Ac',
}
print('Writing Tour de Suisse 2021 fixes...')
for stage_num, url in sorted(SUISSE_2021_FIXES.items()):
    write_manual_stage('Tour de Suisse', 2021, stage_num, url)

# -- Tour de Suisse 2022 Stage 2 -- individual fix --
write_manual_stage('Tour de Suisse', 2022, 2, 'https://www.youtube.com/watch?v=e8uRO7AfKb8')

print('\nDone.')

Writing Tour de Suisse 2021 fixes...
  Stage 1: written (xrVkzcmLNzU)
  Stage 3: written (Zo8rYTZnNxI)
  Stage 4: written (kXMFd9e-vhs)
  Stage 5: written (On8T5aw6sNE)
  Stage 6: written (b2wI44CF5Ac)
  Stage 2: written (e8uRO7AfKb8)

Done.


In [86]:
# ── DIAGNOSTIC: does Tour Down Under exist anywhere? ─────────────────────────

print('Searching race_index for any "down under" match:')
found_any = False
for _, row in race_index.iterrows():
    if 'down under' in str(row['Race Name']).lower():
        print(f"  {row['Date'].year}  {row['Race Name']!r}")
        found_any = True
if not found_any:
    print('  (none found)')

print('\nSearching race_index for "santos":')
found_any = False
for _, row in race_index.iterrows():
    if 'santos' in str(row['Race Name']).lower():
        print(f"  {row['Date'].year}  {row['Race Name']!r}")
        found_any = True
if not found_any:
    print('  (none found)')

print(f'\nTotal rows in race_index: {len(race_index)}')
print(f'Years present: {sorted(race_index["Date"].dt.year.unique())}')
print(f'Unique race names containing "tour": ')
tour_races = sorted(set(
    parse_race_name_and_stage(rn)[0] for rn in race_index['Race Name'] if 'tour' in rn.lower()
))
for t in tour_races:
    print(f'  {t!r}')

Searching race_index for any "down under" match:
  (none found)

Searching race_index for "santos":
  (none found)

Total rows in race_index: 896
Years present: [np.int32(2017), np.int32(2018), np.int32(2019), np.int32(2020), np.int32(2021), np.int32(2022), np.int32(2023)]
Unique race names containing "tour": 
  'Benelux Tour'
  'BinckBank Tour'
  'Presidential Cycling Tour of Turkey'
  'Ronde van Vlaanderen - Tour des Flandres'
  'Tour de France'
  'Tour de Pologne'
  'Tour de Romandie'
  'Tour de Suisse'
  'Tour of Guangxi'
  'Tour of Oman'
  'UAE Tour'


In [ ]:
# ── CELL 10: Claude tactical extraction ──────────────────────────────────────
# Reads saved transcripts and extracts structured tactical events.
# Unchanged from v2.

EXTRACTION_PROMPT = """You are analyzing cycling race commentary for {race_name}.

Extract tactically significant information from this transcript.
Focus only on information useful for pre-race analysis of future similar stages.
Ignore music, crowd noise, and generic excitement phrases.

Return a JSON object:
{{
  "race_summary": "2-3 sentence tactical summary of what happened",
  "decisive_moment": "the single most important tactical event",
  "commentary_quality": "rich|moderate|thin",
  "events": [
    {{
      "event_type": "attack|breakaway|chase|team_tactic|rider_observation|weather|terrain",
      "riders": ["LASTNAME Firstname"],
      "teams": ["Team Name"],
      "description": "what happened concisely",
      "tactical_significance": "why this matters for future analysis"
    }}
  ],
  "rider_form_signals": [
    {{
      "rider": "LASTNAME Firstname",
      "signal": "positive|negative|neutral",
      "observation": "what commentary reveals about their condition"
    }}
  ],
  "terrain_observations": [
    "observation about how terrain affected racing dynamics"
  ],
  "usable_for_rag": true
}}

Return only valid JSON, no other text.
Transcript:
{transcript}"""


def extract_one(label: str, transcript_text: str) -> dict:
    max_chars = 8000
    if len(transcript_text) > max_chars:
        half = max_chars // 2
        text = transcript_text[:half] + '\n[...middle omitted...]\n' + transcript_text[-half:]
    else:
        text = transcript_text

    response = claude.messages.create(
        model='claude-sonnet-4-5',
        max_tokens=3000,
        messages=[{'role': 'user', 'content': EXTRACTION_PROMPT.format(
            race_name=label, transcript=text
        )}]
    )
    raw = re.sub(r'```json|```', '', response.content[0].text).strip()
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        return {'error': 'json_parse_failed', 'raw': raw[:200]}


def run_extraction(max_extractions: int = 50, verbose: bool = True) -> dict:
    pending = [
        path for path in sorted(RAW_DIR.glob('*.json'))
        if not (EXTRACTED_DIR / path.name).exists()
        and json.load(open(path, encoding='utf-8')).get('status') == 'transcript_saved'
    ]

    total_cost = success = skipped = errors = 0
    print(f'Transcripts pending extraction: {len(pending)}')
    print(f'Processing up to: {max_extractions}')
    print(f'Est. cost: ${min(len(pending), max_extractions) * 0.02:.2f}')
    print(f'{chr(8212)*55}')

    for path in pending[:max_extractions]:
        extracted_path = EXTRACTED_DIR / path.name
        with open(path, encoding='utf-8') as f:
            data = json.load(f)
        transcript_text = data.get('transcript', {}).get('clean_text', '')
        if not transcript_text.strip():
            skipped += 1
            continue

        label = data.get('label', path.stem)
        chars = len(transcript_text)
        cost  = min(chars, 8000) / 4 / 1_000_000 * 3

        if verbose:
            print(f'Extracting: {label}')
            print(f'  Chars: {chars:,} | Est: ${cost:.4f}')

        try:
            events = extract_one(label, transcript_text)
            output = {
                'label':       label,
                'race_name':   data.get('race_name'),
                'race_date':   data.get('race_date'),
                'stage':       data.get('stage'),
                'video_title': data.get('video', {}).get('title'),
                'video_url':   data.get('video', {}).get('url'),
                'channel':     data.get('video', {}).get('channel'),
                'extraction':  events,
                'extracted_at': str(datetime.datetime.utcnow()),
            }
            with open(extracted_path, 'w', encoding='utf-8') as f:
                json.dump(output, f, indent=2, ensure_ascii=False)

            total_cost += cost
            success    += 1
            if verbose:
                quality  = events.get('commentary_quality', 'unknown')
                n_events = len(events.get('events', []))
                print(f'  ✓ Quality: {quality} | Events: {n_events}')
        except Exception as e:
            print(f'  ✗ Error: {e}')
            errors += 1

        time.sleep(0.5)

    print(f'\n{chr(8212)*55}')
    print(f'Extraction complete')
    print(f'  Extracted:   {success}')
    print(f'  Skipped:     {skipped}')
    print(f'  Errors:      {errors}')
    print(f'  Total cost:  ${total_cost:.4f}')
    return {'success': success, 'skipped': skipped, 'errors': errors, 'cost': total_cost}


# ── RUN EXTRACTION ─────────────────────────────────────────────────────────────
# extraction_stats = run_extraction(max_extractions=50, verbose=True)
print('Extraction functions ready — uncomment the line above to run')


In [ ]:
# ── CELL 11: Commentary retrieval ────────────────────────────────────────────
# Used by the agent notebook. Always returns a string — never raises.
# Unchanged from v2.

def get_commentary_context(
    race_name: str,
    race_date: str,
    stage: int = None,
    max_chars: int = 3000,
) -> str:
    label     = make_label(race_name, race_date, stage)
    safe_name = make_safe_name(label)

    extracted_path = EXTRACTED_DIR / f'{safe_name}.json'
    raw_path       = RAW_DIR       / f'{safe_name}.json'

    if extracted_path.exists():
        try:
            with open(extracted_path, encoding='utf-8') as f:
                data = json.load(f)
            ext = data.get('extraction', {})
            if 'error' not in ext:
                parts = [f'[COMMENTARY: {data.get("channel")} | {data.get("video_title","")[:55]}]']
                if ext.get('race_summary'):
                    parts.append(f'Summary: {ext["race_summary"]}')
                if ext.get('decisive_moment'):
                    parts.append(f'Decisive moment: {ext["decisive_moment"]}')
                for e in ext.get('events', [])[:5]:
                    riders = ', '.join(e.get('riders', []))
                    parts.append(f'- [{e["event_type"]}] {e["description"]}' +
                                 (f' ({riders})' if riders else ''))
                for s in ext.get('rider_form_signals', [])[:3]:
                    parts.append(f'- Form: {s["rider"]} ({s["signal"]}): {s["observation"]}')
                return '\n'.join(parts)
        except Exception:
            pass

    if raw_path.exists():
        try:
            with open(raw_path, encoding='utf-8') as f:
                data = json.load(f)
            if data.get('status') == 'transcript_saved':
                text = data.get('transcript', {}).get('clean_text', '')
                if text:
                    if len(text) > max_chars:
                        half = max_chars // 2
                        text = text[:half] + '\n[...]\n' + text[-half:]
                    channel = data.get('video', {}).get('channel', 'unknown')
                    title   = data.get('video', {}).get('title', '')[:55]
                    return f'[RAW COMMENTARY: {channel} | {title}]\n\n{text}'
        except Exception:
            pass

    return f'[NO COMMENTARY for {label}] Analysis based on structured data only.'


# Quick test
print('=== Commentary retrieval ready ===')
